# 09 - Real GPU loop with the word-completion heads

**Generated by `scripts/build_kaggle_notebooks.py` (git=f73985b92d21-dirty src+data_sha256=dbdbcab6a266). Do not edit; change `src/lgmtp/` and rebuild.**

Wall-clock speed of **NoDraft**, **Entropy**, **V1 + lexicon** and three *checked* methods on the 100 Phase-1 real-loop sentences (FLORES devtest), Llama-3-8B target and Llama-3.2-1B draft, 3 repeats. In the checked methods the 1B checks the proposals of a cheap source inside its next call: the V1 heads (**V1+check**), a Hindi token 4-gram (**NGram**, nothing trained) or both (**NGram+V0**). The simulated results (Entropy 1.452x, V1+lex 1.542x, V1+check 1.572x, NGram 1.701x, NGram+V0 1.708x with the 180k heads) have never been timed. Thresholds are the dev-tuned ones from `results/wordheads*/main_table.csv` and `results/checked/params.json`, chosen automatically for the 60k or 180k heads. Prompts and the 8B's dev outputs are embedded, so FLORES access is not needed.

**Inputs (right sidebar, Add Input):**
1. The **output of notebook 07b** (180k heads): *Your Work > Notebooks*, or `wordheads.zip` uploaded as a private Dataset. Notebook 07's heads (60k) also work; if both are attached, the 180k heads are used.
2. **`hindi.csv`** (the Hindi word list), uploaded as a private Dataset, keeping that file name.
3. **`ngram_hi.npz`** (the Hindi 4-gram table, from `scripts/run_checked_sim.py`), in the same Dataset.

The notebook stops within seconds if one is missing, before loading the 8B. Without `hindi.csv` remove `"V1+lex"` from `METHODS`; without `ngram_hi.npz` remove the `"NGram"` methods.

**Settings:** Accelerator **GPU T4 x2**, Internet **On**, secret **`HF_TOKEN`** (Llama access; without it ungated copies are tried). **Run:** *Save Version > Save & Run All (Commit)*. Roughly 2.5-3 h (from the Phase-1 loop times); after 5 sentences the log prints `ESTIMATE: ...`. To shorten it, lower `REPEATS` or `N_SENT`.

**Output:** `/kaggle/working/heads_loop.zip` -> unzip into `results/heads_loop/` in the repo. `SUMMARY.md` has the table.
**If it dies:** attach the failed version's output as an input and rerun; finished decodes are reused.

In [ ]:
!pip -q install -U "transformers==5.22.1" accelerate huggingface_hub regex

In [ ]:
# === lgmtp source, generated by scripts/build_kaggle_notebooks.py: DO NOT EDIT ===
import os, sys, importlib
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
PKG = '/kaggle/working/lgmtp' if os.path.isdir('/kaggle/working') else os.path.abspath('lgmtp')
os.makedirs(PKG, exist_ok=True)
LGMTP_SOURCES = {
    '__init__': '"""lgmtp: linguistically guided multi-token prediction (CS769)."""\n',
    'boundaries': '"""Token bytes, Devanagari akshara units, word boundaries and per-token cut labels.\n\nExtracted from ``notebooks/legacy/test1_hindi_boundaries.ipynb`` (cell 2). The logic of the\nlegacy functions is unchanged; only type hints / docstrings were added, plus ``token_labels``.\n\nFixes bug B4 (never decode single byte-level tokens): everything works on token *bytes* and\nbyte offsets, and text is only decoded for whole sequences.\n"""\nfrom __future__ import annotations\n\nimport json\nimport re\nimport unicodedata\nfrom collections import Counter\nfrom itertools import accumulate\nfrom typing import Iterable, Sequence\n\nimport numpy as np\n\n# --------------------------------------------------------------------------- #\n# token -> bytes\n# --------------------------------------------------------------------------- #\n\n\ndef _bytes_to_unicode() -> dict[int, str]:\n    """GPT-2 byte<->unicode table (used by Llama-3 and Qwen byte-level BPE)."""\n    bs = (list(range(ord("!"), ord("~") + 1))\n          + list(range(ord("¡"), ord("¬") + 1))\n          + list(range(ord("®"), ord("ÿ") + 1)))\n    cs = bs[:]\n    n = 0\n    for b in range(256):\n        if b not in bs:\n            bs.append(b)\n            cs.append(256 + n)\n            n += 1\n    return dict(zip(bs, [chr(c) for c in cs]))\n\n\nBYTE_DECODER: dict[str, int] = {v: k for k, v in _bytes_to_unicode().items()}\n_HEX_RE = re.compile(r"^<0x([0-9A-Fa-f]{2})>$")\n\n\ndef token_to_bytes(tok: str) -> bytes:\n    """Raw bytes of a vocabulary token string (as returned by ``convert_ids_to_tokens``)."""\n    m = _HEX_RE.match(tok)\n    if m:  # SentencePiece byte-fallback token\n        return bytes([int(m.group(1), 16)])\n    if all(ch in BYTE_DECODER for ch in tok):  # GPT-2 style byte-level BPE\n        return bytes(BYTE_DECODER[ch] for ch in tok)\n    return tok.replace("▁", " ").encode("utf-8")  # SentencePiece unicode piece\n\n\nclass HFTok:\n    """Thin wrapper: text -> list of token byte-strings."""\n\n    def __init__(self, name_or_tokenizer):\n        if isinstance(name_or_tokenizer, str):\n            from transformers import AutoTokenizer\n            self.name = name_or_tokenizer\n            self.tok = AutoTokenizer.from_pretrained(name_or_tokenizer)\n        else:\n            self.name = "custom"\n            self.tok = name_or_tokenizer\n\n    def to_bytes(self, text: str) -> list[bytes]:\n        ids = self.tok(text, add_special_tokens=False)["input_ids"]\n        return [token_to_bytes(t) for t in self.tok.convert_ids_to_tokens(ids)]\n\n    def ids_to_bytes(self, ids: Sequence[int]) -> list[bytes]:\n        """Byte-strings for a list of token ids (no decoding of individual tokens)."""\n        return [token_to_bytes(t) for t in self.tok.convert_ids_to_tokens(list(ids))]\n\n\nQWEN2_PRETOK_REGEX = (r"(?i:\'s|\'t|\'re|\'ve|\'m|\'ll|\'d)|[^\\r\\n\\p{L}\\p{N}]?\\p{L}+|\\p{N}"\n                      r"| ?[^\\s\\p{L}\\p{N}]+[\\r\\n]*|\\s*[\\r\\n]+|\\s+(?!\\S)|\\s+")\n\n\ndef legacy_qwen2_pretok_tokenizer(name: str = "Qwen/Qwen3.5-0.8B"):\n    """``name``\'s tokenizer.json with its split regex replaced by the older Qwen2 one (``\\\\p{L}+``,\n    no ``\\\\p{M}``), which splits before every combining mark.\n\n    Only used to explain the legacy Test 1 Qwen number (3.33 tokens/word): the legacy run loaded\n    the tokenizer this way. The model\'s real tokenizer (``[\\\\p{L}\\\\p{M}]+``) gives ~2.16.\n    """\n    from huggingface_hub import hf_hub_download\n    from tokenizers import Tokenizer\n    from transformers import PreTrainedTokenizerFast\n    j = json.load(open(hf_hub_download(name, "tokenizer.json"), encoding="utf-8"))\n    j["pre_tokenizer"]["pretokenizers"][0]["pattern"]["Regex"] = QWEN2_PRETOK_REGEX\n    return PreTrainedTokenizerFast(tokenizer_object=Tokenizer.from_str(json.dumps(j)))\n\n\n# --------------------------------------------------------------------------- #\n# akshara + word boundaries\n# --------------------------------------------------------------------------- #\n\n_C = "क-हक़-य़ॸ-ॿ"      # consonants\n_VIRAMA = "्"\n_NUKTA = "़"\n_ZW = "\u200c\u200d"                                 # ZWNJ / ZWJ\n_IVOW = "ऄ-औॠॡॲ-ॷ"     # independent vowels\n_MATRA = "ऺऻा-ौॎॏॕ-ॗॢॣ"\n_MOD = "ऀ-ः॑-॔"                  # candrabindu, anusvara, visarga, stress marks\n\n_AKSHARA_RE = re.compile(\n    rf"[{_C}]{_NUKTA}?(?:{_VIRAMA}[{_ZW}]?[{_C}]{_NUKTA}?)*(?:{_VIRAMA}[{_ZW}]?|[{_MATRA}]*)[{_MOD}]*"\n    rf"|[{_IVOW}]{_NUKTA}?[{_MATRA}]*[{_MOD}]*"\n)\n\n\ndef akshara_units(text: str) -> list[tuple[int, int]]:\n    """Return list of (start, end) char spans. Non-Devanagari chars are single units\n    with any following combining marks / ZW joiners attached."""\n    units, i, n = [], 0, len(text)\n    while i < n:\n        m = _AKSHARA_RE.match(text, i)\n        if m and m.end() > i:\n            j = m.end()\n        else:\n            j = i + 1\n            while j < n and (unicodedata.category(text[j]).startswith("M") or text[j] in "\u200c\u200d"):\n                j += 1\n        units.append((i, j))\n        i = j\n    return units\n\n\ndef is_word_boundary(text: str, c: int) -> bool:\n    """True if char offset ``c`` is a word boundary (text edge, whitespace or punctuation)."""\n    n = len(text)\n    if c <= 0 or c >= n:\n        return True\n    a, b = text[c - 1], text[c]\n    if a.isspace() or b.isspace():\n        return True\n    if unicodedata.category(a).startswith("P") or unicodedata.category(b).startswith("P"):\n        return True\n    return False\n\n\nEND, WORD, MIDWORD, MIDAKS, MIDUTF8 = "end", "word_boundary", "mid_word", "mid_akshara", "mid_utf8"\nCUT_CATEGORIES = (END, WORD, MIDWORD, MIDAKS, MIDUTF8)\nTAG = {END: "E", WORD: "W", MIDWORD: "w", MIDAKS: "a", MIDUTF8: "u"}\n\n\ndef classify(b: int, total: int, byte2char: dict[int, int], text: str, aks: set[int]) -> str:\n    """Category of a cut at byte offset ``b``.\n\n    byte2char maps byte offsets that start a char (plus ``total``) to char offsets;\n    aks is the set of char offsets that are akshara-unit boundaries.\n    """\n    if b >= total:\n        return END\n    c = byte2char.get(b)\n    if c is None:\n        return MIDUTF8\n    if c not in aks:\n        return MIDAKS\n    return WORD if is_word_boundary(text, c) else MIDWORD\n\n\ndef sentence_features(tb: Sequence[bytes]) -> dict | None:\n    """tb: list of token byte-strings. Returns None if bytes are not valid UTF-8."""\n    full = b"".join(tb)\n    try:\n        text = full.decode("utf-8")\n    except UnicodeDecodeError:\n        return None\n    byte2char, pos = {}, 0\n    for ci, ch in enumerate(text):\n        byte2char[pos] = ci\n        pos += len(ch.encode("utf-8"))\n    byte2char[pos] = len(text)\n    units = akshara_units(text)\n    aks = {0} | {e for _, e in units}\n    ends = list(accumulate(len(b) for b in tb))\n    starts = [0] + ends[:-1]\n    total = len(full)\n    return {\n        "text": text,\n        "units": units,\n        "cut": [classify(e, total, byte2char, text, aks) for e in ends],\n        "start": [classify(s, total, byte2char, text, aks) for s in starts],\n    }\n\n\n# --------------------------------------------------------------------------- #\n# per-token gold labels (analysis only; never read by policies)\n# --------------------------------------------------------------------------- #\n\n\ndef _valid_utf8_prefix_len(full: bytes) -> int:\n    """Length of the longest prefix of ``full`` that is valid UTF-8, allowing only a\n    truncated character at the very end (e.g. generation stopped at MAX_NEW)."""\n    for cut in range(len(full), max(len(full) - 4, -1), -1):\n        try:\n            full[:cut].decode("utf-8")\n            return cut\n        except UnicodeDecodeError:\n            continue\n    raise ValueError("token bytes are not valid UTF-8 (invalid sequence before the end)")\n\n\ndef token_labels(token_bytes: Sequence[bytes],\n                 special: Sequence[bool] | None = None) -> dict[str, np.ndarray]:\n    """Gold labels for each token t of a target sequence (analysis only, never read by policies).\n\n    end_cut[t]      : category of the cut AFTER token t (END/WORD/MIDWORD/MIDAKS/MIDUTF8)\n    start_cut[t]    : category of the cut BEFORE token t\n    word_initial[t] : token t starts a new word (start_cut in {WORD, END})\n    pos_in_word[t]  : 0 for word-initial tokens, then 1, 2, ...\n    word_id[t]      : index of the word the token belongs to\n    char_span[t]    : (start_char, end_char) in the decoded text, shape (n, 2)\n\n    ``special[t]`` marks special/control tokens (e.g. ``<|eot_id|>``). They are stripped before\n    labelling and get ``END`` for both cuts, ``word_initial=True``, ``pos_in_word=0``, a fresh\n    ``word_id`` and an empty char span at the end of the text. If ``special`` is None, no token is\n    treated as special.\n\n    A trailing incomplete UTF-8 character (output truncated mid-character) is tolerated: cuts at\n    its start or inside it are ``MIDUTF8`` (what follows is unknown). Invalid bytes anywhere else\n    raise ``ValueError``.\n    """\n    n = len(token_bytes)\n    special = [False] * n if special is None else [bool(s) for s in special]\n    if len(special) != n:\n        raise ValueError("special must have the same length as token_bytes")\n    keep = [t for t in range(n) if not special[t]]\n    tb = [token_bytes[t] for t in keep]\n\n    full = b"".join(tb)\n    valid = _valid_utf8_prefix_len(full)\n    text = full[:valid].decode("utf-8")\n    full_total = len(full)\n\n    byte2char, pos = {}, 0\n    char_of_byte = np.zeros(full_total + 1, dtype=np.int64)  # char containing byte b\n    for ci, ch in enumerate(text):\n        byte2char[pos] = ci\n        nb = len(ch.encode("utf-8"))\n        char_of_byte[pos:pos + nb] = ci\n        pos += nb\n    byte2char[pos] = len(text)\n    char_of_byte[pos:] = len(text)  # trailing partial char maps past the text\n    aks = {0} | {e for _, e in akshara_units(text)}\n\n    def cat(b: int) -> str:\n        if b >= full_total:\n            return END\n        if b >= valid:  # inside the trailing truncated char\n            return MIDUTF8\n        return classify(b, valid, byte2char, text, aks)\n\n    ends = list(accumulate(len(b) for b in tb))\n    starts = [0] + ends[:-1]\n\n    end_cut = np.full(n, END, dtype=object)\n    start_cut = np.full(n, END, dtype=object)\n    char_span = np.full((n, 2), len(text), dtype=np.int64)\n    for j, t in enumerate(keep):\n        s, e = starts[j], ends[j]\n        end_cut[t] = cat(e)\n        start_cut[t] = cat(s)\n        cs = int(char_of_byte[min(s, full_total)])\n        ce = int(char_of_byte[e - 1]) + 1 if e > s else cs\n        char_span[t] = (cs, min(ce, len(text)) if e <= valid else len(text))\n\n    word_initial = np.array([sc in (WORD, END) for sc in start_cut], dtype=bool)\n    pos_in_word = np.zeros(n, dtype=np.int64)\n    word_id = np.zeros(n, dtype=np.int64)\n    wid = -1\n    for t in range(n):\n        if word_initial[t] or t == 0:\n            wid += 1\n            pos_in_word[t] = 0\n        else:\n            pos_in_word[t] = pos_in_word[t - 1] + 1\n        word_id[t] = wid\n    return dict(end_cut=end_cut, start_cut=start_cut, word_initial=word_initial,\n                pos_in_word=pos_in_word, word_id=word_id, char_span=char_span)\n\n\n# --------------------------------------------------------------------------- #\n# corpus analysis\n# --------------------------------------------------------------------------- #\n\n\ndef analyze(token_bytes_per_sentence: Iterable[Sequence[bytes]], ks: Sequence[int]):\n    """Corpus statistics of token cuts and of fixed-k draft windows. Returns (tot, per_k)."""\n    tot = Counter()\n    per_k = {k: Counter() for k in ks}\n    for tb in token_bytes_per_sentence:\n        f = sentence_features(tb)\n        if f is None:\n            tot["skipped_invalid_utf8"] += 1\n            continue\n        n = len(tb)\n        if n < 2:\n            tot["skipped_short"] += 1\n            continue\n        text, cut, start = f["text"], f["cut"], f["start"]\n        tot["sentences"] += 1\n        tot["tokens"] += n\n        tot["words"] += len(text.split())\n        tot["chars"] += len(text)\n        tot["aksharas"] += len(f["units"])\n        tot["starts_mid_utf8"] += sum(1 for s in start if s == MIDUTF8)\n        tot["starts_mid_akshara"] += sum(1 for s in start if s == MIDAKS)\n        for j in range(n - 1):  # internal cuts only (a token follows)\n            tot["cuts"] += 1\n            tot["cut_" + cut[j]] += 1\n\n        wb = [c in (WORD, END) for c in cut]\n        pw = [0] + list(accumulate(1 if x else 0 for x in wb))\n        last_wb, lw = [], -1\n        for j in range(n):\n            if wb[j]:\n                lw = j\n            last_wb.append(lw)\n        next_wb, nw = [0] * n, n - 1\n        for j in range(n - 1, -1, -1):\n            if wb[j]:\n                nw = j\n            next_wb[j] = nw\n\n        for k in ks:\n            c = per_k[k]\n            for s in range(0, n - k):\n                e = s + k - 1\n                cat = cut[e]\n                c["windows"] += 1\n                c[cat] += 1\n                done = pw[e + 1] - pw[s]\n                c["completed_words"] += done\n                if done == 0:\n                    c["zero_word"] += 1\n                if cat not in (WORD, END):\n                    c["mid_n"] += 1\n                    c["fwd_sum"] += next_wb[e] - e\n                    c["back_sum"] += e - last_wb[e]\n    return tot, per_k\n\n\ndef pct(a: float, b: float) -> float:\n    return 100.0 * a / b if b else float("nan")\n\n\ndef summarize(name: str, tot: Counter, per_k: dict, ks: Sequence[int]) -> dict:\n    """Turn ``analyze`` counters into the Test 1 summary dict."""\n    S = {\n        "tokenizer": name,\n        "sentences": tot["sentences"],\n        "tokens": tot["tokens"],\n        "tokens_per_word": tot["tokens"] / max(tot["words"], 1),\n        "tokens_per_akshara": tot["tokens"] / max(tot["aksharas"], 1),\n        "tokens_per_char": tot["tokens"] / max(tot["chars"], 1),\n        "pct_tokens_start_mid_akshara": pct(tot["starts_mid_akshara"], tot["tokens"]),\n        "pct_tokens_start_mid_utf8": pct(tot["starts_mid_utf8"], tot["tokens"]),\n        "pct_cuts_word_boundary": pct(tot["cut_" + WORD] + tot["cut_" + END], tot["cuts"]),\n        "pct_cuts_mid_word": pct(tot["cut_" + MIDWORD], tot["cuts"]),\n        "pct_cuts_mid_akshara": pct(tot["cut_" + MIDAKS], tot["cuts"]),\n        "pct_cuts_mid_utf8": pct(tot["cut_" + MIDUTF8], tot["cuts"]),\n        "skipped_invalid_utf8": tot["skipped_invalid_utf8"],\n        "per_k": [],\n    }\n    for k in ks:\n        c = per_k[k]\n        w = c["windows"]\n        S["per_k"].append({\n            "k": k,\n            "windows": w,\n            "pct_end_at_word_boundary": pct(c[WORD] + c[END], w),\n            "pct_end_mid_word": pct(c[MIDWORD], w),\n            "pct_end_mid_akshara": pct(c[MIDAKS], w),\n            "pct_end_mid_utf8": pct(c[MIDUTF8], w),\n            "pct_end_anywhere_inside_word": 100.0 - pct(c[WORD] + c[END], w) if w else float("nan"),\n            "mean_words_completed": c["completed_words"] / w if w else float("nan"),\n            "pct_drafts_completing_zero_words": pct(c["zero_word"], w),\n            "mean_tokens_forward_to_boundary": c["fwd_sum"] / c["mid_n"] if c["mid_n"] else float("nan"),\n            "mean_tokens_back_to_boundary": c["back_sum"] / c["mid_n"] if c["mid_n"] else float("nan"),\n        })\n    return S\n\n\ndef print_summary(S: dict) -> None:\n    print(f"\\n=== {S[\'tokenizer\']} ===")\n    print(f"sentences={S[\'sentences\']}  tokens={S[\'tokens\']}  "\n          f"tokens/word={S[\'tokens_per_word\']:.2f}  tokens/akshara={S[\'tokens_per_akshara\']:.2f}  "\n          f"tokens/char={S[\'tokens_per_char\']:.2f}")\n    print(f"tokens that START mid-akshara: {S[\'pct_tokens_start_mid_akshara\']:.1f}%   "\n          f"mid-UTF-8: {S[\'pct_tokens_start_mid_utf8\']:.1f}%")\n    print(f"all token cuts -> word boundary {S[\'pct_cuts_word_boundary\']:.1f}% | mid-word {S[\'pct_cuts_mid_word\']:.1f}% | "\n          f"mid-akshara {S[\'pct_cuts_mid_akshara\']:.1f}% | mid-UTF-8 {S[\'pct_cuts_mid_utf8\']:.1f}%")\n    if S["skipped_invalid_utf8"]:\n        print(f"WARNING: skipped {S[\'skipped_invalid_utf8\']} sentences (token bytes not valid UTF-8)")\n    hdr = ("k", "@word", "mid-word", "mid-aks", "mid-utf8", "inside-any", "words/draft", "0-word%", "fwd", "back")\n    print("\\n" + " ".join(f"{h:>11}" for h in hdr))\n    for r in S["per_k"]:\n        print(" ".join(f"{v:>11}" for v in (\n            r["k"], f"{r[\'pct_end_at_word_boundary\']:.1f}", f"{r[\'pct_end_mid_word\']:.1f}",\n            f"{r[\'pct_end_mid_akshara\']:.1f}", f"{r[\'pct_end_mid_utf8\']:.1f}",\n            f"{r[\'pct_end_anywhere_inside_word\']:.1f}", f"{r[\'mean_words_completed\']:.2f}",\n            f"{r[\'pct_drafts_completing_zero_words\']:.1f}",\n            f"{r[\'mean_tokens_forward_to_boundary\']:.2f}", f"{r[\'mean_tokens_back_to_boundary\']:.2f}")))\n    print("(columns 2-6 are % of k-token drafts; fwd/back = avg tokens to move to the nearest word boundary "\n          "when the draft ends inside a word)")\n\n\n# --------------------------------------------------------------------------- #\n# examples for slides\n# --------------------------------------------------------------------------- #\n\n\ndef _show(b: bytes) -> str:\n    try:\n        return b.decode("utf-8").replace(" ", "␣")  # visible space\n    except UnicodeDecodeError:\n        return "<" + b.hex() + ">"\n\n\ndef render_example(tb: Sequence[bytes], k: int) -> str:\n    """Render a tokenised sentence with fixed-k draft ends tagged by cut category."""\n    f = sentence_features(tb)\n    if f is None:\n        return "(invalid utf-8)"\n    out = []\n    for i, b in enumerate(tb):\n        out.append(_show(b))\n        if i < len(tb) - 1:\n            out.append(f" ‖[{TAG[f[\'cut\'][i]]}] " if (i + 1) % k == 0 else " | ")\n    return "".join(out)\n\n\nEXAMPLE_LEGEND = ("Legend: | token boundary,  ‖ end of a fixed-k draft,  tag = what the draft ends on: "\n                  "W=word boundary, w=inside word, a=inside akshara, u=inside UTF-8 char, ␣ = space\\n")\n\n# --------------------------------------------------------------------------- #\n# data\n# --------------------------------------------------------------------------- #\n\nDEMO = [\n    "भारत एक विशाल और विविधताओं से भरा देश है।",\n    "विद्यार्थियों ने विद्यालय में सूर्योदय के समय प्रार्थना की।",\n    "मौसम विभाग ने कल भारी बारिश की चेतावनी जारी की है।",\n    "वैज्ञानिकों का मानना है कि जलवायु परिवर्तन का प्रभाव कृषि पर पड़ेगा।",\n    "उसने अपने दोस्तों को खाने पर बुलाया और सबने मिलकर खाना खाया।",\n]\n\n\ndef load_sentences(path: str, field: str) -> list[str]:\n    out = []\n    with open(path, encoding="utf-8") as fh:\n        if path.endswith(".jsonl"):\n            for line in fh:\n                if line.strip():\n                    out.append(json.loads(line)[field])\n        else:\n            out = [ln.rstrip("\\n") for ln in fh]\n    return [unicodedata.normalize("NFC", s.strip()) for s in out if s.strip()]\n\n\ndef load_flores_hindi(split: str = "devtest") -> list[str]:\n    """FLORES+ hin_Deva sentences (NFC, stripped), as used in Test 1."""\n    from datasets import load_dataset\n    ds = load_dataset("openlanguagedata/flores_plus", "hin_Deva", split=split)\n    return [unicodedata.normalize("NFC", s.strip()) for s in ds["text"] if s.strip()]\n',
    'vocab': '"""Boolean masks over the full vocabulary, built from token *bytes* (``boundaries.token_to_bytes``).\n\nMasks (all shape ``(len(tok),)``, dtype bool):\n\n- ``starts_space``: first byte in ``b" \\\\n\\\\t"``.\n- ``starts_punct``: first decoded char has Unicode category ``P*`` (``।``, ``,``, ``.``, ``"``, ...).\n- ``starts_mark``: first decoded char has category ``M*`` (matra, virama, nukta, anusvara,\n  candrabindu, visarga, ...) or is ZWJ / ZWNJ.\n- ``starts_cont``: first byte is a UTF-8 continuation byte (0x80-0xBF): the token starts mid-character.\n  Such a token is never ``starts_mark`` / ``starts_punct``.\n- ``is_special``: special / control tokens (all registered special tokens and eos ids).\n\nA token whose bytes start with a lead byte but do not contain a complete first character\n(e.g. ``b"\\\\xe0\\\\xa4"``) has an unknown first char and is in none of space/punct/mark/cont.\nSpecial tokens are only ``is_special``.\n"""\nfrom __future__ import annotations\n\nimport unicodedata\nfrom dataclasses import dataclass, field\nfrom typing import Iterable\n\nimport numpy as np\n\nfrom .boundaries import token_to_bytes\n\nZW = "\u200c\u200d"\nMASK_NAMES = ("starts_space", "starts_punct", "starts_mark", "starts_cont", "is_special")\n\n\ndef first_char(b: bytes) -> str:\n    """First complete character of ``b`` (\'\' if none). Continuation-byte starts give \'\'."""\n    if not b or 0x80 <= b[0] <= 0xBF:\n        return ""\n    return b[:4].decode("utf-8", errors="ignore")[:1]\n\n\ndef classify_token_bytes(b: bytes) -> dict[str, bool]:\n    """Mask flags for one non-special token given its raw bytes."""\n    out = dict(starts_space=False, starts_punct=False, starts_mark=False, starts_cont=False,\n               is_special=False)\n    if not b:\n        return out\n    if b[0] in b" \\n\\t":\n        out["starts_space"] = True\n        return out\n    if 0x80 <= b[0] <= 0xBF:\n        out["starts_cont"] = True\n        return out\n    ch = first_char(b)\n    if not ch:\n        return out\n    cat = unicodedata.category(ch)\n    if cat.startswith("P"):\n        out["starts_punct"] = True\n    elif cat.startswith("M") or ch in ZW:\n        out["starts_mark"] = True\n    return out\n\n\n@dataclass\nclass VocabMasks:\n    starts_space: np.ndarray\n    starts_punct: np.ndarray\n    starts_mark: np.ndarray\n    starts_cont: np.ndarray\n    is_special: np.ndarray\n    eos_ids: list[int] = field(default_factory=list)\n\n    @property\n    def size(self) -> int:\n        return len(self.starts_space)\n\n    @property\n    def is_eos(self) -> np.ndarray:\n        m = np.zeros(self.size, dtype=bool)\n        m[[i for i in self.eos_ids if i < self.size]] = True\n        return m\n\n    @property\n    def starts_newword(self) -> np.ndarray:\n        """Tokens that end the current word if they come next: space, punctuation or eos."""\n        return self.starts_space | self.starts_punct | self.is_eos\n\n    def padded(self, name: str, size: int) -> np.ndarray:\n        """Mask padded with False to ``size`` (model logits can be wider than ``len(tok)``)."""\n        m = getattr(self, name)\n        if size < len(m):\n            raise ValueError("size smaller than vocabulary")\n        return np.concatenate([m, np.zeros(size - len(m), dtype=bool)])\n\n    def as_dict(self) -> dict[str, np.ndarray]:\n        return {k: getattr(self, k) for k in MASK_NAMES}\n\n\ndef special_ids(tok) -> set[int]:\n    """All special/control token ids of a Hugging Face tokenizer."""\n    ids = set(getattr(tok, "all_special_ids", []) or [])\n    for i, at in (getattr(tok, "added_tokens_decoder", {}) or {}).items():\n        if getattr(at, "special", False):\n            ids.add(int(i))\n    return ids\n\n\ndef build_masks(tok, eos_ids: Iterable[int] = ()) -> VocabMasks:\n    """Build all masks for a Hugging Face tokenizer (size ``len(tok)``)."""\n    n = len(tok)\n    eos_ids = sorted(set(int(i) for i in eos_ids))\n    spec = special_ids(tok) | set(eos_ids)\n    toks = tok.convert_ids_to_tokens(list(range(n)))\n    arrs = {k: np.zeros(n, dtype=bool) for k in MASK_NAMES}\n    for i, t in enumerate(toks):\n        if i in spec or t is None:\n            arrs["is_special"][i] = True\n            continue\n        for k, v in classify_token_bytes(token_to_bytes(t)).items():\n            if v:\n                arrs[k][i] = True\n    return VocabMasks(**arrs, eos_ids=eos_ids)\n\n\ndef token_bytes_list(tok, ids) -> list[bytes]:\n    """Raw bytes of each id (special tokens give their literal text bytes)."""\n    return [token_to_bytes(t) for t in tok.convert_ids_to_tokens([int(i) for i in ids])]\n',
    'traces': '"""Trace schema (one target sentence), compressed ``.npz`` save/load, gold-label attachment.\n\nIndex convention (PLAN §3): index ``t`` refers to target token ``y[t]``; every draft feature at\n``t`` is a function of ``q_t``, the draft distribution over ``y[t]`` computed teacher-forced on\n``prompt + y[:t]``.\n\nStorage: per-position arrays of all sentences are concatenated into flat arrays and stored with\nan ``offsets`` array (sentence i occupies ``offsets[i]:offsets[i+1]``). ``meta`` is stored as a\nJSON string.\n"""\nfrom __future__ import annotations\n\nimport json\nfrom dataclasses import dataclass, field, fields\nfrom pathlib import Path\n\nimport numpy as np\n\n# per-position 1-D arrays saved to disk (in this order)\nSCALAR_FIELDS = ("target_ids", "draft_argmax", "H", "top1", "top2", "p_target",\n                 "p_space", "p_punct", "p_eos", "p_newword", "p_mark", "p_cont",\n                 "target_logp_of_draft_argmax")\n# per-position (n, K) arrays\nTOPK_FIELDS = ("draft_topk_ids", "draft_topk_logp", "target_topk_ids", "target_topk_logp")\nINT_FIELDS = {"target_ids", "draft_argmax", "draft_topk_ids", "target_topk_ids"}\nTOPK = 20\n\n\n@dataclass\nclass Trace:\n    """One sentence. All arrays have first dimension ``n``."""\n    id: int\n    n: int\n    target_ids: np.ndarray\n    draft_argmax: np.ndarray\n    H: np.ndarray            # draft entropy (nats)\n    top1: np.ndarray         # draft prob of its argmax\n    top2: np.ndarray         # draft prob of its 2nd-best token\n    p_target: np.ndarray     # q_t(y[t])\n    p_space: np.ndarray\n    p_punct: np.ndarray\n    p_eos: np.ndarray\n    p_newword: np.ndarray    # p_space + p_punct + p_eos\n    p_mark: np.ndarray\n    p_cont: np.ndarray\n    draft_topk_ids: np.ndarray\n    draft_topk_logp: np.ndarray\n    target_topk_ids: np.ndarray\n    target_topk_logp: np.ndarray\n    target_logp_of_draft_argmax: np.ndarray\n    match: np.ndarray = None  # derived: draft_argmax == target_ids\n    labels: dict = field(default_factory=dict, repr=False)  # gold labels (analysis only, not saved)\n\n    def __post_init__(self):\n        self.match = np.asarray(self.draft_argmax) == np.asarray(self.target_ids)\n        for f in SCALAR_FIELDS + TOPK_FIELDS:\n            a = getattr(self, f)\n            if len(a) != self.n:\n                raise ValueError(f"trace {self.id}: {f} has length {len(a)} != n={self.n}")\n\n\ndef save_traces(path, traces: list[Trace], meta: dict | None = None) -> None:\n    """Save traces as one compressed ``.npz`` (flat arrays + offsets)."""\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    lens = np.array([t.n for t in traces], dtype=np.int64)\n    offsets = np.concatenate([[0], np.cumsum(lens)]).astype(np.int64)\n    out = dict(ids=np.array([t.id for t in traces], dtype=np.int64), offsets=offsets,\n               meta=np.array(json.dumps(meta or {}, ensure_ascii=False)))\n    for f in SCALAR_FIELDS + TOPK_FIELDS:\n        dt = np.int32 if f in INT_FIELDS else np.float32\n        parts = [np.asarray(getattr(t, f), dtype=dt) for t in traces]\n        if parts:\n            out[f] = np.concatenate(parts, axis=0)\n        else:\n            out[f] = np.zeros((0, TOPK) if f in TOPK_FIELDS else (0,), dtype=dt)\n    np.savez_compressed(path, **out)\n\n\ndef load_traces(path) -> tuple[list[Trace], dict]:\n    """Inverse of ``save_traces``."""\n    z = np.load(path, allow_pickle=False)\n    meta = json.loads(str(z["meta"]))\n    offsets, ids = z["offsets"], z["ids"]\n    flat = {f: z[f] for f in SCALAR_FIELDS + TOPK_FIELDS}\n    traces = []\n    for i, sid in enumerate(ids):\n        a, b = int(offsets[i]), int(offsets[i + 1])\n        traces.append(Trace(id=int(sid), n=b - a, **{f: flat[f][a:b] for f in flat}))\n    return traces, meta\n\n\ndef summary(traces: list[Trace]) -> dict:\n    """Positions, agreement rate, mean draft entropy when match / mismatch."""\n    m = np.concatenate([t.match for t in traces])\n    H = np.concatenate([t.H for t in traces])\n    return dict(sentences=len(traces), positions=int(len(m)), agreement=float(m.mean()),\n                mean_H=float(H.mean()), mean_H_match=float(H[m].mean()) if m.any() else float("nan"),\n                mean_H_mismatch=float(H[~m].mean()) if (~m).any() else float("nan"))\n\n\ndef attach_word_labels(traces: list[Trace], tok) -> None:\n    """Attach gold word/akshara labels from the *target* tokens (fixes B5). Analysis only."""\n    from .boundaries import token_labels\n    from .vocab import special_ids, token_bytes_list\n    from .boundaries import _valid_utf8_prefix_len\n    spec = special_ids(tok)\n    for t in traces:\n        tb = token_bytes_list(tok, t.target_ids)\n        sp = [int(i) in spec for i in t.target_ids]\n        t.labels.update(token_labels(tb, special=sp))\n        full = b"".join(b for b, s in zip(tb, sp) if not s)\n        t.labels["text"] = full[:_valid_utf8_prefix_len(full)].decode("utf-8")  # text the char spans refer to\n\n\ndef trace_field_names() -> list[str]:\n    return [f.name for f in fields(Trace)]\n',
    'gpu_traces': '"""Target generation, teacher-forced draft/target passes and timing (Kaggle notebook 01).\n\nPorted from the legacy Part A (``notebooks/legacy/day2-day3-llama8b-kaggle.ipynb`` cells 4-11).\nEverything here also runs on CPU (``scripts/smoke_cpu.py``) with a small model in fp32.\n\nOutputs (in ``out_dir``) for one split:\n  target_outputs_{split}.jsonl   id, src, ref, prompt_ids, out_ids, out_text   (append + resume)\n  _draft_tf_{split}.pkl          partial draft teacher-forced features         (resume)\n  _target_tf_{split}.pkl         partial target teacher-forced features        (resume)\n  traces_{split}.npz             final traces (``traces.save_traces``)\n  timing.json                    t_draft, t_target, c, verify(g)               (skipped if present)\n  run_info_{split}.json\n"""\nfrom __future__ import annotations\n\nimport gc\nimport json\nimport os\nimport pickle\nimport platform\nimport shutil\nimport time\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .traces import TOPK, Trace, save_traces, summary\nfrom .vocab import VocabMasks, build_masks\n\nPROMPT_TEMPLATE = "Translate the following English sentence into Hindi. Output only the Hindi translation.\\n\\n{s}"\n\n\n# --------------------------------------------------------------------------- #\n# loading\n# --------------------------------------------------------------------------- #\n\ndef _torch():\n    import torch\n    return torch\n\n\ndef sync_all() -> None:\n    torch = _torch()\n    if torch.cuda.is_available():\n        for i in range(torch.cuda.device_count()):\n            torch.cuda.synchronize(i)\n\n\ndef gpu_report(tag: str = "") -> None:\n    torch = _torch()\n    if torch.cuda.is_available():\n        print(tag, " | ".join(f"GPU{i}: {torch.cuda.memory_allocated(i) / 2**30:.1f} GiB allocated"\n                              for i in range(torch.cuda.device_count())))\n\n\ndef from_pretrained(name: str, dtype, **kw):\n    """``AutoModelForCausalLM.from_pretrained`` across transformers versions (dtype vs torch_dtype)."""\n    from transformers import AutoModelForCausalLM\n    try:\n        return AutoModelForCausalLM.from_pretrained(name, dtype=dtype, **kw).eval()\n    except TypeError:\n        return AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype, **kw).eval()\n\n\ndef load_tokenizer(name: str):\n    from transformers import AutoTokenizer\n    tok = AutoTokenizer.from_pretrained(name)\n    tok.padding_side = "left"\n    if tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    return tok\n\n\ndef load_target(name: str, dtype, max_memory: dict | None = None, device_map="auto"):\n    """Target model, layer-split over the available GPUs (``device_map=\'auto\'``) or on CPU."""\n    torch = _torch()\n    if torch.cuda.is_available():\n        m = from_pretrained(name, dtype, device_map=device_map, max_memory=max_memory,\n                            low_cpu_mem_usage=True)\n        print("modules per device:", dict(Counter(m.hf_device_map.values())))\n    else:\n        m = from_pretrained(name, dtype)\n    return m\n\n\ndef load_draft(name: str, dtype, device: str):\n    return from_pretrained(name, dtype).to(device)\n\n\ndef get_eos_ids(model, tok) -> list[int]:\n    """Stop ids: generation-config eos plus ``<|eot_id|>`` (Llama-3 instruct end of turn)."""\n    eos = model.generation_config.eos_token_id\n    ids = set(eos if isinstance(eos, list) else [eos])\n    eot = tok.convert_tokens_to_ids("<|eot_id|>")\n    if eot is not None and eot != tok.unk_token_id:\n        ids.add(eot)\n    return sorted(int(i) for i in ids if i is not None)\n\n\ndef check_draft_target_compat(draft, target, tok_d, tok_t, probe: str) -> None:\n    """Draft and target must share a vocabulary (legacy asserts, kept)."""\n    assert draft.config.vocab_size == target.config.vocab_size, "vocab sizes differ"\n    probe = probe + " नमस्ते दुनिया. Hello world!"\n    assert (tok_d(probe, add_special_tokens=False).input_ids\n            == tok_t(probe, add_special_tokens=False).input_ids), "tokenizers disagree"\n    print("draft/target tokenizers match")\n\n\n# --------------------------------------------------------------------------- #\n# data + prompts\n# --------------------------------------------------------------------------- #\n\ndef load_flores_pairs(split: str, n: int | None = None):\n    """FLORES+ eng_Latn -> hin_Deva rows ``(id, src, ref)`` for ``split`` in {dev, devtest}."""\n    from datasets import load_dataset\n    assert split in ("dev", "devtest"), split\n    en = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split=split)\n    hi = load_dataset("openlanguagedata/flores_plus", "hin_Deva", split=split)\n    hi_by_id = {r["id"]: r["text"] for r in hi}\n    rows = [(int(r["id"]), r["text"], hi_by_id[r["id"]]) for r in en]\n    return rows[:n] if n else rows\n\n\ndef build_prompt(tok, s: str) -> str:\n    msgs = [{"role": "user", "content": PROMPT_TEMPLATE.format(s=s)}]\n    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)\n\n\ndef read_jsonl(path) -> list[dict]:\n    path = Path(path)\n    if not path.exists():\n        return []\n    with open(path, encoding="utf-8") as fh:\n        return [json.loads(l) for l in fh if l.strip()]\n\n\n# --------------------------------------------------------------------------- #\n# 1. batched greedy target generation (resumable)\n# --------------------------------------------------------------------------- #\n\ndef generate_targets(target, tok, rows, out_jsonl, batch: int, max_new: int, eos_ids,\n                     device: str, progress: bool = True) -> list[dict]:\n    """Greedy, batched (left padding) target outputs, appended to ``out_jsonl`` after every batch.\n\n    Resumes if ``out_jsonl`` holds a prefix of ``rows`` (by id); otherwise starts over.\n    """\n    torch = _torch()\n    from tqdm.auto import tqdm\n    out_jsonl = Path(out_jsonl)\n    ids = [r[0] for r in rows]\n    prompts = [build_prompt(tok, r[1]) for r in rows]\n    records = read_jsonl(out_jsonl)\n    if [r["id"] for r in records] != ids[:len(records)]:\n        print("existing output does not match this run; starting over")\n        records = []\n    print(f"resuming: {len(records)} / {len(rows)} sentences already done")\n    eos_set = set(eos_ids)\n    with open(out_jsonl, "a" if records else "w", encoding="utf-8") as f:\n        rng = range(len(records), len(prompts), batch)\n        for i in (tqdm(rng, desc="generate") if progress else rng):\n            enc = tok(prompts[i:i + batch], return_tensors="pt", padding=True,\n                      add_special_tokens=False).to(device)\n            with torch.no_grad():\n                out = target.generate(**enc, do_sample=False, temperature=None, top_p=None,\n                                      max_new_tokens=max_new, eos_token_id=list(eos_set),\n                                      pad_token_id=tok.pad_token_id)\n            gen = out[:, enc.input_ids.shape[1]:].tolist()\n            for j, g in enumerate(gen):\n                cut = next((p + 1 for p, t in enumerate(g) if t in eos_set), len(g))  # keep eos\n                g = g[:cut]\n                pid = enc.input_ids[j][enc.attention_mask[j].bool()].tolist()      # unpadded prompt\n                rid, src, ref = rows[i + j]\n                rec = dict(id=rid, src=src, ref=ref, prompt_ids=pid, out_ids=g,\n                           out_text=tok.decode(g, skip_special_tokens=True).strip())\n                records.append(rec)\n                f.write(json.dumps(rec, ensure_ascii=False) + "\\n")\n            f.flush()\n    n_hit = sum(len(r["out_ids"]) >= max_new for r in records)\n    print(f"{len(records)} target outputs in {out_jsonl.name}; {n_hit} hit MAX_NEW={max_new}")\n    return records\n\n\ndef generate_unbatched(model, tok, prompt_ids: list[int], max_new: int, eos_ids, device: str) -> list[int]:\n    """Plain greedy generation for one prompt, no padding (reference for losslessness, B9)."""\n    torch = _torch()\n    x = torch.tensor([prompt_ids], device=device)\n    with torch.no_grad():\n        out = model.generate(input_ids=x, attention_mask=torch.ones_like(x), do_sample=False,\n                             temperature=None, top_p=None, max_new_tokens=max_new,\n                             eos_token_id=list(eos_ids), pad_token_id=tok.pad_token_id)\n    g = out[0, len(prompt_ids):].tolist()\n    eos_set = set(eos_ids)\n    cut = next((p + 1 for p, t in enumerate(g) if t in eos_set), len(g))\n    return g[:cut]\n\n\n# --------------------------------------------------------------------------- #\n# 2./3. teacher-forced passes\n# --------------------------------------------------------------------------- #\n\ndef _mask_matrix(masks: VocabMasks, vocab_size: int, device):\n    """(V, 5) float32 matrix: columns space, punct, eos, mark, cont."""\n    torch = _torch()\n    cols = [masks.padded("starts_space", vocab_size), masks.padded("starts_punct", vocab_size),\n            np.concatenate([masks.is_eos, np.zeros(vocab_size - masks.size, bool)]),\n            masks.padded("starts_mark", vocab_size), masks.padded("starts_cont", vocab_size)]\n    return torch.tensor(np.stack(cols, 1).astype(np.float32), device=device)\n\n\ndef _logits_for_output(model, prompt_ids, out_ids, device):\n    """Logits (float32) predicting each output token: row t is q_t (conditioned on prompt + y[:t])."""\n    torch = _torch()\n    x = torch.tensor([list(prompt_ids) + list(out_ids)], device=device)\n    with torch.no_grad():\n        lg = model(x).logits[0, len(prompt_ids) - 1:-1]\n    return lg.float()\n\n\ndef draft_features(draft, rec: dict, M, device: str, topk: int = TOPK) -> dict[str, np.ndarray]:\n    """Per-position draft features for one record (all float32 on host)."""\n    torch = _torch()\n    lg = _logits_for_output(draft, rec["prompt_ids"], rec["out_ids"], device)\n    lp = torch.log_softmax(lg, -1)\n    p = lp.exp()\n    H = -(p * lp).sum(-1)\n    t2 = p.topk(2, -1).values\n    tk = lp.topk(topk, -1)\n    y = torch.tensor(rec["out_ids"], device=lg.device)\n    P = p @ M.to(lg.device)                       # (n, 5): space, punct, eos, mark, cont\n    f = dict(draft_argmax=lp.argmax(-1), H=H, top1=t2[:, 0], top2=t2[:, 1],\n             p_target=p.gather(1, y[:, None]).squeeze(1),\n             p_space=P[:, 0], p_punct=P[:, 1], p_eos=P[:, 2], p_newword=P[:, 0] + P[:, 1] + P[:, 2],\n             p_mark=P[:, 3], p_cont=P[:, 4], draft_topk_ids=tk.indices, draft_topk_logp=tk.values)\n    return {k: v.cpu().numpy() for k, v in f.items()}\n\n\ndef target_features(target, rec: dict, draft_argmax: np.ndarray, device: str,\n                    topk: int = TOPK) -> dict[str, np.ndarray]:\n    torch = _torch()\n    lg = _logits_for_output(target, rec["prompt_ids"], rec["out_ids"], device)\n    lp = torch.log_softmax(lg, -1)\n    tk = lp.topk(topk, -1)\n    da = torch.tensor(np.asarray(draft_argmax), device=lg.device, dtype=torch.long)\n    f = dict(target_topk_ids=tk.indices, target_topk_logp=tk.values,\n             target_logp_of_draft_argmax=lp.gather(1, da[:, None]).squeeze(1))\n    return {k: v.cpu().numpy() for k, v in f.items()}\n\n\ndef _resumable_pass(records, fn, partial_path, desc: str, save_every: int = 50, progress=True):\n    """Run ``fn(rec)`` for every record, pickling partial results so a dead session can resume."""\n    from tqdm.auto import tqdm\n    partial_path = Path(partial_path)\n    done: dict = {}\n    if partial_path.exists():\n        with open(partial_path, "rb") as fh:\n            done = pickle.load(fh)\n        print(f"{desc}: resuming with {len(done)} / {len(records)} done")\n    it = tqdm(records, desc=desc) if progress else records\n    since = 0\n    for rec in it:\n        if rec["id"] in done:\n            continue\n        done[rec["id"]] = fn(rec)\n        since += 1\n        if since >= save_every:\n            _atomic_pickle(done, partial_path)\n            since = 0\n    _atomic_pickle(done, partial_path)\n    return done\n\n\ndef _atomic_pickle(obj, path: Path) -> None:\n    tmp = path.with_suffix(".tmp")\n    with open(tmp, "wb") as fh:\n        pickle.dump(obj, fh, protocol=pickle.HIGHEST_PROTOCOL)\n    os.replace(tmp, path)\n\n\ndef run_draft_pass(draft, records, masks: VocabMasks, out_dir, split: str, device: str, progress=True):\n    vocab = draft.get_output_embeddings().weight.shape[0]\n    M = _mask_matrix(masks, vocab, device)\n    return _resumable_pass(records, lambda r: draft_features(draft, r, M, device),\n                           Path(out_dir) / f"_draft_tf_{split}.pkl", "draft teacher-forced", progress=progress)\n\n\ndef run_target_pass(target, records, draft_feats: dict, out_dir, split: str, device: str, progress=True):\n    return _resumable_pass(records, lambda r: target_features(target, r, draft_feats[r["id"]]["draft_argmax"], device),\n                           Path(out_dir) / f"_target_tf_{split}.pkl", "target teacher-forced", progress=progress)\n\n\ndef assemble_traces(records, draft_feats: dict, target_feats: dict) -> list[Trace]:\n    out = []\n    for r in records:\n        d, t = draft_feats[r["id"]], target_feats[r["id"]]\n        out.append(Trace(id=int(r["id"]), n=len(r["out_ids"]), target_ids=np.asarray(r["out_ids"], np.int32),\n                         **d, **t))\n    return out\n\n\ndef target_argmax_check(traces: list[Trace]) -> dict:\n    """Rate at which the unpadded teacher-forced target argmax equals the generated y[t]."""\n    agree, diffs, total = 0, [], 0\n    for t in traces:\n        am = t.target_topk_ids[:, 0]\n        eq = am == t.target_ids\n        agree += int(eq.sum())\n        total += t.n\n        for pos in np.flatnonzero(~eq):\n            lp = t.target_topk_logp[pos]\n            diffs.append(dict(id=t.id, pos=int(pos), y=int(t.target_ids[pos]), argmax=int(am[pos]),\n                              margin=float(lp[0] - lp[1])))\n    return dict(rate=agree / max(total, 1), positions=total, n_diff=len(diffs), diffs=diffs)\n\n\n# --------------------------------------------------------------------------- #\n# 4. timing\n# --------------------------------------------------------------------------- #\n\ndef time_fwd(model, q: int = 1, ctx: int = 150, steps: int = 60, reps: int = 3, device: str = "cuda:0") -> float:\n    """Median latency (s) of a forward with ``q`` new tokens on top of a KV cache of ``ctx`` tokens."""\n    torch = _torch()\n    ts = []\n    with torch.no_grad():\n        for _ in range(reps):\n            x = torch.randint(1000, 20000, (1, ctx), device=device)\n            o = model(x, use_cache=True)\n            pkv = o.past_key_values\n            for s in range(steps + 10):\n                inp = torch.randint(1000, 20000, (1, q), device=device)\n                sync_all()\n                t = time.perf_counter()\n                o = model(inp, past_key_values=pkv, use_cache=True)\n                sync_all()\n                dt = time.perf_counter() - t\n                pkv = o.past_key_values\n                if s >= 10:\n                    ts.append(dt)\n    return float(np.median(ts))\n\n\ndef run_timing(draft, target, out_path, device: str, setup: dict, steps: int = 60, reps: int = 3,\n               max_g: int = 8) -> dict:\n    """Legacy timing (c and verify(g), g=1..max_g) plus draft q=1 at ctx 150 and 400."""\n    out_path = Path(out_path)\n    if out_path.exists():\n        print(f"{out_path.name} exists, timing skipped")\n        return json.loads(out_path.read_text())\n    t_d = time_fwd(draft, 1, steps=steps, reps=reps, device=device)\n    t_t = time_fwd(target, 1, steps=steps, reps=reps, device=device)\n    print(f"draft decode step: {t_d * 1000:.2f} ms | target decode step: {t_t * 1000:.2f} ms | c = {t_d / t_t:.3f}")\n    ver = {g: time_fwd(target, g + 1, steps=steps, reps=reps, device=device) for g in range(1, max_g + 1)}\n    print({g: f"{v * 1000:.1f} ms" for g, v in ver.items()})\n    extra = {str(ctx): time_fwd(draft, 1, ctx=ctx, steps=steps, reps=reps, device=device) for ctx in (150, 400)}\n    T = dict(t_draft=t_d, t_target=t_t, c=t_d / t_t, t_target_verify={str(g): v for g, v in ver.items()},\n             t_draft_by_ctx=extra, setup=setup)\n    out_path.write_text(json.dumps(T, indent=1))\n    return T\n\n\n# --------------------------------------------------------------------------- #\n# run info + zip\n# --------------------------------------------------------------------------- #\n\ndef run_info(git_hash: str, **extra) -> dict:\n    torch = _torch()\n    import transformers\n    gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []\n    return dict(git_hash=git_hash, torch=torch.__version__, transformers=transformers.__version__,\n                python=platform.python_version(), gpus=gpus, time=time.strftime("%Y-%m-%d %H:%M:%S"), **extra)\n\n\ndef zip_dir(src_dir, zip_path, include=None) -> str:\n    """Zip files of ``src_dir`` (only names in ``include`` if given; flat, no folder) to ``zip_path``.\n    Never deletes or moves anything in ``src_dir``. Returns the zip path."""\n    import zipfile\n    src_dir, zip_path = Path(src_dir), Path(zip_path)\n    names = list(include) if include is not None else sorted(p.name for p in src_dir.iterdir() if p.is_file())\n    missing = [n for n in names if not (src_dir / n).exists()]\n    if missing:\n        raise FileNotFoundError(f"cannot zip, missing in {src_dir}: {missing}")\n    tmp = zip_path.with_name(zip_path.name + ".part")\n    with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED) as z:\n        for n in names:\n            z.write(src_dir / n, arcname=n)\n    os.replace(tmp, zip_path)\n    return str(zip_path)\n\n\ndef free(*objs) -> None:\n    for o in objs:\n        del o\n    gc.collect()\n    torch = _torch()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\n# --------------------------------------------------------------------------- #\n# full pipeline for one split (used by notebook 01 and the smoke test)\n# --------------------------------------------------------------------------- #\n\ndef run_split(*, split: str, out_dir, target, draft, tok, tok_d, n: int | None, batch: int,\n              max_new: int, device: str, git_hash: str, timing_kw: dict | None = None,\n              setup_extra: dict | None = None, rows=None, progress: bool = True,\n              zip_name: str | None = None) -> dict:\n    """Steps 1-6 of PLAN T2 for one split. Returns a dict with paths and summaries.\n\n    timing_kw: passed to ``run_timing`` (steps, reps, max_g); setup_extra: added to timing.json setup\n    (e.g. max_memory, GPU names).\n    """\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    rows = rows if rows is not None else load_flores_pairs(split, n)\n    eos_ids = get_eos_ids(target, tok)\n    print(f"{split}: {len(rows)} sentences; eos ids {eos_ids}")\n\n    records = generate_targets(target, tok, rows, out_dir / f"target_outputs_{split}.jsonl",\n                               batch, max_new, eos_ids, device, progress=progress)\n    check_draft_target_compat(draft, target, tok_d, tok, records[0]["out_text"])\n\n    masks = build_masks(tok, eos_ids)\n    dfe = run_draft_pass(draft, records, masks, out_dir, split, device, progress=progress)\n    tfe = run_target_pass(target, records, dfe, out_dir, split, device, progress=progress)\n    traces = assemble_traces(records, dfe, tfe)\n    chk = target_argmax_check(traces)\n    print(f"target teacher-forced argmax == y[t]: {chk[\'rate\']:.4f} ({chk[\'n_diff\']} positions differ)")\n    for d in chk["diffs"][:20]:\n        print("   ", d)\n    if chk["rate"] < 0.995:\n        print("WARNING: target argmax agreement below 0.995")\n\n    import transformers\n    meta = dict(split=split, target=getattr(target.config, "_name_or_path", ""),\n                draft=getattr(draft.config, "_name_or_path", ""), max_new=max_new, batch=batch,\n                eos_ids=eos_ids, git_hash=git_hash, transformers=transformers.__version__,\n                target_argmax_check={k: v for k, v in chk.items() if k != "diffs"},\n                target_argmax_diffs=chk["diffs"][:500])\n    npz = out_dir / f"traces_{split}.npz"\n    save_traces(npz, traces, meta)\n    S = summary(traces)\n    print(f"positions: {S[\'positions\']} | agreement: {S[\'agreement\']:.3f} | mean H: {S[\'mean_H\']:.3f} "\n          f"(match {S[\'mean_H_match\']:.3f}, mismatch {S[\'mean_H_mismatch\']:.3f})")\n\n    setup = dict(target=meta["target"], draft=meta["draft"], dtype=str(next(target.parameters()).dtype),\n                 device=device, **(setup_extra or {}))\n    T = run_timing(draft, target, out_dir / "timing.json", device, setup, **(timing_kw or {}))\n\n    info = run_info(git_hash, split=split, n_sentences=len(rows), summary=S,\n                    target=meta["target"], draft=meta["draft"])\n    (out_dir / f"run_info_{split}.json").write_text(json.dumps(info, indent=1, ensure_ascii=False))\n    zip_name = zip_name or f"traces_v2_{split}"\n    zp = zip_dir(out_dir, out_dir.parent / f"{zip_name}.zip",\n                 include=[npz.name, f"target_outputs_{split}.jsonl", "timing.json", f"run_info_{split}.json"])\n    return dict(zip=zp, npz=str(npz), summary=S, timing=T, target_check=chk)\n\n\n# --------------------------------------------------------------------------- #\n# notebook 01 entry point (also called by scripts/smoke_cpu.py)\n# --------------------------------------------------------------------------- #\n\nDEFAULT_CONFIG_01 = dict(\n    SPLIT="dev",                                    # "dev" or "devtest": one split per run\n    N=None,                                         # None = all sentences of the split\n    BATCH=16, MAX_NEW=256,\n    SMOKE=False,                                    # True: CPU, fp32, one model as target+draft\n    TARGET="meta-llama/Meta-Llama-3-8B-Instruct",\n    DRAFT="meta-llama/Llama-3.2-1B-Instruct",\n    MAX_MEMORY={0: "8GiB", 1: "10GiB"},\n    OUT_ROOT="/kaggle/working",\n    GIT_HASH="unknown",\n    TIMING=dict(steps=60, reps=3, max_g=8),\n    RESUME_FROM=None,   # folder with partial outputs of an earlier (dead) run, e.g. /kaggle/input/<...>\n)\n\n\ndef _prepare_out_dir(out_dir: Path, out_root: Path, resume_from) -> None:\n    """Copy partial outputs from ``resume_from`` (searched recursively) and reuse a timing.json\n    measured by another split\'s run in this session."""\n    out_dir.mkdir(parents=True, exist_ok=True)\n    if resume_from:\n        src = Path(resume_from)\n        for f in src.rglob("*"):\n            if f.is_file() and f.parent.name == out_dir.name and not (out_dir / f.name).exists():\n                shutil.copy2(f, out_dir / f.name)\n                print("resume: copied", f)\n    if not (out_dir / "timing.json").exists():\n        for other in sorted(out_root.glob("traces_v2_*/timing.json")):\n            shutil.copy2(other, out_dir / "timing.json")\n            print("reusing timing from", other)\n            break\n\n\ndef notebook01_main(cfg: dict) -> dict:\n    """Run PLAN T2 steps 1-6 for ``cfg[\'SPLIT\']``. Writes ``{OUT_ROOT}/traces_v2_{split}/`` and\n    ``{OUT_ROOT}/traces_v2_{split}.zip``. With ``SMOKE=True`` the draft model is used as both\n    target and draft (fp32, CPU, loaded once) and only 3 sentences are processed."""\n    torch = _torch()\n    cfg = {**DEFAULT_CONFIG_01, **cfg}\n    split = cfg["SPLIT"]\n    out_dir = Path(cfg["OUT_ROOT"]) / f"traces_v2_{split}"\n    _prepare_out_dir(out_dir, Path(cfg["OUT_ROOT"]), cfg["RESUME_FROM"])\n    if cfg["SMOKE"]:\n        device, dtype = "cpu", torch.float32\n        tok = load_tokenizer(cfg["DRAFT"])\n        draft = target = load_draft(cfg["DRAFT"], dtype, device)\n        tok_d = tok\n        n = cfg["N"] or 3\n        setup_extra = dict(smoke=True)\n    else:\n        assert torch.cuda.device_count() >= 2, "Need 2 GPUs: Settings > Accelerator > GPU T4 x2"\n        for i in range(torch.cuda.device_count()):\n            p = torch.cuda.get_device_properties(i)\n            print(f"GPU{i}: {p.name}, {p.total_memory / 2**30:.1f} GiB")\n        device, dtype = "cuda:0", torch.float16\n        tok = load_tokenizer(cfg["TARGET"])\n        tok_d = load_tokenizer(cfg["DRAFT"])\n        target = load_target(cfg["TARGET"], dtype, max_memory=cfg["MAX_MEMORY"])\n        gpu_report("after loading target:")\n        draft = load_draft(cfg["DRAFT"], dtype, device)\n        gpu_report("after loading draft:")\n        n = cfg["N"]\n        setup_extra = dict(max_memory={str(k): v for k, v in cfg["MAX_MEMORY"].items()},\n                           gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])\n    res = run_split(split=split, out_dir=out_dir, target=target, draft=draft, tok=tok, tok_d=tok_d,\n                    n=n, batch=cfg["BATCH"], max_new=cfg["MAX_NEW"], device=device,\n                    git_hash=cfg["GIT_HASH"], timing_kw=cfg["TIMING"], setup_extra=setup_extra)\n    print("\\n" + "=" * 70)\n    print(f"DONE. Download  {Path(res[\'zip\']).name}  from the notebook\'s Output tab")\n    print(f"and unzip it into  data/traces_v2/  in the repo (files land directly in that folder).")\n    print("=" * 70)\n    return res\n',
    'simulator': '"""Trace simulator for greedy speculative decoding (PLAN T4, fixes B3, B8).\n\nPorted from ``notebooks/legacy/day2-day3-llama8b-kaggle.ipynb`` (cells 14-17), with:\n  1. the ``Policy`` interface ``stop(view, p, g)`` called with g = 1..cap-1 after each drafted token\n     (dynamic policies always draft >= 1 token; NoDraft / FixedK(0) / Oracle decide g = 0 via\n     ``initial_stop(view, p)``);\n  2. a causality guard (``TraceView``) - see PLAN §3;\n  3. lookahead cost: a lookahead-1 policy that stops by itself pays one extra draft forward ``c``;\n  4. per-sentence emitted / cost arrays and round records;\n  5. measured verification cost as the default.\n\nExactness for greedy decoding: within a round the draft\'s own tokens coincide with the target\'s up\nto the first mismatch, and everything after it is rejected, so accepted = leading matches.\nOnly the stopping decision after a mismatch can differ from a real loop (teacher-forcing\napproximation, PLAN §3).\n"""\nfrom __future__ import annotations\n\nimport json\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Callable, Sequence\n\nimport numpy as np\n\nfrom .traces import Trace\n\nGAMMA_MAX = 8\n\n# trace fields a policy may read through ``view.feat`` (functions of q_t only)\nDRAFT_FEATURES = {"H", "top1", "top2", "p_space", "p_punct", "p_eos", "p_newword", "p_mark", "p_cont",\n                  "draft_argmax", "draft_topk_ids", "draft_topk_logp"}\n# never readable by non-oracle policies\nGOLD_FIELDS = {"match", "target_ids", "p_target", "target_topk_ids", "target_topk_logp",\n               "target_logp_of_draft_argmax"}\n\n\nclass CausalityError(RuntimeError):\n    """A policy read information a real decoder would not have at that moment."""\n\n\nclass TraceView:\n    """Read-only, causality-checked view of one trace for policies.\n\n    The simulator sets ``limit = p + g - 1 + policy.lookahead`` and ``p`` before each call.\n      feat(name, idx): draft feature at index idx (idx <= limit).\n      token(idx):      committed target token for idx < p, the draft\'s own token for p <= idx <= p+g-1\n                       (idx <= limit; for lookahead policies idx = p+g gives the draft argmax at p+g).\n      gold(name, idx): any field incl. match / target ids / labels; oracle policies only.\n    """\n\n    __slots__ = ("trace", "limit", "p", "is_oracle", "n", "_labels")\n\n    def __init__(self, trace: Trace, is_oracle: bool = False):\n        self.trace, self.is_oracle, self.n = trace, is_oracle, trace.n\n        self.limit = -1\n        self.p = 0\n        self._labels = trace.labels\n\n    def _check(self, idx: int) -> None:\n        if idx > self.limit and not self.is_oracle:\n            raise CausalityError(f"read index {idx} > limit {self.limit} (p={self.p})")\n        if idx < 0 or idx >= self.n:\n            raise IndexError(idx)\n\n    def feat(self, name: str, idx: int):\n        if name not in DRAFT_FEATURES and not self.is_oracle:\n            raise CausalityError(f"policy may not read \'{name}\' via feat()")\n        self._check(idx)\n        return getattr(self.trace, name)[idx]\n\n    def token(self, idx: int) -> int:\n        self._check(idx)\n        if idx < self.p:\n            return int(self.trace.target_ids[idx])\n        return int(self.trace.draft_argmax[idx])\n\n    def gold(self, name: str, idx: int):\n        if not self.is_oracle:\n            raise CausalityError(f"non-oracle policy read gold field \'{name}\'")\n        if name in self._labels:\n            return self._labels[name][idx]\n        return getattr(self.trace, name)[idx]\n\n\nclass Policy:\n    name: str = "policy"\n    lookahead: int = 0          # 0 or 1\n    is_oracle: bool = False\n\n    def initial_stop(self, view: TraceView, p: int) -> bool:\n        """Decision at g = 0 (before drafting; view.limit = p - 1). Only NoDraft / FixedK(0) and the\n        oracle can return True: every other dynamic policy drafts at least one token."""\n        return False\n\n    def stop(self, view: TraceView, p: int, g: int) -> bool:  # g >= 1\n        raise NotImplementedError\n\n    def reset(self, view: TraceView) -> None:\n        """Called once per sentence (policies with per-sentence caches)."""\n\n    def params(self) -> dict:\n        return {}\n\n    def __repr__(self):\n        ps = ", ".join(f"{k}={v}" for k, v in self.params().items())\n        return f"{self.name}({ps})"\n\n\n# --------------------------------------------------------------------------- #\n# cost model\n# --------------------------------------------------------------------------- #\n\ndef load_timing(path) -> dict:\n    return json.loads(Path(path).read_text())\n\n\nclass MeasuredVerify:\n    """Measured verification cost for g drafted tokens (g+1 target tokens) relative to one decode\n    step, linearly interpolated, clipped below at 1 (dips are timing noise). verify(0) = 1.\n    A class (not a closure) so it can be sent to worker processes."""\n\n    def __init__(self, timing: dict):\n        self.gs = sorted(int(g) for g in timing["t_target_verify"])\n        self.vs = [max(1.0, timing["t_target_verify"][str(g)] / timing["t_target"]) for g in self.gs]\n        self.table = {0: 1.0}\n        for g in range(1, max(self.gs) + 1):\n            self.table[g] = float(np.interp(g, self.gs, self.vs))\n\n    def __call__(self, g: int) -> float:\n        return self.table[g] if g in self.table else float(np.interp(g, self.gs, self.vs))\n\n\ndef make_verify(timing: dict) -> MeasuredVerify:\n    return MeasuredVerify(timing)\n\n\ndef unit_verify(g: int) -> float:\n    return 1.0\n\n\ndef formula_speedup(alpha: float, gamma: int, c: float) -> float:\n    """f = (1 - a^(g+1)) / ((1 - a)(g c + 1)), i.i.d. acceptance assumption (legacy)."""\n    exp_tokens = (gamma + 1) if alpha >= 1 else (1 - alpha ** (gamma + 1)) / (1 - alpha)\n    return exp_tokens / (gamma * c + 1)\n\n\n# --------------------------------------------------------------------------- #\n# simulator\n# --------------------------------------------------------------------------- #\n\n@dataclass\nclass SimResult:\n    rounds: int = 0\n    drafted: int = 0\n    accepted: int = 0\n    rejections: int = 0\n    emitted: int = 0\n    cost: float = 0.0\n    lookahead_charges: int = 0\n    policy_stops: int = 0\n    draft_lens: list = field(default_factory=list)\n    sent_ids: np.ndarray = None\n    sent_emitted: np.ndarray = None\n    sent_cost: np.ndarray = None\n    records: list = field(default_factory=list)   # (sentence_id, p, g, accepted, stopped_by_policy)\n\n\ndef simulate(traces: Sequence[Trace], policy: Policy, gamma_max: int = GAMMA_MAX, c: float = 0.334,\n             verify: Callable[[int], float] | None = None, charge_lookahead: bool = True,\n             record_rounds: bool = True) -> SimResult:\n    """Simulate greedy speculative decoding of every trace with ``policy``.\n\n    One round at committed position p: draft g tokens (cap = min(gamma_max, n - p)), accept the\n    leading matches a, emit min(a + 1, n - p) tokens, cost g*c + verify(g)\n    (+ c if the policy has lookahead 1, stopped by itself before the cap, and charge_lookahead).\n    """\n    verify = verify or unit_verify\n    R = SimResult()\n    ns = len(traces)\n    R.sent_ids = np.array([t.id for t in traces])\n    R.sent_emitted = np.zeros(ns, dtype=np.int64)\n    R.sent_cost = np.zeros(ns, dtype=np.float64)\n    la = int(policy.lookahead)\n    for si, tr in enumerate(traces):\n        n, match = tr.n, tr.match\n        view = TraceView(tr, policy.is_oracle)\n        policy.reset(view)\n        p = 0\n        s_emit, s_cost = 0, 0.0\n        while p < n:\n            r = n - p\n            cap = min(gamma_max, r)\n            view.p = p\n            g = 0\n            by_policy = False\n            view.limit = p - 1\n            if cap > 0 and not policy.initial_stop(view, p):\n                g = 1\n                while g < cap:\n                    view.limit = p + g - 1 + la\n                    if policy.stop(view, p, g):\n                        by_policy = True\n                        break\n                    g += 1\n            elif cap > 0:\n                by_policy = True\n            a = 0\n            while a < g and match[p + a]:\n                a += 1\n            gained = min(a + 1, r)\n            cost = g * c + verify(g)\n            if la and by_policy and g > 0 and charge_lookahead:\n                cost += c\n                R.lookahead_charges += 1\n            R.rounds += 1\n            R.drafted += g\n            R.accepted += a\n            R.rejections += int(a < g)\n            R.emitted += gained\n            R.cost += cost\n            R.policy_stops += int(by_policy)\n            R.draft_lens.append(g)\n            if record_rounds:\n                R.records.append((tr.id, p, g, a, by_policy))\n            s_emit += gained\n            s_cost += cost\n            p += gained\n        R.sent_emitted[si] = s_emit\n        R.sent_cost[si] = s_cost\n    return R\n\n\ndef summarize(res: SimResult) -> dict:\n    """Legacy summary metrics."""\n    return dict(\n        acceptance_rate=res.accepted / max(res.drafted, 1),                    # accepted / drafted\n        alpha_hat=res.accepted / max(res.accepted + res.rejections, 1),        # per-token accept prob (geometric MLE)\n        tokens_per_verif=res.emitted / max(res.rounds, 1),\n        mean_draft_len=res.drafted / max(res.rounds, 1),\n        speedup=res.emitted / res.cost if res.cost else float("nan"),\n    )\n\n\n# --------------------------------------------------------------------------- #\n# legacy protocol (sanity check against the PDF numbers, PLAN T4)\n# --------------------------------------------------------------------------- #\n\ndef legacy_protocol(traces: Sequence[Trace], c: float, seed: int = 0) -> dict:\n    """Reproduce the old evaluation: random half split of devtest (default_rng(0).permutation),\n    entropy grid lam = linspace(-1.5, 0.95, 40), beta in {0.5, 1, 2}, unit verify, test on the 2nd half.\n    Expected (PDF): fixed k=2 ~ 1.36, entropy ~ 1.45 (within ~0.03 after regeneration)."""\n    from .policies import Entropy, FixedK, Oracle\n    idx = np.random.default_rng(seed).permutation(len(traces))\n    dev = [traces[i] for i in idx[: len(idx) // 2]]\n    test = [traces[i] for i in idx[len(idx) // 2:]]\n    ks = range(2, 9)\n    best_k = max(ks, key=lambda k: summarize(simulate(dev, FixedK(k), k, c, record_rounds=False))["speedup"])\n    rows = []\n    for beta in (0.5, 1.0, 2.0):\n        for lam in np.round(np.linspace(-1.5, 0.95, 40), 3):\n            m = summarize(simulate(dev, Entropy(float(lam), beta), GAMMA_MAX, c, record_rounds=False))\n            rows.append((m["speedup"], float(lam), beta))\n    _, lam, beta = max(rows)\n    fixed_all = {k: summarize(simulate(traces, FixedK(k), k, c, record_rounds=False)) for k in ks}\n    return dict(\n        best_k_dev=best_k,\n        fixed_test=summarize(simulate(test, FixedK(best_k), best_k, c, record_rounds=False)),\n        fixed_k2_all=fixed_all[2],\n        entropy_params=dict(lam=lam, beta=beta),\n        entropy_test=summarize(simulate(test, Entropy(lam, beta), GAMMA_MAX, c, record_rounds=False)),\n        oracle_test=summarize(simulate(test, Oracle(), GAMMA_MAX, c, record_rounds=False)),\n        expected=dict(fixed_k2=1.361, entropy=1.452, oracle=1.905),\n    )\n',
    'policies': '"""Stopping policies (PLAN T5, T6). Each reads only through ``simulator.TraceView``.\n\nCausal (lookahead 0) policies read draft features at indices <= p+g-1. Lookahead-1 policies also read\nindex p+g (the next draft distribution q_{p+g}, one extra draft forward; the simulator charges it\nwhen the policy stops by itself). Oracles (``is_oracle``) may read gold fields; they are upper bounds.\n\n"Extend" semantics (AksharaSafeEntropy, Hybrid, MorphEntropy): let g0 be the first g at which the\nentropy rule fired in this round (recomputed from H[p..p+g-1], so there is no hidden state). At every\ng >= g0 the policy stops if its linguistic condition holds at g, or once g - g0 >= max_ext.\n"""\nfrom __future__ import annotations\n\nimport math\nfrom dataclasses import dataclass\n\nimport numpy as np\n\nfrom .boundaries import END, WORD\nfrom .simulator import Policy, TraceView\n\n\ndef entropy_fires(H: float, lam: float, beta: float) -> bool:\n    """AdaEDL/SVIP-style rule: stop when 1 - sqrt(beta * H) < lam."""\n    return 1.0 - math.sqrt(beta * max(float(H), 0.0)) < lam\n\n\ndef first_fire(view: TraceView, p: int, g: int, lam: float, beta: float) -> int | None:\n    """Smallest g\' in 1..g at which the entropy rule fired (reads H[p..p+g-1] only)."""\n    for k in range(1, g + 1):\n        if entropy_fires(view.feat("H", p + k - 1), lam, beta):\n            return k\n    return None\n\n\nclass NoDraft(Policy):\n    name = "NoDraft"\n\n    def initial_stop(self, view, p):\n        return True\n\n    def stop(self, view, p, g):\n        return True\n\n\nclass FixedK(Policy):\n    name = "FixedK"\n\n    def __init__(self, k: int):\n        self.k = int(k)\n\n    def initial_stop(self, view, p):\n        return self.k <= 0\n\n    def stop(self, view, p, g):\n        return g >= self.k\n\n    def params(self):\n        return dict(k=self.k)\n\n\nclass Entropy(Policy):\n    name = "Entropy"\n\n    def __init__(self, lam: float, beta: float = 1.0):\n        self.lam, self.beta = float(lam), float(beta)\n\n    def stop(self, view, p, g):\n        return entropy_fires(view.feat("H", p + g - 1), self.lam, self.beta)\n\n    def params(self):\n        return dict(lam=self.lam, beta=self.beta)\n\n\nclass WordBoundarySoft(Policy):\n    """Stop when g >= min_g and the next draft distribution puts > tau mass on new-word tokens."""\n    name = "WordBoundarySoft"\n    lookahead = 1\n\n    def __init__(self, tau: float, min_g: int = 1):\n        self.tau, self.min_g = float(tau), int(min_g)\n\n    def stop(self, view, p, g):\n        return g >= self.min_g and view.feat("p_newword", p + g) > self.tau\n\n    def params(self):\n        return dict(tau=self.tau, min_g=self.min_g)\n\n\nclass WordBoundaryHard(Policy):\n    """Corrected legacy word-boundary policy: stop when g >= min_g and the draft\'s next token\n    (argmax of q_{p+g}) starts with space / punctuation or is eos. Needs the vocabulary mask."""\n    name = "WordBoundaryHard"\n    lookahead = 1\n\n    def __init__(self, min_g: int, newword_mask: np.ndarray):\n        self.min_g = int(min_g)\n        self.mask = np.asarray(newword_mask, dtype=bool)\n\n    def stop(self, view, p, g):\n        if g < self.min_g:\n            return False\n        t = view.token(p + g)\n        return bool(t < len(self.mask) and self.mask[t])\n\n    def params(self):\n        return dict(min_g=self.min_g)\n\n\nclass _EntropyExtend(Policy):\n    """Entropy gate + linguistic condition at the lookahead position, with up to max_ext extensions."""\n    lookahead = 1\n\n    def __init__(self, lam: float, beta: float, max_ext: int):\n        self.lam, self.beta, self.max_ext = float(lam), float(beta), int(max_ext)\n\n    def cond(self, view: TraceView, p: int, g: int) -> bool:\n        raise NotImplementedError\n\n    def stop(self, view, p, g):\n        g0 = first_fire(view, p, g, self.lam, self.beta)\n        if g0 is None:\n            return False\n        return self.cond(view, p, g) or (g - g0) >= self.max_ext\n\n\nclass AksharaSafeEntropy(_EntropyExtend):\n    """Entropy rule, but only stop where the next token is unlikely to be a combining mark\n    (p_mark[p+g] < tau_m), i.e. the last akshara is probably complete."""\n    name = "AksharaSafeEntropy"\n\n    def __init__(self, lam: float, beta: float, tau_m: float, max_ext: int):\n        super().__init__(lam, beta, max_ext)\n        self.tau_m = float(tau_m)\n\n    def cond(self, view, p, g):\n        return view.feat("p_mark", p + g) < self.tau_m\n\n    def params(self):\n        return dict(lam=self.lam, beta=self.beta, tau_m=self.tau_m, max_ext=self.max_ext)\n\n\nclass Hybrid(_EntropyExtend):\n    """Entropy rule, but only stop where the current word is probably complete (p_newword[p+g] > tau_w)."""\n    name = "Hybrid"\n\n    def __init__(self, lam: float, beta: float, tau_w: float, max_ext: int):\n        super().__init__(lam, beta, max_ext)\n        self.tau_w = float(tau_w)\n\n    def cond(self, view, p, g):\n        return view.feat("p_newword", p + g) > self.tau_w\n\n    def params(self):\n        return dict(lam=self.lam, beta=self.beta, tau_w=self.tau_w, max_ext=self.max_ext)\n\n\nclass Oracle(Policy):\n    """Upper bound: drafts exactly the leading run of matches (can draft 0), never rejects."""\n    name = "Oracle"\n    is_oracle = True\n\n    def initial_stop(self, view, p):\n        return not view.gold("match", p)\n\n    def stop(self, view, p, g):\n        return not view.gold("match", p + g)\n\n\nclass GoldWordOracle(Policy):\n    """Upper bound for word-boundary stopping: stop (g >= min_g) when the *gold* cut after token\n    p+g-1 of the target is a word boundary. Free lookahead; needs ``attach_word_labels``."""\n    name = "GoldWordOracle"\n    is_oracle = True\n\n    def __init__(self, min_g: int = 1):\n        self.min_g = int(min_g)\n\n    def stop(self, view, p, g):\n        return g >= self.min_g and view.gold("end_cut", p + g - 1) in (WORD, END)\n\n    def params(self):\n        return dict(min_g=self.min_g)\n\n\n# --------------------------------------------------------------------------- #\n# T6: morpheme policies\n# --------------------------------------------------------------------------- #\n\n@dataclass\nclass TokenTable:\n    """Per-vocabulary-id raw bytes and a word-start mask (space / punctuation / special)."""\n    token_bytes: list\n    word_start: np.ndarray\n\n    @classmethod\n    def from_tokenizer(cls, tok, masks=None) -> "TokenTable":\n        from .vocab import build_masks, token_bytes_list\n        masks = masks or build_masks(tok)\n        tb = token_bytes_list(tok, range(len(tok)))\n        return cls(tb, masks.starts_space | masks.starts_punct | masks.is_special)\n\n\ndef _complete_utf8(b: bytes) -> str:\n    for cut in range(len(b), max(len(b) - 4, -1), -1):\n        try:\n            return b[:cut].decode("utf-8")\n        except UnicodeDecodeError:\n            continue\n    return b.decode("utf-8", errors="ignore")\n\n\ndef current_word_prefix(view: TraceView, p: int, g: int, table: TokenTable, max_back: int = 32) -> str:\n    """Text of the current word after drafting g tokens: committed target tokens before p plus drafted\n    tokens p..p+g-1, scanning back to the last token that starts a word (that token\'s leading space\n    is stripped). Only complete UTF-8 characters are decoded."""\n    end = p + g - 1\n    parts = []\n    lo = max(0, end - max_back)\n    for idx in range(end, lo - 1, -1):\n        t = view.token(idx)\n        b = table.token_bytes[t] if t < len(table.token_bytes) else b""\n        parts.append(b)\n        if t < len(table.word_start) and table.word_start[t]:\n            break\n    raw = b"".join(reversed(parts)).lstrip(b" \\n\\t")\n    return _complete_utf8(raw)\n\n\ndef morph_prob(view, p: int, g: int, pm, table: TokenTable) -> float:\n    """pm(current word prefix) after drafting g tokens from p.\n\n    In the simulator the value at (trace, p, g) is the same for every config of every morph policy, so it\n    is cached on the trace object (keyed by the prefix model\'s unique ``uid``). It is computed the first time through the\n    causality-checked view, and (p, g) fixes exactly what the view allows, so reuse cannot leak\n    information. Live views (no ``trace``) are never cached."""\n    tr = getattr(view, "trace", None)\n    if tr is None:\n        return pm.prob(current_word_prefix(view, p, g, table))\n    cache = tr.__dict__.setdefault("_pm_cache", {})\n    key = (getattr(pm, "uid", id(pm)), p, g)\n    v = cache.get(key)\n    if v is None:\n        v = cache[key] = pm.prob(current_word_prefix(view, p, g, table))\n    return v\n\n\nclass MorphEntropy(_EntropyExtend):\n    """Entropy rule; stop where the causal prefix model predicts a morpheme boundary (pm > tau_morph).\n    If use_akshara (lookahead 1): the condition also holds when the word is complete\n    (p_newword[p+g] > 0.5), and additionally requires p_mark[p+g] < tau_m.\n    Without use_akshara the policy is causal (lookahead 0). Extends up to max_ext tokens."""\n    name = "MorphEntropy"\n\n    def __init__(self, lam: float, beta: float, tau_morph: float, use_akshara: bool, tau_m: float,\n                 max_ext: int, prefix_model, table: TokenTable):\n        super().__init__(lam, beta, max_ext)\n        self.tau_morph, self.use_akshara, self.tau_m = float(tau_morph), bool(use_akshara), float(tau_m)\n        self.pm, self.table = prefix_model, table\n        self.lookahead = 1 if self.use_akshara else 0\n\n    def cond(self, view, p, g):\n        ok = morph_prob(view, p, g, self.pm, self.table) > self.tau_morph\n        if self.use_akshara:\n            ok = (ok or view.feat("p_newword", p + g) > 0.5) and view.feat("p_mark", p + g) < self.tau_m\n        return ok\n\n    def params(self):\n        return dict(lam=self.lam, beta=self.beta, tau_morph=self.tau_morph, use_akshara=self.use_akshara,\n                    tau_m=self.tau_m, max_ext=self.max_ext)\n\n\nclass MorphOnly(Policy):\n    """Pure morpheme policy (no entropy, causal): stop when g >= min_g and pm > tau_morph."""\n    name = "MorphOnly"\n\n    def __init__(self, tau_morph: float, min_g: int, prefix_model, table: TokenTable):\n        self.tau_morph, self.min_g = float(tau_morph), int(min_g)\n        self.pm, self.table = prefix_model, table\n\n    def stop(self, view, p, g):\n        return g >= self.min_g and morph_prob(view, p, g, self.pm, self.table) > self.tau_morph\n\n    def params(self):\n        return dict(tau_morph=self.tau_morph, min_g=self.min_g)\n',
    'stats': '"""Bootstrap statistics over sentences (PLAN T7.4).\n\nSpeed-up on a resample = sum(emitted) / sum(cost) over the resampled sentences (ratio of sums, the\nsame estimator as the point value). Paired comparisons use the same resampled indices for both\npolicies, which must have been simulated on the same sentence list.\n"""\nfrom __future__ import annotations\n\nimport numpy as np\n\n\ndef _idx(n: int, n_boot: int, seed: int) -> np.ndarray:\n    return np.random.default_rng(seed).integers(0, n, size=(n_boot, n))\n\n\ndef bootstrap_ci(emitted, cost, n_boot: int = 1000, seed: int = 0, level: float = 0.95) -> dict:\n    e, c = np.asarray(emitted, float), np.asarray(cost, float)\n    I = _idx(len(e), n_boot, seed)\n    s = e[I].sum(1) / c[I].sum(1)\n    a = (1 - level) / 2\n    return dict(point=float(e.sum() / c.sum()), lo=float(np.quantile(s, a)), hi=float(np.quantile(s, 1 - a)),\n                n_boot=n_boot, seed=seed)\n\n\ndef paired_bootstrap(emitted_a, cost_a, emitted_b, cost_b, n_boot: int = 1000, seed: int = 0,\n                     level: float = 0.95) -> dict:\n    """Difference speedup(a) - speedup(b) with a paired bootstrap over sentences.\n    p_better = fraction of resamples where the difference is > 0."""\n    ea, ca = np.asarray(emitted_a, float), np.asarray(cost_a, float)\n    eb, cb = np.asarray(emitted_b, float), np.asarray(cost_b, float)\n    if not (len(ea) == len(ca) == len(eb) == len(cb)):\n        raise ValueError("paired bootstrap needs per-sentence arrays of equal length")\n    I = _idx(len(ea), n_boot, seed)\n    d = ea[I].sum(1) / ca[I].sum(1) - eb[I].sum(1) / cb[I].sum(1)\n    a = (1 - level) / 2\n    return dict(diff=float(ea.sum() / ca.sum() - eb.sum() / cb.sum()), lo=float(np.quantile(d, a)),\n                hi=float(np.quantile(d, 1 - a)), p_better=float((d > 0).mean()), n_boot=n_boot, seed=seed)\n\n\ndef bootstrap_stat_by_sentence(groups: list, fn, n_boot: int = 1000, seed: int = 0, level: float = 0.95) -> dict:\n    """Generic sentence-level bootstrap: ``groups`` is a list of per-sentence objects, ``fn`` maps a\n    list of them to a scalar."""\n    I = _idx(len(groups), n_boot, seed)\n    vals = np.array([fn([groups[i] for i in row]) for row in I], float)\n    n_valid = int(np.isfinite(vals).sum())\n    vals = vals[np.isfinite(vals)]\n    a = (1 - level) / 2\n    lo = float(np.quantile(vals, a)) if n_valid else float("nan")      # e.g. constant variable\n    hi = float(np.quantile(vals, 1 - a)) if n_valid else float("nan")\n    return dict(point=float(fn(groups)), lo=lo, hi=hi, n_boot=n_boot, n_valid=n_valid, seed=seed)\n',
    'spec_loop': '"""Real greedy draft-and-verify loop, batch size 1, KV caches for both models (PLAN T9).\n\nThe *same* policy classes as the simulator run live: ``LiveView`` exposes ``feat`` / ``token`` with\nthe simulator\'s index convention (absolute output positions) and the same causality limit\n(``p + g - 1 + lookahead``). Draft features are computed from the draft\'s actual logits\n(conditioned on its own drafted tokens).\n\nRound at committed output length p (pending token = last committed token, not yet in the caches):\n  1. draft: feed every committed token missing from the draft cache, then draft greedily; after\n     each drafted token ask the policy (g >= 1). A lookahead-1 policy needs q_{p+g}, i.e. one more\n     draft forward; if the policy continues, that forward produced the next drafted token.\n  2. verify: one target forward on [pending, d_1..d_g]; accept the leading matches a; the target\n     argmax at position a is the correction / bonus token.\n  3. crop both caches to the committed length minus one (``crop_to``; transformers 5.22 ``crop``\n     takes the number of tokens to *remove*).\nOutput = committed tokens truncated after the first eos or at ``max_new``.\n"""\nfrom __future__ import annotations\n\nimport json\nimport time\nfrom dataclasses import dataclass, field\n\nimport numpy as np\n\nfrom .simulator import GAMMA_MAX, CausalityError, Policy\n\nFEATURE_NAMES = ("H", "top1", "top2", "p_space", "p_punct", "p_eos", "p_newword", "p_mark", "p_cont")\n\n\ndef _torch():\n    import torch\n    return torch\n\n\ndef crop_to(cache, length: int) -> None:\n    """Crop a DynamicCache to ``length`` tokens (no-op if already shorter).\n\n    transformers 5.22: ``DynamicCache.crop(n)`` removes ``-n`` tokens for negative n and rejects positive n\n    (older versions took the final length). The result is checked so an API change fails loudly\n    instead of silently corrupting the cache."""\n    cur = cache.get_seq_length()\n    if cur > length:\n        cache.crop(-(cur - length))\n        got = cache.get_seq_length()\n        if got != length:\n            raise RuntimeError(f"cache crop failed: expected length {length}, got {got} (transformers API change?)")\n\n\nclass NoisyDraft:\n    """CPU smoke-test helper: wraps a causal LM and adds deterministic Gaussian noise to its logits,\n    seeded by the token that precedes each predicted position. Teacher-forced and live calls therefore\n    see the same draft distribution for the same context, while draft and target now disagree, so the\n    rejection / cache-crop path of the real loop is exercised. Not used on Kaggle."""\n\n    def __init__(self, model, sigma: float = 3.0, seed: int = 1234):\n        self.model, self.sigma, self.seed = model, float(sigma), int(seed)\n\n    def __getattr__(self, name):\n        return getattr(self.model, name)\n\n    def __call__(self, input_ids=None, **kw):\n        torch = _torch()\n        out = self.model(input_ids=input_ids, **kw)\n        V = out.logits.shape[-1]\n        prev = input_ids[0].tolist()\n        noise = torch.stack([torch.randn(V, generator=torch.Generator().manual_seed(self.seed * 1_000_003 + int(t)))\n                             for t in prev]).to(out.logits.device, out.logits.dtype)\n        out.logits = out.logits + self.sigma * noise[None]\n        return out\n\n\ndef _sync():\n    torch = _torch()\n    if torch.cuda.is_available():\n        for i in range(torch.cuda.device_count()):\n            torch.cuda.synchronize(i)\n\n\nclass LiveView:\n    """Causality-checked view of the live decoding state (same interface as ``TraceView``)."""\n\n    def __init__(self):\n        self.out: list[int] = []        # committed output tokens\n        self.draft: dict[int, int] = {}  # absolute idx -> drafted token (this round, incl. lookahead argmax)\n        self.feats: dict[int, dict] = {}  # absolute idx -> features of q_idx\n        self.p = 0\n        self.limit = -1\n        self.is_oracle = False\n        self.n = 10 ** 9\n\n    def _check(self, idx):\n        if idx > self.limit:\n            raise CausalityError(f"live read index {idx} > limit {self.limit} (p={self.p})")\n\n    def feat(self, name, idx):\n        self._check(idx)\n        if name == "draft_argmax":\n            return self.draft[idx]\n        if idx not in self.feats:\n            raise CausalityError(f"feature {name}[{idx}] not computed live (committed by the target)")\n        return self.feats[idx][name]\n\n    def token(self, idx):\n        self._check(idx)\n        return self.out[idx] if idx < self.p else self.draft[idx]\n\n    def gold(self, name, idx):\n        raise CausalityError("oracle policies cannot run live")\n\n\n@dataclass\nclass LoopResult:\n    out_ids: list\n    rounds: list = field(default_factory=list)  # dicts: p, g, accepted, extra_lookahead, t_draft, t_verify\n    wall: float = 0.0\n    n_draft_forwards: int = 0\n    n_target_forwards: int = 0\n\n\nclass SpecDecoder:\n    def __init__(self, target, draft, masks, eos_ids, device: str = "cuda:0", gamma_max: int = GAMMA_MAX):\n        torch = _torch()\n        from .gpu_traces import _mask_matrix\n        self.target, self.draft, self.device = target, draft, device\n        self.eos = set(int(i) for i in eos_ids)\n        self.gamma_max = gamma_max\n        V = draft.get_output_embeddings().weight.shape[0]\n        self.M = _mask_matrix(masks, V, device)            # (V, 5): space, punct, eos, mark, cont\n        self.torch = torch\n\n    # ---- model calls ---------------------------------------------------- #\n    def _fwd(self, model, ids, cache):\n        torch = self.torch\n        x = torch.tensor([ids], device=self.device)\n        with torch.no_grad():\n            return model(input_ids=x, past_key_values=cache, use_cache=True).logits[0].float()\n\n    def _features(self, logits_row) -> tuple[int, dict]:\n        torch = self.torch\n        lp = torch.log_softmax(logits_row, -1)\n        p = lp.exp()\n        top = p.topk(2)\n        P = p @ self.M.to(p.device)\n        H = -(p * lp).sum()\n        vals = torch.stack([H, top.values[0], top.values[1], P[0], P[1], P[2], P[0] + P[1] + P[2], P[3],\n                            P[4]]).cpu().tolist()\n        f = dict(zip(FEATURE_NAMES, vals))\n        return int(top.indices[0].item()), f\n\n    # ---- decoding --------------------------------------------------------- #\n    def decode(self, prompt_ids: list[int], policy: Policy, max_new: int, record: bool = True) -> LoopResult:\n        from transformers import DynamicCache\n        if policy.is_oracle:\n            raise ValueError("oracle policies cannot run live")\n        tc, dc = DynamicCache(), DynamicCache()\n        la = int(policy.lookahead)\n        res = LoopResult(out_ids=[])\n        _sync()\n        t_start = time.perf_counter()\n        # prefill: target on prompt[:-1], draft on prompt[:-1]; pending token = prompt[-1]\n        seq = list(prompt_ids)\n        if len(seq) > 1:\n            self._fwd(self.target, seq[:-1], tc)\n            self._fwd(self.draft, seq[:-1], dc)\n            res.n_target_forwards += 1\n            res.n_draft_forwards += 1\n        view = LiveView()\n        policy.reset(view)\n        n_prompt = len(prompt_ids)\n        done = False\n        while not done:\n            p = len(seq) - n_prompt          # committed output length\n            view.p, view.out = p, seq[n_prompt:]\n            view.draft, view.feats = {}, {}\n            cap = min(self.gamma_max, max_new - p)\n            _sync()\n            t0 = time.perf_counter()\n            drafted: list[int] = []\n            extra = False\n            g = 0\n            if cap > 0 and not policy.initial_stop(view, p):\n                # draft forward on all committed tokens missing from the draft cache -> q_p\n                missing = seq[dc.get_seq_length():]\n                lg = self._fwd(self.draft, missing, dc)[-1]\n                res.n_draft_forwards += 1\n                tok, f = self._features(lg)\n                view.draft[p], view.feats[p] = tok, f\n                drafted.append(tok)\n                g = 1\n                while g < cap:\n                    if la:   # q_{p+g}: feed the last drafted token\n                        lg = self._fwd(self.draft, [drafted[-1]], dc)[-1]\n                        res.n_draft_forwards += 1\n                        tok, f = self._features(lg)\n                        view.draft[p + g], view.feats[p + g] = tok, f\n                    view.limit = p + g - 1 + la\n                    if policy.stop(view, p, g):\n                        extra = bool(la)\n                        break\n                    if not la:\n                        lg = self._fwd(self.draft, [drafted[-1]], dc)[-1]\n                        res.n_draft_forwards += 1\n                        tok, f = self._features(lg)\n                        view.draft[p + g], view.feats[p + g] = tok, f\n                    drafted.append(view.draft[p + g])\n                    g += 1\n            _sync()\n            t1 = time.perf_counter()\n            # verify [pending] + drafted\n            lg = self._fwd(self.target, [seq[-1]] + drafted, tc)\n            res.n_target_forwards += 1\n            am = lg.argmax(-1).tolist()\n            a = 0\n            while a < g and am[a] == drafted[a]:\n                a += 1\n            new = drafted[:a] + [am[a]]\n            _sync()\n            t2 = time.perf_counter()\n            seq += new\n            L = len(seq)\n            crop_to(tc, L - 1)\n            crop_to(dc, L - 1)\n            if record:\n                res.rounds.append(dict(p=p, g=g, accepted=a, extra_lookahead=extra, t_draft=t1 - t0, t_verify=t2 - t1))\n            out = seq[n_prompt:]\n            if any(t in self.eos for t in new) or len(out) >= max_new:\n                done = True\n        _sync()\n        res.wall = time.perf_counter() - t_start\n        out = seq[n_prompt:]\n        cut = next((i + 1 for i, t in enumerate(out) if t in self.eos), len(out))\n        res.out_ids = out[:min(cut, max_new)]\n        return res\n\n\ndef first_divergence(a: list[int], b: list[int]) -> int | None:\n    for i, (x, y) in enumerate(zip(a, b)):\n        if x != y:\n            return i\n    return None if len(a) == len(b) else min(len(a), len(b))\n\n\ndef target_margin_at(target, prompt_ids, prefix_out, device) -> dict:\n    """Target top-1 vs top-2 logit margin for the next token after prompt + prefix_out (fp16 near-tie check)."""\n    torch = _torch()\n    x = torch.tensor([list(prompt_ids) + list(prefix_out)], device=device)\n    with torch.no_grad():\n        lg = target(x).logits[0, -1].float()\n    v, i = lg.topk(2)\n    return dict(top1=int(i[0]), top2=int(i[1]), margin=float(v[0] - v[1]))\n\n\n# --------------------------------------------------------------------------- #\n# notebook 03 entry point (also called by scripts/smoke_cpu.py --part 03)\n# --------------------------------------------------------------------------- #\n\n# MorphEntropy runs in its constrained form: the selected config has max_ext=0 and is identical to Entropy\n# (user decision 2026-11-02).\nLIVE_FAMILIES = ["NoDraft", "FixedK", "Entropy", "WordBoundarySoft", "Hybrid", "MorphEntropy_ext1"]\n\nDEFAULT_CONFIG_03 = dict(\n    N_SENT=100, REPEATS=3, MAX_NEW=256, SMOKE=False,\n    TARGET="meta-llama/Meta-Llama-3-8B-Instruct",\n    DRAFT="meta-llama/Llama-3.2-1B-Instruct",\n    MAX_MEMORY={0: "8GiB", 1: "10GiB"},\n    OUT_ROOT="/kaggle/working",\n    GIT_HASH="unknown",\n    POLICIES=LIVE_FAMILIES,\n    SELECTED=None,          # {family: params} from results/phase1/selected_params.json\n    PREFIX_MODEL_STATE=None,  # PrefixBoundaryModel state dict (decoded from the embedded gzip+base64)\n    IDS=None,               # devtest sentence ids (results/phase1/real_loop_ids.json)\n)\n\n\ndef _append_jsonl(path, rec):\n    with open(path, "a", encoding="utf-8") as fh:\n        fh.write(json.dumps(rec, ensure_ascii=False) + "\\n")\n\n\ndef notebook03_main(cfg: dict) -> dict:\n    """PLAN T9 steps 1-8 on the GPU (or CPU with SMOKE=True: draft model as both target and draft)."""\n    import shutil\n    from pathlib import Path\n    from . import gpu_traces as G\n    from .morph import PrefixBoundaryModel\n    from .policies import TokenTable\n    from .traces import save_traces\n    from .tuning import Context, make_policy\n    from .vocab import build_masks\n    torch = _torch()\n    cfg = {**DEFAULT_CONFIG_03, **cfg}\n    out_dir = Path(cfg["OUT_ROOT"]) / "real_loop"\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    # ---- models ---- #\n    if cfg["SMOKE"]:\n        device, dtype = "cpu", torch.float32\n        tok = G.load_tokenizer(cfg["DRAFT"])\n        draft = target = G.load_draft(cfg["DRAFT"], dtype, device)\n        if cfg.get("SMOKE_DRAFT_NOISE"):          # exercise rejections + cache cropping on CPU\n            draft = NoisyDraft(target, cfg["SMOKE_DRAFT_NOISE"])\n    else:\n        assert torch.cuda.device_count() >= 2, "Need 2 GPUs: Settings > Accelerator > GPU T4 x2"\n        device, dtype = "cuda:0", torch.float16\n        tok = G.load_tokenizer(cfg["TARGET"])\n        target = G.load_target(cfg["TARGET"], dtype, max_memory=cfg["MAX_MEMORY"])\n        draft = G.load_draft(cfg["DRAFT"], dtype, device)\n        G.check_draft_target_compat(draft, target, G.load_tokenizer(cfg["DRAFT"]), tok, "परीक्षण")\n    eos_ids = G.get_eos_ids(target, tok)\n    masks = build_masks(tok, eos_ids)\n    ctx = Context(newword_mask=masks.starts_newword, table=TokenTable.from_tokenizer(tok, masks),\n                  prefix_model=PrefixBoundaryModel.from_state(cfg["PREFIX_MODEL_STATE"]))\n    sel = cfg["SELECTED"]\n    policies = {f: make_policy(f, sel.get(f, {}), ctx) for f in cfg["POLICIES"]}\n    print("policies:", policies)\n    dec = SpecDecoder(target, draft, masks, eos_ids, device)\n\n    # ---- sentences ---- #\n    rows = {r[0]: r for r in G.load_flores_pairs("devtest")}\n    ids = list(cfg["IDS"])[: cfg["N_SENT"]]\n    prompts = {i: tok(G.build_prompt(tok, rows[i][1]), add_special_tokens=False)["input_ids"] for i in ids}\n\n    # ---- unbatched greedy reference (B9) ---- #\n    ref_path = out_dir / "greedy_unbatched.jsonl"\n    refs = {r["id"]: r for r in G.read_jsonl(ref_path)}\n    for i in ids:\n        if i not in refs:\n            o = G.generate_unbatched(target, tok, prompts[i], cfg["MAX_NEW"], eos_ids, device)\n            refs[i] = dict(id=i, src=rows[i][1], ref=rows[i][2], prompt_ids=prompts[i], out_ids=o,\n                           out_text=tok.decode(o, skip_special_tokens=True).strip())\n            _append_jsonl(ref_path, refs[i])\n    print(f"unbatched greedy references: {len(refs)}")\n\n    # ---- warm-up ---- #\n    for f, pol in policies.items():\n        dec.decode(prompts[ids[0]], pol, min(32, cfg["MAX_NEW"]), record=False)\n\n    # ---- timed runs (resumable) ---- #\n    res_path = out_dir / "loop_results.jsonl"\n    done = {(r["rep"], r["id"], r["policy"]) for r in G.read_jsonl(res_path)}\n    t_first = time.perf_counter()\n    n_new = 0\n    for rep in range(cfg["REPEATS"]):\n        for si, i in enumerate(ids):\n            for f, pol in policies.items():\n                if (rep, i, f) in done:\n                    continue\n                r = dec.decode(prompts[i], pol, cfg["MAX_NEW"])\n                lossless = r.out_ids == refs[i]["out_ids"]\n                _append_jsonl(res_path, dict(rep=rep, id=i, policy=f, params=sel.get(f, {}), wall=r.wall,\n                                             n_out=len(r.out_ids), out_ids=r.out_ids, lossless=lossless,\n                                             first_divergence=None if lossless else first_divergence(r.out_ids, refs[i]["out_ids"]),\n                                             n_draft_forwards=r.n_draft_forwards,\n                                             n_target_forwards=r.n_target_forwards, rounds=r.rounds))\n                n_new += 1\n            if rep == 0 and si == 4 and n_new:\n                per = (time.perf_counter() - t_first) / 5\n                est = per * len(ids) * cfg["REPEATS"] / 3600\n                print(f"ESTIMATE: {per:.1f}s per sentence (all policies) -> about {est:.1f} h for "\n                      f"N_SENT={len(ids)} x REPEATS={cfg[\'REPEATS\']}. Lower N_SENT / REPEATS if that is too long.")\n        print(f"repeat {rep} done")\n\n    # ---- losslessness details ---- #\n    results = G.read_jsonl(res_path)\n    div = []\n    for r in results:\n        if r["rep"] == 0 and not r["lossless"]:\n            pos = r["first_divergence"]\n            ref = refs[r["id"]]["out_ids"]\n            m = target_margin_at(target, prompts[r["id"]], ref[:pos], device) if pos is not None and pos < len(ref) else {}\n            div.append(dict(id=r["id"], policy=r["policy"], pos=pos, ref_token=ref[pos] if pos is not None and pos < len(ref) else None,\n                            loop_token=r["out_ids"][pos] if pos is not None and pos < len(r["out_ids"]) else None, **m))\n    (out_dir / "divergences.json").write_text(json.dumps(div, indent=1))\n    n_loss = sum(r["lossless"] for r in results)\n    print(f"losslessness: {n_loss} / {len(results)} decodes identical to unbatched greedy; {len(div)} divergences logged (rep 0)")\n\n    # ---- teacher-forced traces of the unbatched greedy outputs (for notebook 04) ---- #\n    recs = [refs[i] for i in ids]\n    dfe = G.run_draft_pass(draft, recs, masks, out_dir, "real100", device)\n    tfe = G.run_target_pass(target, recs, dfe, out_dir, "real100", device)\n    traces = G.assemble_traces(recs, dfe, tfe)\n    save_traces(out_dir / "traces_real100.npz", traces, dict(split="devtest-real100", eos_ids=eos_ids,\n                                                              git_hash=cfg["GIT_HASH"], max_new=cfg["MAX_NEW"]))\n\n    info = G.run_info(cfg["GIT_HASH"], n_sent=len(ids), repeats=cfg["REPEATS"], policies=list(policies),\n                      selected=sel, smoke=cfg["SMOKE"], target=cfg["TARGET"] if not cfg["SMOKE"] else cfg["DRAFT"],\n                      draft=cfg["DRAFT"], max_memory={str(k): v for k, v in cfg["MAX_MEMORY"].items()})\n    (out_dir / "run_info.json").write_text(json.dumps(info, indent=1, ensure_ascii=False))\n    zp = G.zip_dir(out_dir, Path(cfg["OUT_ROOT"]) / "real_loop.zip",\n                   include=["greedy_unbatched.jsonl", "loop_results.jsonl", "divergences.json",\n                            "traces_real100.npz", "run_info.json"])\n    print("\\n" + "=" * 70)\n    print("DONE. Download  real_loop.zip  from the notebook\'s Output tab")\n    print("and unzip it into  data/real_loop/  in the repo (files land directly in that folder).")\n    print("=" * 70)\n    return dict(zip=zp, n_lossless=n_loss, n_results=len(results), divergences=div)\n',
    'wordheads': '"""Word-completion heads on the draft model (PLAN_wordheads.md).\n\nPosition convention (same as the Phase-1 traces): for an output ``y[0..n-1]`` after prompt ``P``,\n``h_t`` is the draft\'s last (post-norm) hidden state at input index ``len(P) + t - 1``; the frozen LM\nhead maps it to the draft distribution over ``y[t]`` (row t of a trace).\n\n* head k (k = 1..K) on ``h_t`` predicts ``y[t+k]``: Medusa-1 residual block ``z = h + SiLU(W_k h)``\n  (``W_k`` zero-initialised) followed by the frozen, shared LM head;\n* the continuation head ``Linear(d, K)`` predicts ``c_k[t] = 1`` iff ``y[t+1..t+k]`` all continue the\n  word of ``y[t]`` (none of them is word-initial).\n\nVariants trained side by side on the same draft forward passes (same data, same order):\n  v0  plain MTP:     sum_k w_k CE_k over all positions\n  v1  word-guided:   CE_k only where c_k = 1, plus LAMBDA_CONT * BCE(continuation head)\n  v2  adaptive:      v1\'s K+1 losses with uncertainty weighting  sum_i exp(-s_i) L_i + s_i\n"""\nfrom __future__ import annotations\n\nimport json\nimport math\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\n\nVARIANTS = ("v0", "v1", "v2")\n\nDEFAULT_WH = dict(\n    DRAFT=["meta-llama/Llama-3.2-1B-Instruct", "unsloth/Llama-3.2-1B-Instruct"],\n    K=3, HEAD_W=0.8, LAMBDA_CONT=0.5,\n    TRAIN_DATASET="cfilt/iitb-english-hindi", N_TRAIN=60000, N_VAL=500, SEED=0,\n    MIN_HI_CHARS=15, MAX_HI_CHARS=400, MIN_WORDS=6, MAX_LEN=320,\n    BATCH=32, LR=1e-3, WD=0.0, WARMUP=100, POS_PER_HEAD=768, EVAL_EVERY=300, MAX_STEPS=None,\n    DUMP_TOPK=8, TIME_Q=[1, 2, 3, 4, 5, 6, 8], TIME_CTX=150, TIME_STEPS=40, TIME_REPS=3,\n    OUT_ROOT="/kaggle/working", SEARCH=["/kaggle/input"], SMOKE=False,\n)\n\n\ndef _torch():\n    import torch\n    return torch\n\n\n# --------------------------------------------------------------------------- #\n# labels\n# --------------------------------------------------------------------------- #\n\ndef word_initial_flags(tok, ids) -> np.ndarray:\n    """Gold \'token starts a new word\' flags of an output sequence (same labeller as Phase 1)."""\n    from .boundaries import token_labels\n    from .vocab import special_ids, token_bytes_list\n    spec = special_ids(tok)\n    tb = token_bytes_list(tok, ids)\n    return np.asarray(token_labels(tb, special=[int(i) in spec for i in ids])["word_initial"], bool)\n\n\ndef continuation_targets(word_initial: np.ndarray, K: int) -> np.ndarray:\n    """c[t, k-1] = 1 iff y[t+1..t+k] exist and none is word-initial (all continue y[t]\'s word)."""\n    n = len(word_initial)\n    c = np.zeros((n, K), bool)\n    for t in range(n):\n        ok = True\n        for k in range(1, K + 1):\n            ok = ok and (t + k < n) and not word_initial[t + k]\n            c[t, k - 1] = ok\n    return c\n\n\ndef continuation_vocab(tok) -> np.ndarray:\n    """Ids of tokens that can only continue a Devanagari word: non-empty, no leading space, and every\n    byte decodes to Devanagari (U+0900-U+097F), ZWJ/ZWNJ, or is part of an incomplete UTF-8 sequence\n    of such a character. Inside a word these are the only possible next tokens, so a word-completion\n    head can score this small vocabulary instead of the full one."""\n    from .vocab import special_ids, token_bytes_list\n    spec = special_ids(tok)\n    n = len(tok)\n    tb = token_bytes_list(tok, list(range(n)))\n    keep = []\n    for i, b in enumerate(tb):\n        if i in spec or not b or b[:1] in (b" ", b"\\n", b"\\t"):\n            continue\n        txt = b.decode("utf-8", errors="ignore")\n        if not txt and all(x >= 0x80 for x in b):        # pure UTF-8 fragment (part of a character)\n            keep.append(i)\n            continue\n        if txt and all("ऀ" <= ch <= "ॿ" or ch in "\u200c\u200d" for ch in txt):\n            keep.append(i)\n    return np.asarray(keep, np.int64)\n\n\n# --------------------------------------------------------------------------- #\n# model\n# --------------------------------------------------------------------------- #\n\ndef make_heads(d: int, K: int):\n    torch = _torch()\n    nn = torch.nn\n\n    class WordHeads(nn.Module):\n        def __init__(self):\n            super().__init__()\n            self.K = K\n            self.blocks = nn.ModuleList([nn.Linear(d, d) for _ in range(K)])\n            for b in self.blocks:\n                nn.init.zeros_(b.weight)\n                nn.init.zeros_(b.bias)\n            self.cont = nn.Linear(d, K)\n            self.log_var = nn.Parameter(torch.zeros(K + 1))   # used by v2 only\n\n        def z(self, h, k: int):                     # k = 1..K\n            return h + torch.nn.functional.silu(self.blocks[k - 1](h))\n\n    return WordHeads()\n\n\ndef lm_logits(z, W):\n    """Logits of the frozen LM head (weight W, [V, d], half precision) for hidden rows z (float32)."""\n    return (z.to(W.dtype) @ W.t()).float()\n\n\ndef get_backbone_and_lm_weight(model):\n    base = getattr(model, "model", None) or model.get_decoder()\n    return base, model.get_output_embeddings().weight\n\n\ndef hidden_states(model, input_ids, attention_mask):\n    """Last post-norm hidden states [B, T, d] (what the LM head reads)."""\n    torch = _torch()\n    base, _ = get_backbone_and_lm_weight(model)\n    with torch.no_grad():\n        return base(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state\n\n\n# --------------------------------------------------------------------------- #\n# data\n# --------------------------------------------------------------------------- #\n\ndef encode_pair(tok, src: str, hi: str, eos_id: int) -> tuple[list[int], list[int]]:\n    from .gpu_traces import build_prompt\n    p = tok(build_prompt(tok, src), add_special_tokens=False).input_ids\n    y = tok(hi, add_special_tokens=False).input_ids + [eos_id]\n    return p, y\n\n\ndef make_example(tok, prompt_ids, out_ids, K: int) -> dict:\n    wi = word_initial_flags(tok, out_ids)\n    return dict(prompt_ids=list(prompt_ids), out_ids=list(out_ids),\n                word_initial=wi, cont=continuation_targets(wi, K))\n\n\ndef _deva_frac(s: str) -> float:\n    letters = [ch for ch in s if ch.isalpha()]\n    return sum("ऀ" <= ch <= "ॿ" for ch in letters) / max(1, len(letters))\n\n\n_UI_CHARS = set(r"%_<>/\\{}[]|=@#~^*")\n\n\ndef keep_pair(en: str, hi: str, cfg: dict) -> bool:\n    """Sentence-like pairs only: IITB contains many software-UI strings (GNOME/KDE localisation)."""\n    if not (cfg["MIN_HI_CHARS"] <= len(hi) <= cfg["MAX_HI_CHARS"]):\n        return False\n    if len(en.split()) < cfg["MIN_WORDS"] or len(hi.split()) < cfg["MIN_WORDS"] - 1:\n        return False\n    if en[-1:] not in ".?!\\"\'" or any(ch in _UI_CHARS for ch in en):\n        return False\n    return _deva_frac(hi) >= 0.9\n\n\ndef load_train_pairs(cfg: dict, log=print) -> list[tuple[str, str]]:\n    """N_TRAIN + N_VAL IITB en-hi pairs, sampled uniformly at random (seeded) from the whole train split\n    and filtered with ``keep_pair``."""\n    if cfg.get("TRAIN_PAIRS") is not None:\n        return list(cfg["TRAIN_PAIRS"])\n    from datasets import load_dataset\n    ds = load_dataset(cfg["TRAIN_DATASET"], split="train")\n    need = cfg["N_TRAIN"] + cfg["N_VAL"]\n    order = np.random.default_rng(cfg["SEED"]).permutation(len(ds))\n    out, seen = [], 0\n    for start in range(0, len(order), 20000):\n        rows = ds.select(order[start:start + 20000].tolist())["translation"]\n        for r in rows:\n            seen += 1\n            en, hi = r["en"].strip(), r["hi"].strip()\n            if keep_pair(en, hi, cfg):\n                out.append((en, hi))\n                if len(out) >= need:\n                    break\n        if len(out) >= need:\n            break\n    log(f"train pairs: {len(out)} kept of {seen} sampled (split size {len(ds)})")\n    return out\n\n\ndef collate(batch: list[dict], pad_id: int, device):\n    """Right-padded inputs + index tensors of the output rows h_t (t = 0..n-1) for every example."""\n    torch = _torch()\n    L = max(len(b["prompt_ids"]) + len(b["out_ids"]) for b in batch)\n    ids = torch.full((len(batch), L), pad_id, dtype=torch.long)\n    am = torch.zeros((len(batch), L), dtype=torch.long)\n    rows_b, rows_i, tgt, tvalid, cont = [], [], [], [], []\n    for bi, b in enumerate(batch):\n        seq = b["prompt_ids"] + b["out_ids"]\n        ids[bi, :len(seq)] = torch.tensor(seq)\n        am[bi, :len(seq)] = 1\n        P, n = len(b["prompt_ids"]), len(b["out_ids"])\n        y = np.asarray(b["out_ids"], np.int64)\n        K = b["cont"].shape[1]\n        for t in range(n):\n            rows_b.append(bi)\n            rows_i.append(P + t - 1)\n            tk = np.array([y[t + k] if t + k < n else -100 for k in range(1, K + 1)], np.int64)\n            tgt.append(tk)\n            tvalid.append(tk >= 0)\n            cont.append(b["cont"][t])\n    T = lambda a, dt: torch.tensor(np.asarray(a), dtype=dt, device=device)\n    return dict(ids=ids.to(device), am=am.to(device), rows_b=T(rows_b, torch.long), rows_i=T(rows_i, torch.long),\n                tgt=T(tgt, torch.long), tvalid=T(tvalid, torch.bool), cont=T(cont, torch.bool))\n\n\n# --------------------------------------------------------------------------- #\n# losses\n# --------------------------------------------------------------------------- #\n\ndef variant_losses(name: str, heads, h, B, W, K: int, head_w: float, lam_cont: float,\n                   pos_per_head: int, gen) -> tuple:\n    """Total loss and per-part float values for one variant on hidden rows h [N, d] (float32)."""\n    torch = _torch()\n    F = torch.nn.functional\n    parts, logs = [], {}\n    for k in range(1, K + 1):\n        valid = B["tvalid"][:, k - 1]\n        if name in ("v1", "v2"):\n            valid = valid & B["cont"][:, k - 1]\n        idx = valid.nonzero(as_tuple=True)[0]\n        if len(idx) == 0:\n            parts.append(h.sum() * 0.0)\n            continue\n        if len(idx) > pos_per_head:\n            idx = idx[torch.randperm(len(idx), generator=gen, device="cpu")[:pos_per_head].to(idx.device)]\n        lg = lm_logits(heads.z(h[idx], k), W)\n        ce = F.cross_entropy(lg, B["tgt"][idx, k - 1])\n        parts.append(ce)\n        logs[f"ce{k}"] = ce.item()\n    if name == "v0":\n        total = sum(head_w ** (k - 1) * parts[k - 1] for k in range(1, K + 1))\n        return total, logs\n    # continuation head on all positions where y[t+1] exists\n    m = B["tvalid"][:, 0]\n    bce = F.binary_cross_entropy_with_logits(heads.cont(h[m]), B["cont"][m].float())\n    logs["bce"] = bce.item()\n    if name == "v1":\n        total = sum(head_w ** (k - 1) * parts[k - 1] for k in range(1, K + 1)) + lam_cont * bce\n    else:\n        allp = parts + [bce]\n        total = sum(torch.exp(-heads.log_var[i]) * allp[i] + heads.log_var[i] for i in range(len(allp)))\n        logs["w"] = [round(float(torch.exp(-s.detach())), 3) for s in heads.log_var]\n    return total, logs\n\n\n# --------------------------------------------------------------------------- #\n# evaluation helpers\n# --------------------------------------------------------------------------- #\n\ndef head_accuracy(heads, h, B, W, K: int) -> dict:\n    """Top-1 accuracy per head on all valid positions and on in-word (c_k = 1) positions."""\n    torch = _torch()\n    out = {}\n    with torch.no_grad():\n        for k in range(1, K + 1):\n            v = B["tvalid"][:, k - 1]\n            if not v.any():\n                continue\n            pred = lm_logits(heads.z(h[v], k), W).argmax(-1)\n            hit = pred == B["tgt"][v, k - 1]\n            cw = B["cont"][v, k - 1]\n            out[f"acc{k}_all"] = float(hit.float().mean())\n            out[f"acc{k}_inword"] = float(hit[cw].float().mean()) if cw.any() else float("nan")\n        m = B["tvalid"][:, 0]\n        if m.any():\n            pc = torch.sigmoid(heads.cont(h[m])) > 0.5\n            out["cont_acc"] = float((pc == B["cont"][m]).float().mean())\n    return out\n\n\n# --------------------------------------------------------------------------- #\n# training\n# --------------------------------------------------------------------------- #\n\ndef train_heads(model, tok, train: list[dict], val: list[dict], cfg: dict, device, log=print) -> tuple:\n    torch = _torch()\n    K = cfg["K"]\n    base, W = get_backbone_and_lm_weight(model)\n    W = W.detach()\n    d = W.shape[1]\n    torch.manual_seed(cfg["SEED"])\n    heads = {v: make_heads(d, K).to(device) for v in VARIANTS}\n    opt = {v: torch.optim.AdamW(heads[v].parameters(), lr=cfg["LR"], weight_decay=cfg["WD"]) for v in VARIANTS}\n    order = sorted(range(len(train)), key=lambda i: len(train[i]["prompt_ids"]) + len(train[i]["out_ids"]))\n    batches = [order[i:i + cfg["BATCH"]] for i in range(0, len(order), cfg["BATCH"])]\n    random.Random(cfg["SEED"]).shuffle(batches)\n    total = len(batches) if not cfg.get("MAX_STEPS") else min(len(batches), cfg["MAX_STEPS"])\n    sched = {v: torch.optim.lr_scheduler.LambdaLR(\n        opt[v], lambda s: min(1.0, (s + 1) / cfg["WARMUP"]) * max(0.05, 0.5 * (1 + math.cos(math.pi * s / total))))\n        for v in VARIANTS}\n    gen = torch.Generator().manual_seed(cfg["SEED"])\n    pad = tok.pad_token_id if tok.pad_token_id is not None else 0\n    hist, t0 = [], time.time()\n    val_b = [collate(val[i:i + cfg["BATCH"]], pad, device) for i in range(0, len(val), cfg["BATCH"])]\n\n    def evaluate(step):\n        acc = {v: [] for v in VARIANTS}\n        for B in val_b:\n            h = hidden_states(model, B["ids"], B["am"])[B["rows_b"], B["rows_i"]].float()\n            for v in VARIANTS:\n                acc[v].append(head_accuracy(heads[v], h, B, W, K))\n        res = {v: {k: float(np.nanmean([a[k] for a in acc[v] if k in a])) for k in acc[v][0]} for v in VARIANTS}\n        hist.append(dict(step=step, val=res, t=round(time.time() - t0, 1)))\n        log(f"[val step {step}] " + " | ".join(\n            f"{v}: in-word acc " + "/".join(f"{res[v].get(f\'acc{k}_inword\', float(\'nan\')):.3f}" for k in range(1, K + 1))\n            for v in VARIANTS))\n        return res\n\n    for step, bidx in enumerate(batches[:total]):\n        B = collate([train[i] for i in bidx], pad, device)\n        h = hidden_states(model, B["ids"], B["am"])[B["rows_b"], B["rows_i"]].float()\n        logs = {}\n        for v in VARIANTS:\n            loss, lg = variant_losses(v, heads[v], h, B, W, K, cfg["HEAD_W"], cfg["LAMBDA_CONT"],\n                                      cfg["POS_PER_HEAD"], gen)\n            opt[v].zero_grad(set_to_none=True)\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(heads[v].parameters(), 1.0)\n            opt[v].step()\n            sched[v].step()\n            logs[v] = lg\n        if step % 50 == 0:\n            log(f"step {step}/{total} {time.time() - t0:.0f}s " + json.dumps(\n                {v: {k: (round(x, 3) if isinstance(x, float) else x) for k, x in logs[v].items()} for v in VARIANTS}))\n        if cfg["EVAL_EVERY"] and step % cfg["EVAL_EVERY"] == 0 and step:\n            evaluate(step)\n    final = evaluate(total)\n    return heads, dict(history=hist, final_val=final, steps=total, seconds=round(time.time() - t0, 1))\n\n\n# --------------------------------------------------------------------------- #\n# dump along the target\'s outputs (for the exact trace simulation)\n# --------------------------------------------------------------------------- #\n\ndef dump_predictions(model, tok, heads: dict, records: list[dict], cfg: dict, device) -> dict:\n    """Per output position t of every record: the draft argmax for y[t] (alignment check against the\n    Phase-1 traces), and per variant the heads\' top-k ids / log-probs for y[t+1..t+K] and the\n    continuation-head probabilities. Flat arrays + offsets, like the Phase-1 traces."""\n    torch = _torch()\n    K, TK = cfg["K"], cfg["DUMP_TOPK"]\n    _, W = get_backbone_and_lm_weight(model)\n    W = W.detach()\n    cv = torch.tensor(continuation_vocab(tok), device=W.device)\n    Wcv = W[cv]\n    out = {"ids": [], "lens": [], "draft_argmax": []}\n    for v in VARIANTS:\n        out[f"{v}_ids"], out[f"{v}_logp"], out[f"{v}_cont"] = [], [], []\n        out[f"{v}_cv_id"], out[f"{v}_cv_logp"] = [], []\n    with torch.no_grad():\n        for r in records:\n            P, y = list(r["prompt_ids"]), list(r["out_ids"])\n            x = torch.tensor([P + y], device=device)\n            h = hidden_states(model, x, torch.ones_like(x))[0, len(P) - 1:len(P) - 1 + len(y)].float()\n            out["ids"].append(int(r["id"]))\n            out["lens"].append(len(y))\n            out["draft_argmax"].append(lm_logits(h, W).argmax(-1).cpu().numpy().astype(np.int32))\n            for v in VARIANTS:\n                ids_k, lp_k, cid_k, clp_k = [], [], [], []\n                for k in range(1, K + 1):\n                    z = heads[v].z(h, k)\n                    lp = torch.log_softmax(lm_logits(z, W), -1)\n                    tk = lp.topk(TK, -1)\n                    ids_k.append(tk.indices.cpu().numpy().astype(np.int32))\n                    lp_k.append(tk.values.cpu().numpy().astype(np.float16))\n                    lc = torch.log_softmax(lm_logits(z, Wcv), -1).max(-1)   # restricted to continuation tokens\n                    cid_k.append(cv[lc.indices].cpu().numpy().astype(np.int32))\n                    clp_k.append(lc.values.cpu().numpy().astype(np.float16))\n                out[f"{v}_ids"].append(np.stack(ids_k, 1))      # [n, K, TK]\n                out[f"{v}_logp"].append(np.stack(lp_k, 1))\n                out[f"{v}_cv_id"].append(np.stack(cid_k, 1))    # [n, K]\n                out[f"{v}_cv_logp"].append(np.stack(clp_k, 1))\n                out[f"{v}_cont"].append(torch.sigmoid(heads[v].cont(h)).cpu().numpy().astype(np.float16))\n    lens = np.asarray(out.pop("lens"), np.int64)\n    res = dict(ids=np.asarray(out.pop("ids"), np.int64), offsets=np.concatenate([[0], np.cumsum(lens)]),\n               cont_vocab=cv.cpu().numpy())\n    for k, v in out.items():\n        res[k] = np.concatenate(v, 0)\n    return res\n\n\ndef time_heads(model, tok, heads, cfg: dict, device) -> dict:\n    """Draft latency for q new tokens (KV cache of TIME_CTX) and the heads\' latency (all K heads in one\n    batched matmul; full vocabulary and continuation vocabulary), relative to the draft\'s q = 1 call."""\n    from .gpu_traces import sync_all, time_fwd\n    torch = _torch()\n    tq = {q: time_fwd(model, q=q, ctx=cfg["TIME_CTX"], steps=cfg["TIME_STEPS"], reps=cfg["TIME_REPS"], device=device)\n          for q in cfg["TIME_Q"]}\n    _, W = get_backbone_and_lm_weight(model)\n    Wcv = W[torch.tensor(continuation_vocab(tok), device=W.device)]\n    h = torch.randn(1, W.shape[1], device=device)\n\n    def run(Wm):\n        ts = []\n        with torch.no_grad():\n            for s in range(cfg["TIME_STEPS"] + 10):\n                sync_all()\n                t = time.perf_counter()\n                z = torch.cat([heads.z(h, k) for k in range(1, cfg["K"] + 1)], 0)\n                lm_logits(z, Wm).argmax(-1)\n                torch.sigmoid(heads.cont(h))\n                sync_all()\n                if s >= 10:\n                    ts.append(time.perf_counter() - t)\n        return float(np.median(ts))\n\n    t_full, t_cv = run(W), run(Wcv)\n    return dict(t_draft={str(q): v for q, v in tq.items()},\n                r_q={str(q): v / tq[1] for q, v in tq.items()},\n                t_heads_full=t_full, head_rel_full=t_full / tq[1],\n                t_heads_cv=t_cv, head_rel_cv=t_cv / tq[1], cont_vocab_size=int(Wcv.shape[0]),\n                ctx=cfg["TIME_CTX"])\n\n\n# --------------------------------------------------------------------------- #\n# notebook entry point\n# --------------------------------------------------------------------------- #\n\ndef _find_heads(cfg: dict) -> Path | None:\n    for root in cfg["SEARCH"]:\n        p = Path(root)\n        if p.exists():\n            for f in p.rglob("wordheads_v0.pt"):\n                return f.parent\n    return None\n\n\ndef wordheads_main(cfg_in: dict, target_records: dict) -> dict:\n    """Train v0/v1/v2 heads on the frozen draft, dump predictions along ``target_records``\n    ({split: [records with id, prompt_ids, out_ids]}), time the draft and heads, zip everything."""\n    torch = _torch()\n    from . import gpu_traces as G\n    cfg = dict(DEFAULT_WH)\n    cfg.update(cfg_in)\n    out = Path(cfg["OUT_ROOT"]) / "wordheads"\n    out.mkdir(parents=True, exist_ok=True)\n    logf = open(out / "log.txt", "a", encoding="utf-8")\n\n    def log(msg):\n        print(msg, flush=True)\n        logf.write(msg + "\\n")\n        logf.flush()\n\n    device = "cuda:0" if torch.cuda.is_available() else "cpu"\n    dtype = torch.float16 if device.startswith("cuda") else torch.float32\n    model, name = None, None\n    for nm in cfg["DRAFT"]:\n        try:\n            model = G.from_pretrained(nm, dtype).to(device)\n            name = nm\n            break\n        except Exception as e:  # noqa: BLE001\n            log(f"could not load {nm}: {e}")\n    assert model is not None, "no draft model could be loaded"\n    for p in model.parameters():\n        p.requires_grad_(False)\n    tok = G.load_tokenizer(name)\n    tok.padding_side = "right"\n    eos = G.get_eos_ids(model, tok)\n    eot = tok.convert_tokens_to_ids("<|eot_id|>")\n    eos_id = eot if (eot is not None and eot != tok.unk_token_id and eot in eos) else eos[0]\n    log(f"draft {name} on {device}; eos {eos_id}")\n\n    K = cfg["K"]\n    prev = _find_heads(cfg)\n    info = dict(draft=name, cfg={k: v for k, v in cfg.items() if k != "TRAIN_PAIRS"})\n    if prev is not None:\n        d = get_backbone_and_lm_weight(model)[1].shape[1]\n        heads = {}\n        for v in VARIANTS:\n            heads[v] = make_heads(d, K).to(device)\n            heads[v].load_state_dict({k: t.float() for k, t in torch.load(prev / f"wordheads_{v}.pt").items()})\n        log(f"heads: reused from {prev}")\n        info["train"] = json.loads((prev / "train_info.json").read_text()) if (prev / "train_info.json").exists() else {}\n    else:\n        pairs = load_train_pairs(cfg, log)\n        rnd = random.Random(cfg["SEED"])\n        rnd.shuffle(pairs)\n        ex = []\n        for en, hi in pairs:\n            p, y = encode_pair(tok, en, hi, eos_id)\n            if len(p) + len(y) <= cfg["MAX_LEN"]:\n                ex.append(make_example(tok, p, y, K))\n        val, train = ex[:cfg["N_VAL"]], ex[cfg["N_VAL"]:]\n        wi = np.concatenate([e["word_initial"] for e in train])\n        log(f"examples: train {len(train)} val {len(val)}; output tokens {len(wi)}; continuation share {1 - wi.mean():.3f}")\n        heads, tinfo = train_heads(model, tok, train, val, cfg, device, log)\n        tinfo["continuation_share"] = float(1 - wi.mean())\n        info["train"] = tinfo\n        for v in VARIANTS:\n            torch.save({k: t.half() for k, t in heads[v].state_dict().items()}, out / f"wordheads_{v}.pt")\n        (out / "train_info.json").write_text(json.dumps(tinfo, indent=1))\n    for v in VARIANTS:\n        heads[v].eval()\n\n    info["dump"] = {}\n    for split, recs in target_records.items():\n        res = dump_predictions(model, tok, heads, recs, cfg, device)\n        np.savez_compressed(out / f"heads_{split}.npz", **res)\n        info["dump"][split] = dict(records=len(recs), positions=int(res["offsets"][-1]))\n        log(f"dump {split}: {info[\'dump\'][split]}")\n    info["timing"] = time_heads(model, tok, heads["v1"], cfg, device)\n    log("timing: " + json.dumps(info["timing"]))\n    info["run"] = G.run_info(cfg.get("BUILD", "nobuild"), draft=name)\n    (out / "run_info.json").write_text(json.dumps(info, indent=1, default=str))\n    zp = G.zip_dir(out, Path(cfg["OUT_ROOT"]) / "wordheads.zip")\n    log(f"wrote {zp}")\n    logf.close()\n    return info\n',
    'refund_sim': '"""Exact teacher-forced simulation of the "token-tax refund" drafters (PLAN_wordheads.md §3).\n\nSame protocol as Phase 1: greedy, lossless speculative decoding of the 8B outputs stored in the\nPhase-1 traces. A round at committed position ``p`` drafts ``g`` tokens, the 8B accepts the leading\nmatches ``a`` and emits ``min(a + 1, n - p)`` tokens; the round costs drafting + ``verify(g)``, all in\nunits of one 8B decode step. Speed-up = emitted tokens / cost (ratio of sums over sentences).\n\nDrafters (``mode``):\n  entropy   the 1B drafts every token, stop rule ``1 - sqrt(beta H) < lam`` (Phase 1).\n  freecont  as ``entropy`` but drafted word-continuation tokens cost ``kappa * c`` (ceiling).\n  noisy     as ``freecont`` with each continuation token kept right only with probability ``rho``.\n  dict      the 1B drafts word starts; a frequency lexicon completes the word (optionally gated by\n            the top word\'s share of the matching words\' frequency, ``tau``).\n  heads     the 1B drafts a token; Medusa heads on the same hidden state propose up to K following\n            tokens (stop at a word-initial proposal, at log-prob < log ``tau_h``, or for V1/V2 at\n            continuation probability < ``tau_c``); the 1B then reads them in one call.\n\nApproximation shared with Phase 1: stopping decisions after the first wrong draft token use the\nteacher-forced (true-path) features. Accepted tokens are exact.\n"""\nfrom __future__ import annotations\n\nimport bisect\nimport json\nimport math\nimport re\nfrom collections import Counter\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .policies import entropy_fires\n\nGMAX = 8\nBETA = 4.0\nLAM_GRID = (-1.3, -1.1, -0.9, -0.8, -0.683, -0.6, -0.5, -0.4, -0.3)\nVARIANTS = ("v0", "v1", "v2")\nDEV_RE = re.compile(r"[ऀ-ॿ\u200c\u200d]+")\n\n\n# --------------------------------------------------------------------------- cost model\n@dataclass\nclass Cost:\n    c: float                       # one 1B call / one 8B decode step\n    verify: object                 # verify(g) -> cost of the 8B pass over g drafted tokens\n    r_q: dict | None = None        # 1B call reading q new tokens, relative to q = 1 (measured)\n    head_rel_cv: float = 0.0177    # all K heads, continuation vocabulary, relative to a 1B call\n    head_rel_full: float = 0.114   # all K heads, full vocabulary\n\n    def r(self, q: int) -> float:\n        """Cost of a 1B call that reads q new tokens, relative to q = 1. Measured values below 1 are\n        timing noise and are clipped to 1; beyond the measured range the last value is used."""\n        if q <= 1 or not self.r_q:\n            return 1.0\n        ks = sorted(self.r_q)\n        k = max([x for x in ks if x <= q] or [ks[0]])\n        return max(1.0, float(self.r_q[k]))\n\n\ndef load_cost(timing_path, heads_run_info=None) -> Cost:\n    from .simulator import load_timing, make_verify\n    timing = load_timing(timing_path)\n    cost = Cost(c=float(timing["c"]), verify=make_verify(timing))\n    if heads_run_info:\n        tm = json.loads(Path(heads_run_info).read_text())["timing"]\n        cost.r_q = {int(k): float(v) for k, v in tm["r_q"].items()}\n        cost.head_rel_cv = float(tm["head_rel_cv"])\n        cost.head_rel_full = float(tm["head_rel_full"])\n    return cost\n\n\n# --------------------------------------------------------------------------- data\n@dataclass\nclass Heads:\n    """Head predictions of one variant for one sentence (rows aligned with the trace positions)."""\n    ids: np.ndarray       # [n, K, TK] full-vocabulary top-k\n    logp: np.ndarray\n    cv_id: np.ndarray     # [n, K] argmax within the continuation vocabulary (renormalised log-prob)\n    cv_logp: np.ndarray\n    cont: np.ndarray      # [n, K] P(y[t+1..t+k] all continue the word of y[t])\n\n\ndef attach_heads(traces, npz_path, check_agreement: float = 0.97) -> dict:\n    """Attach ``t.heads[v]`` to every trace from ``heads_{split}.npz``; returns alignment info."""\n    z = dict(np.load(npz_path))        # decompress every array once (NpzFile re-reads per access)\n    off, ids = z["offsets"], z["ids"]\n    by_id = {int(s): i for i, s in enumerate(ids)}\n    agree = tot = 0\n    vmax = int(max(z["cont_vocab"].max(), max(int(t.target_ids.max()) for t in traces),\n                   int(z["draft_argmax"].max()), int(z["v0_ids"].max()))) + 1\n    in_cv = np.zeros(vmax, bool)\n    in_cv[z["cont_vocab"]] = True\n    for t in traces:\n        t._in_cv = in_cv\n        i = by_id[t.id]\n        a, b = int(off[i]), int(off[i + 1])\n        if b - a != t.n:\n            raise ValueError(f"sentence {t.id}: heads have {b - a} rows, trace has {t.n}")\n        agree += int((z["draft_argmax"][a:b] == t.draft_argmax).sum())\n        tot += t.n\n        t.heads = {v: Heads(z[f"{v}_ids"][a:b], z[f"{v}_logp"][a:b].astype(np.float32),\n                            z[f"{v}_cv_id"][a:b], z[f"{v}_cv_logp"][a:b].astype(np.float32),\n                            z[f"{v}_cont"][a:b].astype(np.float32)) for v in VARIANTS}\n    rate = agree / max(tot, 1)\n    if rate < check_agreement:\n        raise ValueError(f"heads rows misaligned with traces: draft argmax agreement {rate:.4f}")\n    return dict(agreement=rate, positions=tot, cont_vocab=z["cont_vocab"])\n\n\ndef devanagari_mask(tok, vocab_size: int | None = None) -> np.ndarray:\n    """mask[id] = token text contains a Devanagari character or is a bare UTF-8 fragment\n    (fragments of Llama-3 Hindi tokens are pieces of Devanagari characters)."""\n    from .vocab import token_bytes_list\n    V = vocab_size or len(tok)\n    out = np.zeros(V, bool)\n    for i, b in enumerate(token_bytes_list(tok, range(V))):\n        s = b.decode("utf-8", errors="ignore")\n        out[i] = bool(DEV_RE.search(s)) or (len(b) > 0 and not s)\n    return out\n\n\n# --------------------------------------------------------------------------- lexicon (dictionary drafter)\nclass Lexicon:\n    """Word frequencies (dev outputs + word list). ``complete`` returns the continuation tokens of the\n    most frequent word consistent with the current word prefix ([] = no proposal)."""\n\n    def __init__(self, freq: Counter, tok):\n        self.freq, self.tok = freq, tok\n        self.words = sorted(freq)\n        self._wt, self._cache = {}, {}\n\n    @classmethod\n    def build(cls, tok, output_jsonl, wordlist_csv=None):\n        freq = Counter()\n        for line in open(output_jsonl, encoding="utf-8"):\n            freq.update(DEV_RE.findall(json.loads(line)["out_text"]))\n        if wordlist_csv and Path(wordlist_csv).exists():\n            for i, line in enumerate(open(wordlist_csv, encoding="utf-8")):\n                w = line.split(",")[0].strip()\n                if i and w and w not in freq and DEV_RE.fullmatch(w):\n                    freq[w] = 1\n        return cls(freq, tok)\n\n    def tbytes(self, i: int) -> bytes:\n        tb = self.__dict__.setdefault("_tb", {})\n        if i not in tb:\n            from .vocab import token_bytes_list\n            tb[i] = token_bytes_list(self.tok, [i])[0]\n        return tb[i]\n\n    def is_prefix(self, ids) -> bool:\n        """Is the text of these tokens (one word, possibly cut mid-character) the start of a lexicon word?\n        Words that are not Devanagari cannot be judged and pass."""\n        key = tuple(ids)\n        pc = self.__dict__.setdefault("_pc", {})\n        if key not in pc:\n            txt = b"".join(self.tbytes(i) for i in key).decode("utf-8", errors="ignore").strip()\n            if not txt or not DEV_RE.fullmatch(txt):\n                pc[key] = True\n            else:\n                k = bisect.bisect_left(self.words, txt)\n                pc[key] = k < len(self.words) and self.words[k].startswith(txt)\n        return pc[key]\n\n    def wtoks(self, w):\n        if w not in self._wt:\n            self._wt[w] = self.tok.encode(" " + w, add_special_tokens=False)\n        return self._wt[w]\n\n    def candidates(self, prefix_ids: tuple, prefix_txt: str):\n        """[(word, continuation token ids)] for lexicon words whose tokenisation extends prefix_ids."""\n        key = (prefix_ids, prefix_txt)\n        if key not in self._cache:\n            lo = bisect.bisect_left(self.words, prefix_txt)\n            hi = bisect.bisect_left(self.words, prefix_txt + "\uffff")\n            k, cons = len(prefix_ids), []\n            for w in self.words[lo:hi]:\n                if w == prefix_txt:\n                    cons.append((w, ()))\n                    continue\n                t = self.wtoks(w)\n                if len(t) > k and tuple(t[:k]) == prefix_ids:\n                    cons.append((w, tuple(t[k:])))\n            self._cache[key] = cons\n        return self._cache[key]\n\n    def complete(self, prefix_ids, prefix_txt, tau: float = 0.0, min_freq: int = 1):\n        cons = self.candidates(tuple(prefix_ids), prefix_txt)\n        if not cons:\n            return ()\n        w, c = max(cons, key=lambda x: self.freq[x[0]])\n        tot = sum(self.freq[x] for x, _ in cons) or 1\n        if self.freq[w] / tot < tau or self.freq[w] < min_freq:\n            return ()\n        return c\n\n\ndef word_prefix(t, p, tok):\n    """Token ids and text of the current Devanagari word ending at position p-1 (text \'\' if none)."""\n    wi = t.labels["word_initial"]\n    if p == 0:\n        return (), ""\n    j = p - 1\n    while j > 0 and not wi[j]:\n        j -= 1\n    ids = tuple(int(x) for x in t.target_ids[j:p])\n    cache = t.labels.setdefault("_wp", {})\n    if (j, p) not in cache:\n        txt = tok.decode(list(ids)).strip()\n        cache[(j, p)] = txt if txt and DEV_RE.fullmatch(txt) else ""\n    return ids, cache[(j, p)]\n\n\n# --------------------------------------------------------------------------- one sentence\ndef _round_entropy(t, p, cost, lam, beta, kappa=None, correct=None):\n    n, H, wi = t.n, t.H, t.labels.get("word_initial")\n    m = t.match if correct is None else correct\n    g, c_draft, stop = 0, 0.0, False\n    cap = min(GMAX, n - p)\n    while g < cap and not stop:\n        pos = p + g\n        cheap = kappa is not None and not wi[pos]\n        c_draft += cost.c * (kappa if cheap else 1.0)\n        stop = entropy_fires(H[pos], lam, beta)\n        g += 1\n    ok = 0\n    while ok < g and m[p + ok]:\n        ok += 1\n    return g, ok, c_draft\n\n\ndef _round_heads(t, p, cost, lam, beta, v, src, tau_h, tau_c, gate, lex=None):\n    """1B drafts a token; heads propose the following tokens; repeat until the entropy rule fires.\n    src: "cv"   argmax within the continuation vocabulary (cheap heads);\n         "full" full-vocabulary argmax, a non-continuation token ends the proposal;\n         "lexfilter"  as "cv", but stop once the word stops being a prefix of a lexicon word;\n         "lexrerank"  full-vocabulary top-k: take the best continuation token that keeps the word a\n                      lexicon prefix (needs full-vocabulary heads, so it pays head_rel_full)."""\n    n, H, m, y = t.n, t.H, t.match, t.target_ids\n    hd = t.heads[v]\n    K = hd.cv_id.shape[1]\n    log_tau = math.log(tau_h) if tau_h > 0 else -math.inf\n    head_cost = cost.c * (cost.head_rel_cv if src in ("cv", "lexfilter") else cost.head_rel_full)\n    wi = t.labels.get("word_initial")\n    in_cv = t._in_cv\n    cap = min(GMAX, n - p)\n    g, ok, on, pending, c_draft = 0, 0, True, 0, 0.0\n    while g < cap:\n        pos = p + g\n        c_draft += cost.c * cost.r(1 + pending)       # 1B call drafts y[pos]\n        pending = 0\n        if on and m[pos]:\n            ok += 1\n        else:\n            on = False\n        g += 1\n        if entropy_fires(H[pos], lam, beta) or g >= cap:\n            break\n        if gate is not None and not gate[int(t.draft_argmax[pos])]:\n            continue\n        c_draft += head_cost                           # heads on h_pos propose y[pos+1..pos+K]\n        props = []\n        word = None\n        if src.startswith("lex"):                      # current word (true path) up to and including y[pos]\n            j0 = pos\n            while j0 > 0 and not wi[j0]:\n                j0 -= 1\n            word = [int(z) for z in y[j0:pos + 1]]\n        for j in range(min(K, cap - g)):\n            if src in ("cv", "lexfilter"):\n                x, lp = int(hd.cv_id[pos, j]), float(hd.cv_logp[pos, j])\n                if src == "lexfilter" and not lex.is_prefix(word + props + [x]):\n                    break\n            elif src == "full":\n                x, lp = int(hd.ids[pos, j, 0]), float(hd.logp[pos, j, 0])\n                if not in_cv[x]:\n                    break                              # head predicts a new word\n            else:                                      # lexrerank\n                x = None\n                for r in range(hd.ids.shape[2]):\n                    cand = int(hd.ids[pos, j, r])\n                    if not in_cv[cand]:\n                        if r == 0:\n                            break                      # most likely: a new word starts\n                        continue\n                    if lex.is_prefix(word + props + [cand]):\n                        x, lp = cand, float(hd.logp[pos, j, r])\n                        break\n                if x is None:\n                    break\n            if lp < log_tau or (tau_c is not None and hd.cont[pos, j] < tau_c):\n                break\n            props.append(x)\n        for j, x in enumerate(props):\n            if on and x == int(y[pos + 1 + j]):\n                ok += 1\n            else:\n                on = False\n        g += len(props)\n        pending = len(props)\n    return g, ok, c_draft\n\n\ndef _round_dict(t, p, cost, lam, beta, lex, tok, tau, min_freq):\n    n, H, m, y = t.n, t.H, t.match, t.target_ids\n    cap = min(GMAX, n - p)\n    g, ok, on, pending, c_draft, stop = 0, 0, True, 0, 0.0, False\n    while g < cap and not stop:\n        pos = p + g\n        comp = ()\n        if on:                                         # off-path: no prefix to complete\n            pid, ptxt = word_prefix(t, pos, tok)\n            if ptxt:\n                comp = lex.complete(pid, ptxt, tau, min_freq)\n        if comp:\n            comp = comp[:cap - g]\n            for x in comp:\n                if on and int(y[p + g]) == x:\n                    ok += 1\n                else:\n                    on = False\n                g += 1\n            pending += len(comp)\n            continue\n        c_draft += cost.c * cost.r(1 + pending)\n        pending = 0\n        if on and m[pos]:\n            ok += 1\n        else:\n            on = False\n        stop = entropy_fires(H[pos], lam, beta)\n        g += 1\n    return g, ok, c_draft\n\n\ndef simulate(traces, mode: str, cost: Cost, lam: float = -0.683, beta: float = BETA, **kw):\n    """Returns (speed-up, emitted per sentence, cost per sentence, stats)."""\n    rng = np.random.default_rng(kw.get("seed", 0))\n    E, C = np.zeros(len(traces)), np.zeros(len(traces))\n    st = Counter()\n    for i, t in enumerate(traces):\n        correct = None\n        if mode == "noisy":\n            wi = t.labels["word_initial"]\n            correct = t.match & (wi | (rng.random(t.n) < kw["rho"]))\n        p = 0\n        while p < t.n:\n            if mode == "entropy":\n                g, ok, cd = _round_entropy(t, p, cost, lam, beta)\n            elif mode in ("freecont", "noisy"):\n                g, ok, cd = _round_entropy(t, p, cost, lam, beta, kappa=kw.get("kappa", 0.05), correct=correct)\n            elif mode == "heads":\n                g, ok, cd = _round_heads(t, p, cost, lam, beta, kw["variant"], kw.get("src", "cv"),\n                                         kw.get("tau_h", 0.0), kw.get("tau_c"), kw.get("gate"), kw.get("lex"))\n            elif mode == "dict":\n                g, ok, cd = _round_dict(t, p, cost, lam, beta, kw["lex"], kw["tok"], kw.get("tau", 0.0),\n                                        kw.get("min_freq", 1))\n            else:\n                raise ValueError(mode)\n            gained = min(ok + 1, t.n - p)\n            E[i] += gained\n            C[i] += cd + cost.verify(g)\n            st["rounds"] += 1\n            st["drafted"] += g\n            st["accepted"] += ok\n            p += gained\n    return E.sum() / C.sum(), E, C, dict(st)\n\n\n# --------------------------------------------------------------------------- statistics\ndef boot_ci(E, C, B=1000, seed=0):\n    rng = np.random.default_rng(seed)\n    idx = rng.integers(0, len(E), (B, len(E)))\n    s = E[idx].sum(1) / C[idx].sum(1)\n    return [float(x) for x in np.percentile(s, [2.5, 97.5])]\n\n\ndef paired_ci(a, b, B=1000, seed=0):\n    """CI of speed-up(a) - speed-up(b) with the same sentence resamples. a, b = (E, C)."""\n    rng = np.random.default_rng(seed)\n    idx = rng.integers(0, len(a[0]), (B, len(a[0])))\n    d = a[0][idx].sum(1) / a[1][idx].sum(1) - b[0][idx].sum(1) / b[1][idx].sum(1)\n    return [float(x) for x in np.percentile(d, [2.5, 97.5])]\n\n\ndef tune(dev, grid: list[dict], mode: str, cost: Cost, **fixed):\n    """Best kwargs on dev (ties -> first). Returns (best, [(kwargs, dev speed-up)])."""\n    scored = [(g, simulate(dev, mode, cost, **{**fixed, **g})[0]) for g in grid]\n    best = max(scored, key=lambda x: x[1])[0]\n    return best, scored\n\n\n# --------------------------------------------------------------------------- head diagnostics\ndef head_accuracy(traces, variant: str, src: str = "cv") -> dict:\n    """Along the 8B outputs: share of word-continuation targets y[t+k] (all of y[t+1..t+k] continue\n    the word of y[t]) that head k gets right, vs the 1B\'s own accuracy on those tokens (teacher forced).\n    Also rho = P(head right | 1B right) on the first continuation token."""\n    K = traces[0].heads[variant].cv_id.shape[1]\n    hit, b1, n = np.zeros(K), np.zeros(K), np.zeros(K)\n    both = b1_first = 0\n    for t in traces:\n        hd, wi, y, m = t.heads[variant], t.labels["word_initial"], t.target_ids, t.match\n        pred = hd.cv_id if src == "cv" else hd.ids[:, :, 0]\n        for pos in range(t.n):\n            for k in range(1, K + 1):\n                q = pos + k\n                if q >= t.n or wi[pos + 1:q + 1].any():\n                    break\n                n[k - 1] += 1\n                hit[k - 1] += pred[pos, k - 1] == y[q]\n                b1[k - 1] += m[q]\n                if k == 1 and m[q]:\n                    b1_first += 1\n                    both += pred[pos, 0] == y[q]\n    return dict(n=n.astype(int).tolist(), head_acc=(hit / np.maximum(n, 1)).round(4).tolist(),\n                draft_acc=(b1 / np.maximum(n, 1)).round(4).tolist(), rho_first=round(both / max(b1_first, 1), 4))\n',
    'checked': '"""Checked drafting: a cheap source proposes tokens, the 1B checks them inside a draft call it makes anyway.\n\nOne 1B call reads the pending token plus a proposal F and gives its own argmax after every token read, so it\nyields the leading tokens of F it agrees with plus one more of its own. Every drafted token is therefore the\n1B\'s own greedy token (the draft chain does not change, only the number of 1B calls), and the 8B verifies the\ndraft as usual. The idea is the vertical cascade of Cascade Speculative Drafting with an n-gram at the bottom;\nhere it is applied to this project\'s draft, entropy rule and word-completion heads.\n\nProposals: the heads\' continuation tokens (stop at probability < ``tau_h``), extended by the n-gram up to ``L``\ntokens. The entropy rule decides only whether another 1B call is made; tokens a call has produced are kept.\nThe live counterpart is ``heads_loop.HeadsDecoder`` (method kind "checked").\n"""\nfrom __future__ import annotations\n\nimport math\nfrom collections import Counter\n\nimport numpy as np\n\nfrom .policies import entropy_fires\nfrom .refund_sim import BETA, GMAX\n\n\nclass NGram:\n    """Most frequent next token after the longest known context of up to ``order`` tokens (contexts seen at\n    least twice). Stored as sorted integer keys, looked up by bisection."""\n\n    def __init__(self, vals, tab: dict):\n        self.vals = np.asarray(vals)                       # token of each compact id; id 0 separates sentences\n        self.tab = {int(k): v for k, v in tab.items()}     # k -> (sorted context keys, next tokens)\n        self.order, self.V = max(self.tab), len(self.vals)\n        self.ix = {int(v): i for i, v in enumerate(self.vals) if v >= 0}\n\n    @classmethod\n    def build(cls, seqs, order: int = 4) -> "NGram":\n        flat = np.concatenate([np.asarray(list(s) + [-1], dtype=np.int64) for s in seqs])\n        vals, inv = np.unique(flat, return_inverse=True)\n        assert vals[0] == -1 and len(vals) ** order < 2 ** 63, "contexts of this order do not fit into 63 bits"\n        tab, n = {}, len(inv)\n        for k in range(1, order + 1):\n            m = max(n - k, 0)\n            key, ok = np.zeros(m, dtype=np.uint64), np.ones(m, dtype=bool)\n            for j in range(k):\n                c = inv[j:j + m].astype(np.uint64)\n                ok &= c > 0\n                key = key * np.uint64(len(vals)) + c\n            nx = inv[k:k + m]\n            ok &= nx > 0\n            key, nx = key[ok], nx[ok].astype(np.uint64)\n            if not len(key):\n                tab[k] = (np.zeros(0, np.uint64), np.zeros(0, np.int32))\n                continue\n            o = np.lexsort((nx, key))\n            key, nx = key[o], nx[o]\n            st = np.flatnonzero(np.r_[True, (key[1:] != key[:-1]) | (nx[1:] != nx[:-1])])\n            cnt = np.diff(np.r_[st, len(key)])\n            key, nx = key[st], nx[st]\n            o = np.lexsort((-cnt, key))                    # ties: smallest token id\n            key, nx, cnt = key[o], nx[o], cnt[o]\n            first = np.r_[True, key[1:] != key[:-1]]\n            keep = np.add.reduceat(cnt, np.flatnonzero(first)) >= 2\n            tab[k] = (key[first][keep], vals[nx[first][keep].astype(np.int64)].astype(np.int32))\n        return cls(vals, tab)\n\n    def save(self, path) -> None:\n        np.savez_compressed(path, vals=self.vals, **{f"k{k}": a for k, (a, _) in self.tab.items()},\n                            **{f"n{k}": b for k, (_, b) in self.tab.items()})\n\n    @classmethod\n    def load(cls, path) -> "NGram":\n        with np.load(path, allow_pickle=False) as z:\n            return cls(z["vals"], {int(f[1:]): (z[f], z["n" + f[1:]]) for f in z.files if f[0] == "k"})\n\n    def next(self, ctx):\n        ids = [self.ix.get(int(c)) for c in ctx[-self.order:]]\n        for k in range(len(ids), 0, -1):\n            if None in ids[-k:]:\n                continue\n            key = 0\n            for c in ids[-k:]:\n                key = key * self.V + c\n            ks, nx = self.tab[k]\n            i = int(np.searchsorted(ks, np.uint64(key)))\n            if i < len(ks) and int(ks[i]) == key:\n                return int(nx[i])\n        return None\n\n    def chain(self, ctx, L: int) -> list[int]:\n        ctx, out = [int(c) for c in ctx[-self.order:]], []\n        while len(out) < L:\n            x = self.next(ctx)\n            if x is None:\n                break\n            out.append(x)\n            ctx.append(x)\n        return out\n\n\ndef proposal(ctx, head_tokens, ngram, L: int, room: int) -> list[int]:\n    """Tokens to append after ``ctx`` (output so far): the heads\' tokens, extended by the n-gram to ``L``."""\n    F = list(head_tokens)\n    if ngram is not None and len(F) < L:\n        F += ngram.chain([int(x) for x in ctx[-ngram.order:]] + F, L - len(F))\n    return F[:max(room, 0)]\n\n\n# --------------------------------------------------------------------------- exact simulation\ndef round_checked(t, p, cost, lam, beta=BETA, variant=None, tau_h=0.0, ngram=None, L=6, gate=None, memo=None):\n    """(drafted, accepted by the 8B, draft cost, 1B calls). After a wrong 1B token (off the 8B\'s path, no\n    traces) nothing is proposed and every further token costs a call; those tokens are lost anyway."""\n    n, H, m, a, y = t.n, t.H, t.match, t.draft_argmax, t.target_ids\n    hd = t.heads[variant] if variant else None\n    log_tau = math.log(tau_h) if tau_h > 0 else -math.inf\n    cap = min(GMAX, n - p)\n    g, ok, on, cd, calls = 0, 0, True, 0.0, 0\n    while g < cap:\n        pos = p + g\n        F = []\n        if on and cap - g > 1:\n            head = []\n            if hd is not None and calls and (gate is None or gate[int(a[pos - 1])]):\n                for j in range(hd.cv_id.shape[1]):         # heads ran on the state that drafted pos - 1\n                    if float(hd.cv_logp[pos - 1, j]) < log_tau:\n                        break\n                    head.append(int(hd.cv_id[pos - 1, j]))\n            if memo is not None and not head:\n                if pos not in memo:\n                    memo[pos] = proposal(y[:pos], [], ngram, L, L)\n                F = memo[pos][:cap - g - 1]\n            else:\n                F = proposal(y[:pos], head, ngram, L, cap - g - 1)\n        cd += cost.c * cost.r(1 + len(F))\n        calls += 1\n        j = 0\n        while True:\n            q = pos + j\n            if on and m[q]:\n                ok += 1\n            else:\n                on = False\n            g += 1\n            if j < len(F) and on and int(a[q]) == F[j]:    # the 1B agrees: the next token is free\n                j += 1\n                continue\n            break\n        if entropy_fires(H[pos + j], lam, beta) or g >= cap:\n            break\n        if hd is not None:\n            cd += cost.c * cost.head_rel_cv\n    return g, ok, cd, calls\n\n\ndef simulate_checked(traces, cost, lam, beta=BETA, **kw):\n    """Same conventions and return value as ``refund_sim.simulate``."""\n    E, C, st = np.zeros(len(traces)), np.zeros(len(traces)), Counter()\n    ng, L = kw.get("ngram"), kw.get("L", 6)\n    for i, t in enumerate(traces):\n        memo = t.labels.setdefault("_ng", {}).setdefault((ng, L), {})\n        p = 0\n        while p < t.n:\n            g, ok, cd, calls = round_checked(t, p, cost, lam, beta, memo=memo, **kw)\n            gained = min(ok + 1, t.n - p)\n            E[i] += gained\n            C[i] += cd + cost.verify(g)\n            for k, v in (("rounds", 1), ("drafted", g), ("accepted", ok), ("calls", calls)):\n                st[k] += v\n            p += gained\n    return E.sum() / C.sum(), E, C, dict(st)\n',
    'heads_loop': '"""Real greedy speculative-decoding loop with word-completion heads on the draft (batch size 1, KV caches).\n\nLive counterpart of ``refund_sim._round_heads``: the 1B drafts a token; unless the entropy rule fires, the\nheads on the *same* hidden state propose up to K continuation tokens (argmax within the continuation\nvocabulary; a proposal chain stops at head probability < ``tau_h``, continuation probability < ``tau_c``,\nor when the word stops being a lexicon prefix); the next 1B call reads them in one forward. The 8B verifies\neverything, so the output is the 8B\'s own greedy output.\n\nChecked methods (kind "checked", ``checked.py``): an n-gram and/or the heads propose tokens and the 1B checks\nthem inside its next call, so every drafted token is the 1B\'s own greedy token.\n\nAll methods (NoDraft, Entropy, heads, checked) run through the same code, so differences between them come from the\ndrafter alone. Differences from the Phase-1 loop (``spec_loop.SpecDecoder``), applied to every method:\n  * drafting stops at an end-of-text token (Phase 1 kept drafting past it, 3.3% of Entropy\'s time);\n  * NoDraft never calls the draft model (Phase 1 prefilled its cache anyway);\n  * the draft reads the prompt in its first draft call instead of in a separate prefill call;\n  * LM-head logits are computed only for the position that is needed.\n\nFiles written to ``OUT_ROOT/heads_loop`` (zipped to ``heads_loop.zip``): greedy_unbatched.jsonl,\nloop_results.jsonl (one line per decode, with every round), timing.json, summary.json, SUMMARY.md, run_info.json.\n"""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport math\nimport time\nimport unicodedata\nimport zipfile\nfrom collections import Counter\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .checked import NGram, proposal\nfrom .policies import entropy_fires\nfrom .refund_sim import BETA, DEV_RE, GMAX, Lexicon, devanagari_mask\nfrom .spec_loop import crop_to, first_divergence\nfrom .vocab import ZW, first_char, special_ids, token_bytes_list\nfrom .wordheads import continuation_vocab, get_backbone_and_lm_weight, lm_logits, make_heads\n\nPAPER_LEXICON_TYPES = 72797      # refund_sim.Lexicon.build (feasibility.json: 72907, which also counts non-Devanagari rows)\nPAPER_CONT_VOCAB = 1406          # paper, §4\n\n\ndef _torch():\n    import torch\n    return torch\n\n\ndef _sync():\n    torch = _torch()\n    if torch.cuda.is_available():\n        for i in range(torch.cuda.device_count()):\n            torch.cuda.synchronize(i)\n\n\n# --------------------------------------------------------------------------- #\n# causal word starts\n# --------------------------------------------------------------------------- #\n\ndef _last_char(b: bytes) -> str:\n    """Last complete character of ``b`` (\'\' if ``b`` ends inside a character)."""\n    for k in range(1, min(4, len(b)) + 1):\n        if not 0x80 <= b[-k] <= 0xBF:                # ASCII or lead byte of the last character\n            try:\n                return b[-k:].decode("utf-8")\n            except UnicodeDecodeError:\n                return ""\n    return ""\n\n\ndef _separator(ch: str) -> bool:\n    return bool(ch) and (ch.isspace() or unicodedata.category(ch).startswith("P"))\n\n\nclass WordTracker:\n    """Which token starts the current word, from token bytes of the text so far only. Agrees with the gold\n    ``word_initial`` labels the simulator uses (``boundaries.token_labels``), but never looks ahead and\n    never fails on invalid UTF-8 (drafts can contain it)."""\n\n    def __init__(self, tok):\n        self.tok, self._b, self.spec = tok, {}, special_ids(tok)\n\n    def bytes(self, i: int) -> bytes:\n        i = int(i)\n        if i not in self._b:\n            self._b[i] = token_bytes_list(self.tok, [i])[0]\n        return self._b[i]\n\n    def is_initial(self, ids, j: int) -> bool:\n        if j <= 0 or int(ids[j]) in self.spec or int(ids[j - 1]) in self.spec:\n            return True\n        b = self.bytes(ids[j])\n        if not b or 0x80 <= b[0] <= 0xBF:            # empty, or continues a character\n            return False\n        ch = first_char(b)\n        if _separator(ch):\n            return True\n        if ch and (unicodedata.category(ch).startswith("M") or ch in ZW):\n            return False                             # combining mark: attaches to the previous character\n        return _separator(_last_char(b"".join(self.bytes(x) for x in ids[max(0, j - 4):j])))\n\n    def word(self, ids) -> list[int]:\n        """Tokens of the word that the last token of ``ids`` belongs to."""\n        j = len(ids) - 1\n        while j > 0 and not self.is_initial(ids, j):\n            j -= 1\n        return [int(x) for x in ids[j:]]\n\n\n# --------------------------------------------------------------------------- #\n# decoder\n# --------------------------------------------------------------------------- #\n\n@dataclass\nclass Method:\n    """kind: "nodraft" | "entropy" | "heads" | "checked" (``variant`` None: no heads)."""\n    name: str\n    kind: str\n    lam: float = -0.683\n    beta: float = BETA\n    variant: str | None = "v1"\n    tau_h: float = 0.0\n    tau_c: float | None = None\n    lexicon: bool = False\n    ngram: bool = False\n    L: int = 6                       # checked: proposal length\n\n    def params(self) -> dict:\n        if self.kind == "nodraft":\n            return {}\n        d = dict(lam=self.lam, beta=self.beta)\n        if self.kind == "heads":\n            d.update(variant=self.variant, tau_h=self.tau_h, tau_c=self.tau_c, lexicon=self.lexicon)\n        if self.kind == "checked":\n            d.update(variant=self.variant, tau_h=self.tau_h, ngram=self.ngram, L=self.L)\n        return d\n\n\n@dataclass\nclass HeadsResult:\n    out_ids: list\n    rounds: list = field(default_factory=list)  # p, g, accepted, calls, props, props_ok, t_draft, t_verify\n    wall: float = 0.0\n    prefill: float = 0.0                        # target prefill (the draft reads the prompt in its first call)\n\n\nclass HeadsDecoder:\n    def __init__(self, target, draft, tok, eos_ids, device: str = "cuda:0", heads: dict | None = None,\n                 lexicon=None, gate=None, cont_vocab=None, gamma_max: int = GMAX, noise: float = 0.0, ngram=None):\n        """``heads``: {variant: module}; ``gate``: bool array, token may be continued by the heads\n        (``devanagari_mask``); ``noise``: CPU tests only, Gaussian logit noise on the draft (seeded by the\n        last token read) so drafts get rejected even when draft and target are the same model."""\n        torch = _torch()\n        self.torch, self.target, self.draft, self.device = torch, target, draft, device\n        self.base, W = get_backbone_and_lm_weight(draft)\n        self.W = W.detach()\n        self.heads, self.lexicon, self.gate, self.ngram = heads or {}, lexicon, gate, ngram\n        self.words = WordTracker(tok)\n        self.eos = set(int(i) for i in eos_ids)\n        self.gamma_max, self.noise = gamma_max, float(noise)\n        if self.heads:\n            self.cv = [int(i) for i in (continuation_vocab(tok) if cont_vocab is None else cont_vocab)]\n            self.Wcv = self.W[torch.tensor(self.cv, device=self.W.device)]\n            self.K = next(iter(self.heads.values())).K\n\n    # ---- model calls ---------------------------------------------------- #\n    def _draft_call(self, ids: list[int], cache, heads=None, F=()):\n        """Feed ``ids`` and the proposal ``F`` to the draft. Returns (n, token, entropy, props): the draft\'s own\n        greedy continuation is F[:n] + [token]; entropy and props [(token, log p, p_cont)] * K (None without\n        heads) belong to ``token``. Everything crosses to the CPU in one transfer."""\n        torch = self.torch\n        fed = list(ids) + list(F)\n        x = torch.tensor([fed], device=self.device)\n        with torch.no_grad():\n            h = self.base(input_ids=x, past_key_values=cache, use_cache=True).last_hidden_state[0, -1 - len(F):]\n            lg = lm_logits(h, self.W)\n            if self.noise:\n                lg = lg + self.noise * torch.stack([\n                    torch.randn(lg.shape[1], generator=torch.Generator().manual_seed(1_000_003 + int(t)))\n                    for t in fed[-1 - len(F):]]).to(lg.device)\n            parts = []\n            if F:\n                am = lg.argmax(-1)\n                n = (am[:-1] == x[0, -len(F):]).long().cumprod(0).sum()      # tokens of F the draft agrees with\n                h, lg = h[n][None], lg[n]\n                parts.append(n[None].float())\n            else:\n                lg = lg[0]\n            lp = torch.log_softmax(lg, -1)\n            parts += [lp.argmax(-1)[None].float(), -(lp.exp() * lp).sum(-1)[None]]\n            if heads is not None:\n                hf = h.float()\n                z = torch.cat([heads.z(hf, k) for k in range(1, self.K + 1)], 0)\n                top = torch.log_softmax(lm_logits(z, self.Wcv), -1).max(-1)\n                parts += [top.indices.float(), top.values, torch.sigmoid(heads.cont(hf))[0]]\n            v = torch.cat(parts).tolist()            # ids < 2^24 are exact in float32\n        n = int(v.pop(0)) if F else 0\n        if heads is None:\n            return n, int(v[0]), v[1], None\n        K = self.K\n        props = [(self.cv[int(v[2 + k])], v[2 + K + k], v[2 + 2 * K + k]) for k in range(K)]\n        return n, int(v[0]), v[1], props\n\n    def _verify(self, ids: list[int], cache) -> list[int]:\n        x = self.torch.tensor([ids], device=self.device)\n        with self.torch.no_grad():\n            return self.target(input_ids=x, past_key_values=cache, use_cache=True).logits[0].argmax(-1).tolist()\n\n    # ---- drafting one round ---------------------------------------------- #\n    def _propose(self, m: Method, text: list[int], props, room: int) -> list[int]:\n        """Head proposals to append after ``text`` (output so far, ending in the 1B\'s drafted token)."""\n        if room <= 0 or (self.gate is not None and not self.gate[text[-1]]):\n            return []\n        word = self.words.word(text) if m.lexicon else None\n        log_tau = math.log(m.tau_h) if m.tau_h > 0 else -math.inf\n        out: list[int] = []\n        for x, lp, pc in props[:room]:\n            if m.lexicon and not self.lexicon.is_prefix(word + out + [x]):\n                break\n            if lp < log_tau or (m.tau_c is not None and pc < m.tau_c):\n                break\n            out.append(x)\n        return out\n\n    def _checked_round(self, m: Method, seq: list[int], n_prompt: int, cache, cap: int):\n        """Returns (drafted, proposed tokens read, proposed tokens the 1B agreed with, 1B calls)."""\n        heads = self.heads[m.variant] if m.variant else None\n        drafted: list[int] = []\n        props, calls, read, agreed = None, 0, 0, 0\n        while len(drafted) < cap:\n            cur = seq + drafted\n            text = cur[n_prompt:]\n            room = cap - len(drafted) - 1\n            head = self._propose(m, text, props, room) if props else []\n            F = proposal(text, head, self.ngram if m.ngram else None, m.L, room)\n            n, tok, H, props = self._draft_call(cur[cache.get_seq_length():], cache, heads, F)\n            if n < len(F):\n                crop_to(cache, len(cur) + n)          # forget the rejected part of the proposal\n            calls, read, agreed = calls + 1, read + len(F), agreed + n\n            new = F[:n] + [tok]\n            end = next((i + 1 for i, t in enumerate(new) if t in self.eos), None)\n            drafted += new[:end]\n            if end is not None or entropy_fires(H, m.lam, m.beta):\n                break\n        return drafted, read, agreed, calls\n\n    def _draft_round(self, m: Method, seq: list[int], n_prompt: int, cache, cap: int):\n        drafted: list[int] = []\n        from_head: list[bool] = []\n        calls = 0\n        heads = self.heads[m.variant] if m.kind == "heads" else None\n        while len(drafted) < cap:\n            cur = seq + drafted\n            _, tok, H, props = self._draft_call(cur[cache.get_seq_length():], cache, heads)\n            calls += 1\n            drafted.append(tok)\n            from_head.append(False)\n            if tok in self.eos or entropy_fires(H, m.lam, m.beta) or len(drafted) >= cap:\n                break\n            if heads is not None:\n                extra = self._propose(m, seq[n_prompt:] + drafted, props, cap - len(drafted))\n                drafted += extra\n                from_head += [True] * len(extra)\n        return drafted, from_head, calls\n\n    # ---- decoding --------------------------------------------------------- #\n    def decode(self, prompt_ids: list[int], m: Method, max_new: int) -> HeadsResult:\n        from transformers import DynamicCache\n        torch = self.torch\n        tc, dc = DynamicCache(), DynamicCache()\n        res = HeadsResult(out_ids=[])\n        seq, n_prompt = list(prompt_ids), len(prompt_ids)\n        _sync()\n        t_start = time.perf_counter()\n        if len(seq) > 1:                              # target prefill; the pending token is prompt[-1]\n            with torch.no_grad():\n                self.target(input_ids=torch.tensor([seq[:-1]], device=self.device), past_key_values=tc,\n                            use_cache=True, logits_to_keep=1)\n        _sync()\n        res.prefill = time.perf_counter() - t_start\n        while True:\n            p = len(seq) - n_prompt\n            t0 = time.perf_counter()\n            drafted, from_head, calls = ([], [], 0)\n            read = agreed = 0\n            cap = min(self.gamma_max, max_new - p)\n            if m.kind == "checked" and cap > 0:\n                drafted, read, agreed, calls = self._checked_round(m, seq, n_prompt, dc, cap)\n            elif m.kind != "nodraft" and cap > 0:\n                drafted, from_head, calls = self._draft_round(m, seq, n_prompt, dc, cap)\n            _sync()\n            t1 = time.perf_counter()\n            am = self._verify([seq[-1]] + drafted, tc)\n            a = 0\n            while a < len(drafted) and am[a] == drafted[a]:\n                a += 1\n            new = drafted[:a] + [am[a]]\n            _sync()\n            t2 = time.perf_counter()\n            seq += new\n            crop_to(tc, len(seq) - 1)\n            if m.kind != "nodraft":\n                crop_to(dc, len(seq) - 1)\n            res.rounds.append(dict(p=p, g=len(drafted), accepted=a, calls=calls, props=sum(from_head) + read,\n                                   props_ok=sum(from_head[:a]) + agreed, t_draft=t1 - t0, t_verify=t2 - t1))\n            if any(t in self.eos for t in new) or len(seq) - n_prompt >= max_new:\n                break\n        _sync()\n        res.wall = time.perf_counter() - t_start\n        out = seq[n_prompt:]\n        cut = next((i + 1 for i, t in enumerate(out) if t in self.eos), len(out))\n        res.out_ids = out[:min(cut, max_new)]\n        return res\n\n\n# --------------------------------------------------------------------------- #\n# tuned parameters (built into the notebook) and methods\n# --------------------------------------------------------------------------- #\n\nTABLE_ROWS = {"V0": ("Heads V0 (cv vocab)", "v0", False), "V1": ("Heads V1 (cv vocab)", "v1", False),\n              "V2": ("Heads V2 (cv vocab)", "v2", False),\n              "V0+lex": ("Heads V0 (cont. vocab + lexicon filter)", "v0", True),\n              "V1+lex": ("Heads V1 (cont. vocab + lexicon filter)", "v1", True)}\n\n\n# name: (heads variant, n-gram); parameters from results/checked/params.json (scripts/run_checked_sim.py)\nCHECKED_ROWS = {"NGram": (None, True), "V0+check": ("v0", False), "V1+check": ("v1", False),\n                "NGram+V0": ("v0", True), "NGram+V1": ("v1", True)}\n\n\ndef variant_of(name: str) -> str | None:\n    return TABLE_ROWS[name][1] if name in TABLE_ROWS else CHECKED_ROWS.get(name, (None,))[0]\n\n\ndef tuned_params(main_table_csv) -> dict:\n    """{method: {lam, tau_h, tau_c, sim}} from a main_table.csv of ``scripts/run_refund_sim.py`` (dev-tuned;\n    ``sim`` is the simulated devtest speed-up, for reference)."""\n    rows = {r["method"]: r for r in csv.DictReader(open(main_table_csv, encoding="utf-8"))}\n    ent = rows["Entropy (Phase 1, lam=-0.683)"]\n    out = {"Entropy": dict(lam=json.loads(ent["params"])["lam"], sim=float(ent["speedup"]))}\n    for name, (row, _, _) in TABLE_ROWS.items():\n        p = json.loads(rows[row]["params"])\n        out[name] = dict(lam=p["lam"], tau_h=p["tau_h"], tau_c=p["tau_c"], sim=float(rows[row]["speedup"]))\n    return out\n\n\ndef build_methods(names, params: dict) -> list[Method]:\n    out = []\n    for n in names:\n        if n == "NoDraft":\n            out.append(Method(n, "nodraft"))\n        elif n == "Entropy":\n            out.append(Method(n, "entropy", lam=params[n]["lam"]))\n        elif n in CHECKED_ROWS:\n            (v, ng), p = CHECKED_ROWS[n], params[n]\n            out.append(Method(n, "checked", lam=p["lam"], variant=v, tau_h=p["tau_h"], ngram=ng, L=p["L"]))\n        else:\n            _, v, lex = TABLE_ROWS[n]\n            p = params[n]\n            out.append(Method(n, "heads", lam=p["lam"], variant=v, tau_h=p["tau_h"], tau_c=p["tau_c"], lexicon=lex))\n    return out\n\n\n# --------------------------------------------------------------------------- #\n# inputs: heads, word list, earlier run\n# --------------------------------------------------------------------------- #\n\ndef _n_train(d: Path) -> int:\n    """Training pairs of a heads folder (run_info.json, else estimated from the step count)."""\n    try:\n        return int(json.loads((d / "run_info.json").read_text(encoding="utf-8"))["cfg"]["N_TRAIN"])\n    except Exception:  # noqa: BLE001\n        try:\n            return 32 * int(json.loads((d / "train_info.json").read_text(encoding="utf-8"))["steps"])\n        except Exception:  # noqa: BLE001\n            return 0\n\n\ndef find_heads(search, variants, work: Path, log=print) -> Path:\n    """Folder with ``wordheads_<v>.pt`` for every variant; with several, the one trained on most pairs.\n    Zips holding such files are unpacked into ``work``."""\n    need = [f"wordheads_{v}.pt" for v in variants]\n    keep = set(need) | {"run_info.json", "train_info.json"}\n    cands: list[Path] = []\n    for root in map(Path, search):\n        if not root.exists():\n            continue\n        for f in sorted(root.rglob(need[0])):\n            if all((f.parent / n).exists() for n in need):\n                cands.append(f.parent)\n        for i, zp in enumerate(sorted([root] if root.is_file() else root.rglob("*.zip"))):\n            try:\n                with zipfile.ZipFile(zp) as z:\n                    members = {Path(n.replace("\\\\", "/")).name: n for n in z.namelist()}\n                    if not set(need) <= set(members):\n                        continue\n                    dst = Path(work) / f"{zp.stem}_{i}"\n                    dst.mkdir(parents=True, exist_ok=True)\n                    for name in keep & set(members):\n                        (dst / name).write_bytes(z.read(members[name]))\n                    cands.append(dst)\n            except (zipfile.BadZipFile, OSError):\n                continue\n    if not cands:\n        raise FileNotFoundError(f"no {need} under {list(map(str, search))}: attach the output of notebook 07b "\n                                "(wordheads.zip, or the notebook itself via Add Input -> Your Work)")\n    best = max(cands, key=_n_train)\n    for c in cands:\n        log(f"  heads found: {c} (N_TRAIN = {_n_train(c)}){\'  <- used\' if c is best else \'\'}")\n    return best\n\n\ndef find_file(search, name: str) -> Path | None:\n    for root in map(Path, search):\n        if root.exists():\n            for f in sorted(root.rglob(name)):\n                if f.is_file():\n                    return f\n    return None\n\n\ndef build_lexicon(tok, dev_out_ids, wordlist_csv) -> Lexicon:\n    """Same words as ``refund_sim.Lexicon.build``: words of the 8B\'s dev outputs + the word list."""\n    freq = Counter()\n    for ids in dev_out_ids:\n        freq.update(DEV_RE.findall(tok.decode(list(ids), skip_special_tokens=True).strip()))\n    if wordlist_csv:\n        for i, line in enumerate(open(wordlist_csv, encoding="utf-8")):\n            w = line.split(",")[0].strip()\n            if i and w and w not in freq and DEV_RE.fullmatch(w):\n                freq[w] = 1\n    return Lexicon(freq, tok)\n\n\ndef read_records(path) -> list[dict]:\n    """jsonl records; lines cut off by a killed run are dropped."""\n    out = []\n    if Path(path).exists():\n        for line in open(path, encoding="utf-8"):\n            try:\n                out.append(json.loads(line))\n            except json.JSONDecodeError:\n                continue\n    return out\n\n\ndef resume_files(search, out_dir: Path, log=print) -> None:\n    """Copy the finished decodes of an earlier run (a ``heads_loop`` folder or zip in the inputs)."""\n    names = ("greedy_unbatched.jsonl", "loop_results.jsonl", "timing.json")\n    for root in map(Path, search):\n        if not root.exists():\n            continue\n        src = next((f.parent for f in sorted(root.rglob("loop_results.jsonl")) if f.parent.name == "heads_loop"), None)\n        zp = None if src else next((z for z in sorted(root.rglob("heads_loop.zip"))), None)\n        for n in names:\n            dst = out_dir / n\n            if dst.exists():\n                continue\n            try:\n                if src and (src / n).exists():\n                    dst.write_bytes((src / n).read_bytes())\n                elif zp:\n                    with zipfile.ZipFile(zp) as z:\n                        dst.write_bytes(z.read(n))\n                else:\n                    continue\n                log(f"  resume: {n} from {src or zp}")\n            except (KeyError, zipfile.BadZipFile):\n                continue\n\n\ndef _append(path, rec) -> None:\n    with open(path, "a", encoding="utf-8") as fh:\n        fh.write(json.dumps(rec, ensure_ascii=False) + "\\n")\n\n\n# --------------------------------------------------------------------------- #\n# summary\n# --------------------------------------------------------------------------- #\n\ndef summarize_loop(results: list[dict], names: list[str], sim: dict | None = None, n_boot: int = 1000) -> list[dict]:\n    """Per method: wall-clock speed-up over NoDraft (ratio of sums of per-sentence medians over repeats, as in\n    Phase 1), the same without the prefill, bootstrap CI over sentences, paired difference to Entropy, and\n    round statistics. Only sentences with at least one decode of every method count."""\n    from .stats import bootstrap_ci, paired_bootstrap\n    by: dict = {}\n    for r in results:\n        by.setdefault(r["method"], {}).setdefault(r["id"], []).append(r)\n    ids = sorted(set.intersection(*[set(by.get(n, {})) for n in names])) if names else []\n    if not ids or "NoDraft" not in names:\n        return []\n    W = {n: np.array([np.median([r["wall"] for r in by[n][i]]) for i in ids]) for n in names}\n    D = {n: np.array([np.median([r["wall"] - r["prefill"] for r in by[n][i]]) for i in ids]) for n in names}\n    N = {n: np.array([by[n][i][0]["n_out"] for i in ids]) for n in names}\n    rows = []\n    for n in names:\n        ci = bootstrap_ci(W["NoDraft"], W[n], n_boot=n_boot)\n        cd = bootstrap_ci(D["NoDraft"], D[n], n_boot=n_boot)\n        rs = [r for i in ids for r in by[n][i]]\n        rd = [x for r in rs for x in r["rounds"]]\n        s = lambda k: sum(x[k] for x in rd)  # noqa: E731\n        row = dict(method=n, sentences=len(ids), repeats=max(len(by[n][i]) for i in ids),\n                   speedup_wall=ci["point"], lo=ci["lo"], hi=ci["hi"],\n                   speedup_decode_only=cd["point"], decode_lo=cd["lo"], decode_hi=cd["hi"],\n                   speedup_tokens_per_s=float((N[n].sum() / W[n].sum()) / (N["NoDraft"].sum() / W["NoDraft"].sum())),\n                   lossless=float(np.mean([r["lossless"] for r in rs])),\n                   tokens_per_round=sum(r["n_out"] for r in rs) / max(len(rd), 1),\n                   drafted_per_round=s("g") / max(len(rd), 1), draft_calls_per_round=s("calls") / max(len(rd), 1),\n                   head_tokens_per_round=s("props") / max(len(rd), 1),\n                   head_tokens_accepted=s("props_ok") / max(s("props"), 1) if s("props") else None,\n                   sim_devtest=(sim or {}).get(n, {}).get("sim"))\n        if "Entropy" in names and n not in ("Entropy", "NoDraft"):\n            d = paired_bootstrap(W["NoDraft"], W[n], W["NoDraft"], W["Entropy"], n_boot=n_boot)\n            row.update(diff_vs_entropy=d["diff"], diff_lo=d["lo"], diff_hi=d["hi"])\n        rows.append(row)\n    return rows\n\n\ndef summary_md(rows: list[dict], info: dict) -> str:\n    f = lambda x, k=3: "" if x is None else f"{x:.{k}f}"  # noqa: E731\n    L = ["# Real GPU loop with word-completion heads", "",\n         f"{rows[0][\'sentences\']} devtest sentences, {rows[0][\'repeats\']} repeats, "\n         f"heads: {info.get(\'heads_tag\')} pairs. Wall-clock, NoDraft = 1. `sim` is the simulated speed-up on all "\n         "1,012 devtest sentences (simulation has no prefill, so compare it with `decode only`).", "",\n         "| method | wall-clock [95% CI] | decode only | Δ vs Entropy (wall) [95% CI] | sim | lossless | tokens/round | "\n         "1B calls/round | proposed tokens/round | proposed tokens accepted |",\n         "|---|---|---|---|---|---|---|---|---|---|"]\n    for r in rows:\n        d = (f"{r[\'diff_vs_entropy\']:+.3f} [{r[\'diff_lo\']:+.3f}, {r[\'diff_hi\']:+.3f}]" if "diff_vs_entropy" in r else "")\n        L.append(f"| {r[\'method\']} | {f(r[\'speedup_wall\'])} [{f(r[\'lo\'])}, {f(r[\'hi\'])}] | {f(r[\'speedup_decode_only\'])} | {d} "\n                 f"| {f(r[\'sim_devtest\'])} | {f(r[\'lossless\'], 2)} | {f(r[\'tokens_per_round\'])} | {f(r[\'draft_calls_per_round\'])} "\n                 f"| {f(r[\'head_tokens_per_round\'])} | {f(r[\'head_tokens_accepted\'])} |")\n    L += ["", "`lossless`: share of decodes identical to the 8B\'s unbatched greedy output. NoDraft is also below 1 "\n          "(fp16 near-ties), so compare with NoDraft, not with 1. Phase 1: 0.95–0.96 for every method.", "",\n          "`proposed tokens`: for V0/V1/V2 (and +lex) the heads\' tokens, accepted by the 8B; for the checked methods "\n          "(V*+check, NGram*) tokens read in a 1B call, accepted by the 1B."]\n    return "\\n".join(L) + "\\n"\n\n\n# --------------------------------------------------------------------------- #\n# notebook 09 entry point (also scripts/smoke_cpu.py --part wl)\n# --------------------------------------------------------------------------- #\n\nDEFAULT_WL = dict(\n    METHODS=["NoDraft", "Entropy", "V1+lex", "V1+check", "NGram", "NGram+V0"],\n    N_SENT=100, REPEATS=3, MAX_NEW=256,\n    TARGET=["meta-llama/Meta-Llama-3-8B-Instruct", "NousResearch/Meta-Llama-3-8B-Instruct"],\n    DRAFT=["meta-llama/Llama-3.2-1B-Instruct", "unsloth/Llama-3.2-1B-Instruct"],\n    MAX_MEMORY={0: "8GiB", 1: "10GiB"},\n    HEADS_DIR=None, WORDLIST=None, NGRAM=None, PARAMS=None,\n    SEARCH=["/kaggle/input"], OUT_ROOT="/kaggle/working", BUILD="unknown",\n    TIMING=dict(steps=60, reps=3, max_g=8), SMOKE=False, SMOKE_NOISE=0.0,\n)\n\n\ndef _first(names, load, log):\n    """``load(name)`` for the first name that can be loaded (gated repos fall back to mirrors)."""\n    names = [names] if isinstance(names, str) else list(names)\n    for i, n in enumerate(names):\n        try:\n            return load(n), n\n        except (OSError, ValueError) as e:\n            if i == len(names) - 1:\n                raise\n            log(f"  could not load {n} ({type(e).__name__}); trying {names[i + 1]}")\n\n\ndef heads_loop_main(cfg_in: dict, records: list[dict], dev_out_ids: list) -> dict:\n    """``records``: [{id, prompt_ids}] of the sentences to decode; ``dev_out_ids``: the 8B\'s dev outputs (lexicon)."""\n    torch = _torch()\n    from . import gpu_traces as G\n    cfg = {**DEFAULT_WL, **cfg_in}\n    out_dir = Path(cfg["OUT_ROOT"]) / "heads_loop"\n    out_dir.mkdir(parents=True, exist_ok=True)\n    logf = open(out_dir / "log.txt", "a", encoding="utf-8")\n\n    def log(msg):\n        print(msg, flush=True)\n        logf.write(str(msg) + "\\n")\n        logf.flush()\n\n    names = list(cfg["METHODS"])\n    assert names[0] == "NoDraft", "METHODS must start with NoDraft (the baseline)"\n    unknown = [n for n in names if n not in ("NoDraft", "Entropy", *TABLE_ROWS, *CHECKED_ROWS)]\n    assert not unknown, f"unknown methods {unknown}; choose from NoDraft, Entropy, {[*TABLE_ROWS, *CHECKED_ROWS]}"\n\n    # ---- inputs: fail before loading the 8B ---- #\n    variants = sorted({variant_of(n) for n in names} - {None})\n    use_lex = any(TABLE_ROWS[n][2] for n in names if n in TABLE_ROWS)\n    heads_dir, n_train, tag = None, 0, None\n    if variants:\n        heads_dir = find_heads([cfg["HEADS_DIR"]] if cfg["HEADS_DIR"] else cfg["SEARCH"], variants,\n                               Path(cfg["OUT_ROOT"]) / "_heads", log)\n        n_train = _n_train(heads_dir)\n        tag = "180k" if n_train >= 120000 else "60k"\n        log(f"heads: {heads_dir} -> parameters tuned for the {tag} heads")\n        if n_train == 0 and not cfg["SMOKE"]:\n            log("WARNING: the heads folder has no run_info.json / train_info.json, so 60k parameters are assumed")\n    params = cfg["PARAMS"][tag or "180k"]\n    methods = build_methods(names, params)\n    wordlist = None\n    if use_lex:\n        wordlist = Path(cfg["WORDLIST"]) if cfg["WORDLIST"] else find_file(cfg["SEARCH"], "hindi.csv")\n        if wordlist is None or not wordlist.exists():\n            raise FileNotFoundError("hindi.csv (the Hindi word list) not found: upload it as a Kaggle dataset and attach it, or "\n                                    "remove the \'+lex\' methods from METHODS")\n    ng_path = None\n    if any(m.ngram for m in methods):\n        ng_path = Path(cfg["NGRAM"]) if cfg["NGRAM"] else find_file(cfg["SEARCH"], "ngram_hi.npz")\n        if ng_path is None or not ng_path.exists():\n            raise FileNotFoundError("ngram_hi.npz (the Hindi n-gram table) not found: upload it as a Kaggle dataset and attach it, "\n                                    "or remove the \'NGram\' methods from METHODS")\n    log(f"methods: {[(m.name, m.params()) for m in methods]}")\n\n    # ---- models ---- #\n    if cfg["SMOKE"]:\n        device, dtype = "cpu", torch.float32\n        draft, draft_name = _first(cfg["DRAFT"], lambda n: G.load_draft(n, dtype, device), log)\n        target, target_name, tok = draft, draft_name, G.load_tokenizer(draft_name)\n    else:\n        assert torch.cuda.device_count() >= 2, "Need 2 GPUs: Settings > Accelerator > GPU T4 x2"\n        device, dtype = "cuda:0", torch.float16\n        target, target_name = _first(cfg["TARGET"], lambda n: G.load_target(n, dtype, max_memory=cfg["MAX_MEMORY"]), log)\n        tok = G.load_tokenizer(target_name)\n        draft, draft_name = _first(cfg["DRAFT"], lambda n: G.load_draft(n, dtype, device), log)\n        G.check_draft_target_compat(draft, target, G.load_tokenizer(draft_name), tok, "परीक्षण")\n    eos_ids = G.get_eos_ids(target, tok)\n    d = get_backbone_and_lm_weight(draft)[1].shape[1]\n    heads = {}\n    for v in variants:\n        sd = {k: t.float() for k, t in torch.load(heads_dir / f"wordheads_{v}.pt", map_location="cpu").items()}\n        hd = make_heads(d, sum(k.startswith("blocks.") and k.endswith(".weight") for k in sd))\n        hd.load_state_dict(sd)\n        heads[v] = hd.to(device).eval()\n    cv = continuation_vocab(tok) if variants else None\n    gate = devanagari_mask(tok) if variants else None\n    lexicon = build_lexicon(tok, dev_out_ids, wordlist) if use_lex else None\n    ngram = NGram.load(ng_path) if ng_path else None\n    if ngram is not None:\n        log(f"n-gram table: {ng_path}, order {ngram.order}, contexts {[len(a) for a, _ in ngram.tab.values()]}")\n    if not cfg["SMOKE"]:\n        if cv is not None and len(cv) != PAPER_CONT_VOCAB:\n            log(f"WARNING: continuation vocabulary has {len(cv)} tokens, the paper says {PAPER_CONT_VOCAB}")\n        if lexicon is not None:\n            n_lex = len(lexicon.words)\n            log(f"lexicon: {n_lex} words from {wordlist} + dev outputs (paper: {PAPER_LEXICON_TYPES})")\n            if n_lex != PAPER_LEXICON_TYPES:\n                log("WARNING: the lexicon size differs from the paper\'s; is this the same hindi.csv?")\n    dec = HeadsDecoder(target, draft, tok, eos_ids, device, heads, lexicon, gate, cv, noise=cfg["SMOKE_NOISE"],\n                       ngram=ngram)\n\n    recs = list(records)[: cfg["N_SENT"]]\n    resume_files(cfg["SEARCH"], out_dir, log)\n\n    # ---- timing of the model calls in this session (cost model) ---- #\n    if not (out_dir / "timing.json").exists():\n        tm = cfg["TIMING"]\n        G.run_timing(draft, target, out_dir / "timing.json", device, dict(build=cfg["BUILD"]), steps=tm["steps"],\n                     reps=tm["reps"], max_g=tm["max_g"])\n\n    # ---- unbatched greedy references ---- #\n    ref_path = out_dir / "greedy_unbatched.jsonl"\n    kept = read_records(ref_path)\n    ref_path.write_text("".join(json.dumps(r, ensure_ascii=False) + "\\n" for r in kept), encoding="utf-8")\n    refs = {r["id"]: r for r in kept}\n    for r in recs:\n        if r["id"] not in refs:\n            o = G.generate_unbatched(target, tok, r["prompt_ids"], cfg["MAX_NEW"], eos_ids, device)\n            refs[r["id"]] = dict(id=r["id"], out_ids=o, out_text=tok.decode(o, skip_special_tokens=True).strip())\n            _append(ref_path, refs[r["id"]])\n    log(f"greedy references: {len(refs)}")\n\n    # ---- timed runs (resumable; decodes with other parameters or heads are ignored) ---- #\n    ng_tag = hashlib.sha256(Path(ng_path).read_bytes()).hexdigest()[:12] if ng_path else None\n\n    def run_key(m):\n        d = dict(params=m.params(), heads=n_train if m.kind == "heads" or (m.kind == "checked" and m.variant) else None)\n        if m.ngram:\n            d["ngram"] = ng_tag\n        return json.dumps(d, sort_keys=True)\n\n    key = {m.name: run_key(m) for m in methods}\n    res_path = out_dir / "loop_results.jsonl"\n    ids = {r["id"] for r in recs}\n    results = [r for r in read_records(res_path) if key.get(r["method"]) == r["key"] and r["id"] in ids\n               and r["rep"] < cfg["REPEATS"]]\n    res_path.write_text("".join(json.dumps(r, ensure_ascii=False) + "\\n" for r in results), encoding="utf-8")\n    done = {(r["rep"], r["id"], r["method"]) for r in results}\n    if done:\n        log(f"resume: {len(done)} decodes reused")\n    for m in methods:                                 # warm-up\n        dec.decode(recs[0]["prompt_ids"], m, min(32, cfg["MAX_NEW"]))\n    t0, n_new, n_sent = time.perf_counter(), 0, 0\n    for rep in range(cfg["REPEATS"]):\n        for si, r in enumerate(recs):\n            order = methods[(rep + si) % len(methods):] + methods[: (rep + si) % len(methods)]\n            fresh = False\n            for m in order:\n                if (rep, r["id"], m.name) in done:\n                    continue\n                x = dec.decode(r["prompt_ids"], m, cfg["MAX_NEW"])\n                ref = refs[r["id"]]["out_ids"]\n                rec = dict(rep=rep, id=r["id"], method=m.name, key=key[m.name], wall=x.wall, prefill=x.prefill,\n                           n_out=len(x.out_ids), lossless=x.out_ids == ref,\n                           first_divergence=first_divergence(x.out_ids, ref), out_ids=x.out_ids, rounds=x.rounds)\n                _append(res_path, rec)\n                results.append(rec)\n                n_new, fresh = n_new + 1, True\n            n_sent += fresh\n            if fresh and n_sent == 5:\n                per = (time.perf_counter() - t0) / 5\n                log(f"ESTIMATE: {per:.1f} s per sentence (all methods) -> about "\n                    f"{per * len(recs) * cfg[\'REPEATS\'] / 3600:.1f} h for N_SENT={len(recs)} x REPEATS={cfg[\'REPEATS\']}")\n        log(f"repeat {rep} done ({(time.perf_counter() - t0) / 60:.0f} min)")\n\n    rows = summarize_loop(results, names, params)\n    info = G.run_info(cfg["BUILD"], methods={m.name: m.params() for m in methods}, n_sent=len(recs),\n                      repeats=cfg["REPEATS"], heads_dir=str(heads_dir), heads_n_train=n_train, heads_tag=tag,\n                      lexicon_words=len(lexicon.words) if lexicon else None, wordlist=str(wordlist),\n                      ngram=str(ng_path) if ng_path else None, ngram_sha256=ng_tag,\n                      target=target_name, draft=draft_name, smoke=cfg["SMOKE"],\n                      cont_vocab=len(cv) if cv is not None else None, new_decodes=n_new)\n    (out_dir / "run_info.json").write_text(json.dumps(info, indent=1, ensure_ascii=False), encoding="utf-8")\n    (out_dir / "summary.json").write_text(json.dumps(rows, indent=1, ensure_ascii=False), encoding="utf-8")\n    md = summary_md(rows, info)\n    (out_dir / "SUMMARY.md").write_text(md, encoding="utf-8")\n    log("\\n" + md)\n    logf.close()\n    zp = G.zip_dir(out_dir, Path(cfg["OUT_ROOT"]) / "heads_loop.zip")\n    print("DONE. Download heads_loop.zip from the Output tab and unzip it into  results/heads_loop/  in the repo.")\n    return dict(zip=zp, rows=rows, info=info)\n',
}
for _n, _s in LGMTP_SOURCES.items():
    with open(os.path.join(PKG, _n + '.py'), 'w', encoding='utf-8') as _f:
        _f.write(_s)
sys.path.insert(0, os.path.dirname(PKG))
for _m in [m for m in list(sys.modules) if m == 'lgmtp' or m.startswith('lgmtp.')]:
    del sys.modules[_m]
import lgmtp
LGMTP_BUILD = 'git=f73985b92d21-dirty src+data_sha256=dbdbcab6a266'
print('lgmtp build', LGMTP_BUILD)

In [ ]:
import base64, gzip, json
_B64 = '...'
DATA = json.loads(gzip.decompress(base64.b64decode(_B64)))
PARAMS = json.loads('{"60k": {"Entropy": {"lam": -0.683, "sim": 1.4518}, "V0": {"lam": -0.683, "tau_h": 0.8, "tau_c": null, "sim": 1.4971}, "V1": {"lam": -0.8, "tau_h": 0.6, "tau_c": 0.5, "sim": 1.5135}, "V2": {"lam": -0.683, "tau_h": 0.7, "tau_c": 0.4, "sim": 1.5054}, "V0+lex": {"lam": -0.8, "tau_h": 0.5, "tau_c": null, "sim": 1.5288}, "V1+lex": {"lam": -1.1, "tau_h": 0.4, "tau_c": 0.3, "sim": 1.5327}, "NGram": {"lam": -0.6, "L": 7, "tau_h": 0.0, "sim": 1.7008}, "V0+check": {"lam": -0.8, "L": 6, "tau_h": 0.0, "sim": 1.5622}, "V1+check": {"lam": -0.8, "L": 6, "tau_h": 0.0, "sim": 1.5668}, "NGram+V0": {"lam": -0.6, "L": 6, "tau_h": 0.5, "sim": 1.705}, "NGram+V1": {"lam": -0.6, "L": 6, "tau_h": 0.5, "sim": 1.7019}}, "180k": {"Entropy": {"lam": -0.683, "sim": 1.4518}, "V0": {"lam": -0.8, "tau_h": 0.7, "tau_c": null, "sim": 1.5104}, "V1": {"lam": -0.8, "tau_h": 0.6, "tau_c": 0.5, "sim": 1.518}, "V2": {"lam": -0.8, "tau_h": 0.6, "tau_c": 0.5, "sim": 1.5168}, "V0+lex": {"lam": -1.1, "tau_h": 0.5, "tau_c": null, "sim": 1.5376}, "V1+lex": {"lam": -1.1, "tau_h": 0.4, "tau_c": 0.3, "sim": 1.5422}, "NGram": {"lam": -0.6, "L": 7, "tau_h": 0.0, "sim": 1.7008}, "V0+check": {"lam": -0.8, "L": 6, "tau_h": 0.0, "sim": 1.5682}, "V1+check": {"lam": -0.8, "L": 6, "tau_h": 0.0, "sim": 1.5717}, "NGram+V0": {"lam": -0.6, "L": 6, "tau_h": 0.5, "sim": 1.7083}, "NGram+V1": {"lam": -0.6, "L": 6, "tau_h": 0.5, "sim": 1.7049}}}')
print(len(DATA['records']), 'sentences;', len(DATA['dev_out_ids']), 'dev outputs for the lexicon')

## CONFIG

In [ ]:
CONFIG = dict(
    METHODS=["NoDraft", "Entropy", "V1+lex", "V1+check", "NGram", "NGram+V0"],   # also: "V0", "V1", "V2", "V0+lex", "V0+check", "NGram+V1"
    N_SENT=100, REPEATS=3, MAX_NEW=256,
    HEADS_DIR=None,     # None = search /kaggle/input for wordheads_v1.pt (folders and zips)
    WORDLIST=None,      # None = search /kaggle/input for hindi.csv
    NGRAM=None,         # None = search /kaggle/input for ngram_hi.npz
    TARGET=["meta-llama/Meta-Llama-3-8B-Instruct", "NousResearch/Meta-Llama-3-8B-Instruct"],
    DRAFT=["meta-llama/Llama-3.2-1B-Instruct", "unsloth/Llama-3.2-1B-Instruct"],
    MAX_MEMORY={0: "8GiB", 1: "10GiB"},
    SEARCH=["/kaggle/input"], OUT_ROOT="/kaggle/working", BUILD=LGMTP_BUILD,
)

In [ ]:
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret("HF_TOKEN"))
    print("Logged in with Kaggle secret HF_TOKEN")
except Exception as e:
    print(f"No HF_TOKEN ({e}); ungated copies of the models will be tried")
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| GPUs", torch.cuda.device_count())

In [ ]:
from lgmtp.heads_loop import heads_loop_main
RESULT = heads_loop_main(dict(CONFIG, PARAMS=PARAMS), DATA['records'], DATA['dev_out_ids'])

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
display(Markdown(Path(CONFIG["OUT_ROOT"], "heads_loop", "SUMMARY.md").read_text(encoding="utf-8")))